# Robustness analysis notebook

This notebook documents the robustness analysis carried out for the destination choice modelling framework. The purpose of this notebook is not to search for a better specification after observing the results, but to verify whether the main empirical conclusions are sensitive to specific modelling decisions.

The robustness logic is based on a controlled one-change-at-a-time design. Starting from the baseline specification, one modelling component is modified while all remaining elements of the estimation and evaluation pipeline are kept unchanged. Each alternative specification is then re-estimated and evaluated using the same out-of-sample procedure as the baseline model. This makes it possible to interpret differences in predictive performance as the consequence of the specific modelling change being tested.

The notebook compares each robustness specification against the baseline using full-choice-set out-of-sample evaluation. The comparison is based on the same core metrics used in the main model evaluation:

- weighted negative log-likelihood;
- mean reciprocal rank;
- McFadden's pseudo-R2;
- Spearman correlation between observed and predicted destination shares;
- Jensen-Shannon divergence.

The robustness checks are organised into seven families.

## R1: Preprocessing choices

This family tests whether the results depend on specific preprocessing decisions applied before model estimation. The checks include alternative treatments of the destination universe, raw destination-side variables instead of log-transformed variables, removal of the global standardisation step, density-based versions of selected amenity-count variables, repeated leave-one-variable-out specifications, a reduced specification using only the most important destination-side variables, and the exclusion of observations belonging to the Visits leisure category.

The technical purpose of this family is to verify whether the estimated destination attractivity patterns are driven by a particular preprocessing choice or by one specific variable.

## R2: Accessibility component

This family tests whether the results depend on the baseline accessibility component. The baseline EMU-based accessibility term is replaced by simpler impedance proxies, including beeline distance and network travel-distance skims.

The technical purpose is to check whether the substantive findings require the specific EMU formulation, or whether similar predictive patterns are obtained with simpler spatial impedance measures.

## R3: Sampled choice-set construction

This family tests the sensitivity of the model to the construction of the sampled choice set used during estimation. The notebook varies the number of sampled non-chosen alternatives and also evaluates an EMU-guided sampling strategy. When the sampling design changes, the corresponding likelihood correction is applied.

The technical purpose is to verify that the final conclusions do not depend on the exact value of K or on uniform sampling of non-chosen destinations. The notebook also compares sampled train, sampled out-of-sample, and full-choice-set out-of-sample negative log-likelihood to check whether increasing K changes the generalisation behaviour of the model.

## R4: Out-of-sample split design

This family tests whether the results depend on the baseline train and out-of-sample split design. Instead of splitting by person, the alternative specification uses an origin-based split, where trips from the same origin traffic zone are assigned jointly to the same side of the split.

The technical purpose is to check whether the model remains stable when out-of-sample evaluation is made more spatially structured.

## R5: Survey-year sensitivity

This family tests whether the results are stable across survey waves. The model is estimated separately on the 2015 and 2021 observations and compared with the pooled baseline specification.

The technical purpose is to assess whether the estimated destination-choice patterns are specific to one survey year or remain broadly consistent over time.

## R6: Regularisation

This family tests whether the baseline model is affected by instability in the destination-side coefficients. The unpenalised baseline specification is compared with L1- and L2-penalised variants. The penalties are applied only to the destination-side attractivity coefficients, while the accessibility coefficient is left unpenalised.

The technical purpose is to check whether the predictive performance and coefficient patterns depend on potentially unstable correlated destination-side variables.

## R7: Spatial spillovers and residual spatial dependence

This family tests whether adding spatially lagged destination attributes improves the baseline model. The utility specification is extended with a spatial spillover component based on neighbouring destination attributes. After estimation, the notebook evaluates the model out of sample and computes residual destination-share autocorrelation using Moran's I.

The technical purpose is to verify whether remaining prediction errors have a relevant spatial structure and whether a spatially lagged destination-attractivity term materially improves the model.

## Interpretation of the robustness outputs

The outputs of this notebook should be interpreted as sensitivity checks around the baseline specification. The relevant question is not whether an alternative specification performs marginally better in one metric, but whether the main conclusions change when individual modelling choices are modified.

Overall, the robustness analysis supports the stability of the baseline specification. Most alternative choices lead only to limited changes in out-of-sample performance. The more visible deviations are mainly associated with the log transformation and the exclusion of Visits observations, but these changes do not overturn the main empirical conclusions.

# R1: Preprocessing Decisions

## R1.1 Liechtenstein

In [ ]:
# ============================================================
# ROBUSTNESS: WITH Liechtenstein vs WITHOUT Liechtenstein (woLIE)
# FIXED (hygienic + deterministic):
#   - K = 1000
#   - 50 epochs
#   - uniform sampling (chosen + K non-chosen)
#   - z-standardize destination vars on the FULL TZ universe (per scenario)
#   - deterministic seeds per segment (no Python hash randomness)
#
# For each scenario and segment:
#   Sampled TRAIN metrics: NLL, MRR, McFadden_R2, rank_median, Top1/10/50/100
#   Sampled OOS   metrics: NLL, MRR, McFadden_R2, rank_median, Top1/10/50/100
#   FULL-choice-set metrics on OOS subset (all zones as alternatives):
#       NLL_full, MRR_full, McFadden_R2_full, rank_median_full
#       Top1/10/50/100_full
#       share_spearman_full, share_JS_full
#
# Outputs:
#   ModelRuns/LIEComparison/lie_comparison_results_long.csv
#   ModelRuns/LIEComparison/lie_comparison_results_wide.csv
#   ModelRuns/LIEComparison/lie_comparison_deltas_woLIE_minus_withLIE.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path


# -----------------------------
# CONFIG
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

# Two scenarios to compare
SCENARIOS = {
    "withLIE": {
        "TRIPS_DIR": "Trips/byPerson",
        "TZ_GPKG":   "VariableAnalysis/TZ_first_sel_log1p.gpkg",
        "TZ_LAYER":  "TZ_first_sel_log1p",
        "OMX":       Path("TravelCost/skims/MOBi5_2023_ready") / "utilities_2023_ready.omx",
        "MAP_NAME":  "NO",
        "EMU_MAT":   "utility_emu",
    },
    "woLIE": {
        "TRIPS_DIR": "Trips/byPerson_woLIE",
        "TZ_GPKG":   "VariableAnalysis/TZ_first_sel_log1p_woLIE.gpkg",
        "TZ_LAYER":  "TZ_first_sel_log1p_woLIE",
        "OMX":       Path("TravelCost/skims/MOBi5_2023_ready") / "utilities_2023_ready_woLIE.omx",
        "MAP_NAME":  "NO",
        "EMU_MAT":   "utility_emu",
    }
}

OUT_DIR = "ModelRuns/LIEComparison"
os.makedirs(OUT_DIR, exist_ok=True)

# Fixed robustness decision
K = 1000
BASE_SEED = 123

# Torch
DEVICE = "cpu"
DTYPE  = torch.float32

# Optimization
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# Full-choice-set evaluation subset sizes (OOS)
FULL_EVAL_MAX = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256

TOPKS = (1, 10, 50, 100)

LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",          # update if needed
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]


# -----------------------------
# Small helpers
# -----------------------------
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300); q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)


# -----------------------------
# IO helpers
# -----------------------------
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df

def load_emu_matrix_and_mapping(omx_path: Path, emu_mat: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if emu_mat not in f.list_matrices():
        raise ValueError(f"Matrix '{emu_mat}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[emu_mat]
    print(f"[OMX] Loading {emu_mat} into RAM... shape={M.shape} (float32)")
    emu = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return emu, zone_to_idx, idx_to_zone


# -----------------------------
# Preprocess helpers
# -----------------------------
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)

    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts

    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)  # chosen always first

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2
    )


# -----------------------------
# Model: train/eval on sampled sets
# -----------------------------
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, J = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state  # (alpha_hat, beta_hat)

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(1,10,50,100)):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = int(V.shape[1])
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

        out = {
            "NLL": nll,
            "MRR": mrr,
            "McFadden_R2": float(r2),
            "rank_median": rank_median,
            "J": J,
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))

    return out


# -----------------------------
# FULL-choice-set evaluation on OOS subset (all zones available)
# -----------------------------
def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(1,10,50,100),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    # destination fixed term Aj = X_j beta aligned to TZ universe
    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    # Aj aligned to OMX indices
    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    # shares on TZ universe
    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    ll_model = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    top_num = {k: 0.0 for k in topKs}

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        # pass 1: max V
        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        # pass 2: sumexp + rank
        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        nll_num += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        # pass 3: shares accumulation
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    out = {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    return out


# ============================================================
# MAIN
# ============================================================

results = []

for scen_name, cfg in SCENARIOS.items():
    print("\n" + "#" * 120)
    print(f"# SCENARIO = {scen_name}")
    print("# trips_dir:", cfg["TRIPS_DIR"])
    print("# tz:", cfg["TZ_GPKG"], "| layer:", cfg["TZ_LAYER"])
    print("# omx:", cfg["OMX"])
    print("#" * 120)

    # --- Load TZ features (scenario-specific universe)
    print("[LOAD] TZ features...")
    tz = gpd.read_file(cfg["TZ_GPKG"], layer=cfg["TZ_LAYER"])

    need_cols = ["npvm_id"] + LOG1P_COLS
    miss = [c for c in need_cols if c not in tz.columns]
    if miss:
        raise ValueError(f"[{scen_name}] TZ layer missing {miss}")

    tz_feat = tz[["npvm_id"] + LOG1P_COLS + ["geometry"]].copy()
    tz_feat["npvm_id"] = pd.to_numeric(tz_feat["npvm_id"], errors="coerce")
    tz_feat = tz_feat.dropna(subset=["npvm_id"]).copy()
    tz_feat["npvm_id"] = tz_feat["npvm_id"].astype(int)

    # global z-standardize on full TZ universe (per scenario)
    print("[PREP] Global z-standardize destination vars on full TZ universe (scenario-specific)...")
    tz_feat, z_cols, mu_all, sd_all = standardize_all_zones(tz_feat, LOG1P_COLS, suffix="_z")

    # --- Load OMX (scenario-specific)
    print("[LOAD] EMU matrix + mapping...")
    emu_mat, zone_to_idx, idx_to_zone = load_emu_matrix_and_mapping(cfg["OMX"], cfg["EMU_MAT"], cfg["MAP_NAME"])
    print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

    trips_dir = cfg["TRIPS_DIR"]

    for seg in SEGMENTS:
        print("\n" + "=" * 100)
        print(f"Scenario={scen_name} | Segment={seg} | K={K} | epochs={EPOCHS}")
        print("=" * 100)

        train_df = load_segment_csv(trips_dir, seg, "train")
        oos_df   = load_segment_csv(trips_dir, seg, "oos")

        # deterministic seeds (scenario + seg + split)
        scen_offset = 0 if scen_name == "withLIE" else 9999
        seed_tr = BASE_SEED + scen_offset + 1000 * SEG_SEED[seg] + 10
        seed_te = BASE_SEED + scen_offset + 1000 * SEG_SEED[seg] + 20

        # sampled designs
        emu_tr, X_tr, y_tr, w_tr, train_df2 = build_design(
            train_df, tz_feat, z_cols, emu_mat, zone_to_idx, k=K, seed=seed_tr
        )
        emu_te, X_te, y_te, w_te, oos_df2 = build_design(
            oos_df, tz_feat, z_cols, emu_mat, zone_to_idx, k=K, seed=seed_te
        )

        print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
        print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

        # train
        alpha_hat, beta_hat = train_mnl(
            emu_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
        )

        # sampled metrics
        met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS)
        met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS)

        # full metrics on OOS subset
        max_n = FULL_EVAL_MAX.get(seg, 8000)
        seed_full = FULL_EVAL_SEED + scen_offset + 1000 * SEG_SEED[seg]
        full_met = full_choice_eval_oos_subset(
            oos_df=oos_df2,
            tz_feat_seg=tz_feat,
            z_cols=z_cols,
            emu_mat=emu_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=max_n,
            seed=seed_full,
            chunk=FULL_EVAL_CHUNK,
            topKs=TOPKS,
        )

        row = {
            "scenario": scen_name,
            "segment": seg,
            "K": K,
            "alpha_emu": float(alpha_hat),

            "NLL_train": met_tr["NLL"],
            "MRR_train": met_tr["MRR"],
            "McFadden_R2_train": met_tr["McFadden_R2"],
            "rank_median_train": met_tr["rank_median"],
            "Top1_train": met_tr["Top1"],
            "Top10_train": met_tr["Top10"],
            "Top50_train": met_tr["Top50"],
            "Top100_train": met_tr["Top100"],

            "NLL_oos": met_te["NLL"],
            "MRR_oos": met_te["MRR"],
            "McFadden_R2_oos": met_te["McFadden_R2"],
            "rank_median_oos": met_te["rank_median"],
            "Top1_oos": met_te["Top1"],
            "Top10_oos": met_te["Top10"],
            "Top50_oos": met_te["Top50"],
            "Top100_oos": met_te["Top100"],
        }
        row.update(full_met)
        results.append(row)

        print("[SAMPLED TRAIN]",
              f"NLL={row['NLL_train']:.4f} MRR={row['MRR_train']:.4f} R2={row['McFadden_R2_train']:.4f} "
              f"Top1={row['Top1_train']:.3f} Top10={row['Top10_train']:.3f} rMed={row['rank_median_train']:.1f}")
        print("[SAMPLED OOS ]",
              f"NLL={row['NLL_oos']:.4f} MRR={row['MRR_oos']:.4f} R2={row['McFadden_R2_oos']:.4f} "
              f"Top1={row['Top1_oos']:.3f} Top10={row['Top10_oos']:.3f} rMed={row['rank_median_oos']:.1f}")

        if full_met:
            print("[FULL OOS   ]",
                  f"NLL_full={row['NLL_full']:.4f} MRR_full={row['MRR_full']:.4f} R2_full={row['McFadden_R2_full']:.4f} "
                  f"Top1_full={row['Top1_full']:.3f} Top10_full={row['Top10_full']:.3f} rMed_full={row['rank_median_full']:.1f}")
            print("[SHARES FULL ]",
                  f"spearman={row['share_spearman_full']:.4f} JS={row['share_JS_full']:.4f} "
                  f"N_full={row['N_full_eval']} J_full={row['J_full']}")

# -----------------------------
# Save outputs
# -----------------------------
res_df = pd.DataFrame(results).sort_values(["segment", "scenario"]).reset_index(drop=True)

long_path = os.path.join(OUT_DIR, "lie_comparison_results_long.csv")
res_df.to_csv(long_path, index=False)
print(f"\n[OK] wrote {long_path}")

# Wide (pivot)
value_cols = [c for c in res_df.columns if c not in ("scenario", "segment")]
wide = res_df.pivot(index="segment", columns="scenario", values=value_cols)

# flatten MultiIndex columns
wide.columns = [f"{v}__{s}" for (v, s) in wide.columns]
wide = wide.reset_index()

wide_path = os.path.join(OUT_DIR, "lie_comparison_results_wide.csv")
wide.to_csv(wide_path, index=False)
print(f"[OK] wrote {wide_path}")

# Deltas: woLIE - withLIE
delta = pd.DataFrame({"segment": wide["segment"]})
for col in value_cols:
    c_with = f"{col}__withLIE"
    c_wo   = f"{col}__woLIE"
    if c_with in wide.columns and c_wo in wide.columns:
        delta[col + "__delta_woLIE_minus_withLIE"] = wide[c_wo] - wide[c_with]

delta_path = os.path.join(OUT_DIR, "lie_comparison_deltas_woLIE_minus_withLIE.csv")
delta.to_csv(delta_path, index=False)
print(f"[OK] wrote {delta_path}")

print("\n[DONE] Outputs in:", OUT_DIR)

## Log (x + 1)

In [ ]:
# ============================================================
# ROBUSTNESS: LOG1P vs RAW destination variables (with LIE, byPerson)
# FIXED (hygienic + deterministic):
#   - LIE everywhere
#   - byPerson trip split
#   - K = 1000
#   - 50 epochs
#   - uniform sampling (chosen + K non-chosen)
#   - z-standardize destination vars on the FULL TZ universe (per variant)
#   - deterministic seeds per segment (no Python hash randomness)
#
# Variants:
#   A) LOG1P  : use *_log1p columns
#   B) RAW    : use raw columns (same names without _log1p suffix)
#
# For each variant and segment:
#   Sampled TRAIN metrics: NLL, MRR, McFadden_R2, rank_median, Top1/10/50/100
#   Sampled OOS   metrics: NLL, MRR, McFadden_R2, rank_median, Top1/10/50/100
#   FULL-choice-set metrics on OOS subset (all zones as alternatives):
#       NLL_full, MRR_full, McFadden_R2_full, rank_median_full
#       Top1/10/50/100_full
#       share_spearman_full, share_JS_full
#
# Outputs:
#   ModelRuns/Log1PComparison/log1p_comparison_results_long.csv
#   ModelRuns/Log1PComparison/log1p_comparison_results_wide.csv
#   ModelRuns/Log1PComparison/log1p_comparison_deltas_RAW_minus_LOG1P.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path


# -----------------------------
# CONFIG (LIE + byPerson)
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TRIPS_DIR = "Trips/byPerson"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT   = "utility_emu"
MAP_NAME  = "NO"

OUT_DIR = "ModelRuns/Log1PComparison"
os.makedirs(OUT_DIR, exist_ok=True)

# Fixed robustness decision
K = 1000
BASE_SEED = 123

# Torch
DEVICE = "cpu"
DTYPE  = torch.float32

# Optimization
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# Full-choice-set evaluation subset sizes (OOS)
FULL_EVAL_MAX = {
    "YS": 20000,
    "OS": 8000,
    "YL": 8000,
    "OL": 8000,
}
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256

TOPKS = (1, 10, 50, 100)

# LOG1P columns (as in your current pipeline)
LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",         
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

# RAW columns derived by removing suffix _log1p
RAW_COLS = [c.replace("_log1p", "") for c in LOG1P_COLS]

VARIANTS = {
    "LOG1P": LOG1P_COLS,
    "RAW":   RAW_COLS,
}


# -----------------------------
# Small helpers
# -----------------------------
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300); q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)


# -----------------------------
# IO helpers
# -----------------------------
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df

def load_emu_matrix_and_mapping(omx_path: Path, emu_mat: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if emu_mat not in f.list_matrices():
        raise ValueError(f"Matrix '{emu_mat}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[emu_mat]
    print(f"[OMX] Loading {emu_mat} into RAM... shape={M.shape} (float32)")
    emu = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return emu, zone_to_idx, idx_to_zone


# -----------------------------
# Preprocess helpers
# -----------------------------
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    """
    Z-standardize on the FULL TZ universe (all alternatives),
    i.e., a fixed transformation of alternative attributes.
    """
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)

    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts

    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)  # chosen always first

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2
    )


# -----------------------------
# Model: train/eval on sampled sets
# -----------------------------
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, J = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state  # (alpha_hat, beta_hat)

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(1,10,50,100)):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = int(V.shape[1])
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

        out = {
            "NLL": nll,
            "MRR": mrr,
            "McFadden_R2": float(r2),
            "rank_median": rank_median,
            "J": J,
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))

    return out


# -----------------------------
# FULL-choice-set evaluation on OOS subset (all zones available)
# -----------------------------
def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(1,10,50,100),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    ll_model = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    top_num = {k: 0.0 for k in topKs}

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        nll_num += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    out = {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    return out


# ============================================================
# MAIN
# ============================================================

print("[LOAD] EMU matrix + mapping...")
emu_mat, zone_to_idx, idx_to_zone = load_emu_matrix_and_mapping(UTIL_OMX, EMU_MAT, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

print("[LOAD] TZ features...")
tz0 = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)

# basic checks for both variants
need_cols = ["npvm_id"] + list(set(LOG1P_COLS + RAW_COLS))
miss = [c for c in need_cols if c not in tz0.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

tz0 = tz0[["npvm_id"] + list(set(LOG1P_COLS + RAW_COLS)) + ["geometry"]].copy()
tz0["npvm_id"] = pd.to_numeric(tz0["npvm_id"], errors="coerce")
tz0 = tz0.dropna(subset=["npvm_id"]).copy()
tz0["npvm_id"] = tz0["npvm_id"].astype(int)

results = []

for variant_name, feat_cols in VARIANTS.items():
    print("\n" + "#" * 120)
    print(f"# VARIANT = {variant_name} | K={K} | epochs={EPOCHS} | byPerson")
    print("# columns:", feat_cols)
    print("#" * 120)

    # Copy and standardize on full TZ universe (variant-specific)
    tz_feat = tz0[["npvm_id"] + feat_cols + ["geometry"]].copy()

    print("[PREP] Global z-standardize destination vars on full TZ universe (variant-specific)...")
    tz_feat, z_cols, mu_all, sd_all = standardize_all_zones(tz_feat, feat_cols, suffix="_z")

    for seg in SEGMENTS:
        print("\n" + "=" * 100)
        print(f"Variant={variant_name} | Segment={seg} | K={K}")
        print("=" * 100)

        train_df = load_segment_csv(TRIPS_DIR, seg, "train")
        oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

        # deterministic seeds: variant + seg + split
        var_offset = 0 if variant_name == "LOG1P" else 9999
        seed_tr = BASE_SEED + var_offset + 1000 * SEG_SEED[seg] + 10
        seed_te = BASE_SEED + var_offset + 1000 * SEG_SEED[seg] + 20

        emu_tr, X_tr, y_tr, w_tr, train_df2 = build_design(
            train_df, tz_feat, z_cols, emu_mat, zone_to_idx, k=K, seed=seed_tr
        )
        emu_te, X_te, y_te, w_te, oos_df2 = build_design(
            oos_df, tz_feat, z_cols, emu_mat, zone_to_idx, k=K, seed=seed_te
        )

        print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
        print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

        alpha_hat, beta_hat = train_mnl(
            emu_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
        )

        met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS)
        met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS)

        max_n = FULL_EVAL_MAX.get(seg, 8000)
        seed_full = FULL_EVAL_SEED + var_offset + 1000 * SEG_SEED[seg]
        full_met = full_choice_eval_oos_subset(
            oos_df=oos_df2,
            tz_feat_seg=tz_feat,
            z_cols=z_cols,
            emu_mat=emu_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=max_n,
            seed=seed_full,
            chunk=FULL_EVAL_CHUNK,
            topKs=TOPKS,
        )

        row = {
            "variant": variant_name,
            "segment": seg,
            "K": K,
            "alpha_emu": float(alpha_hat),

            "NLL_train": met_tr["NLL"],
            "MRR_train": met_tr["MRR"],
            "McFadden_R2_train": met_tr["McFadden_R2"],
            "rank_median_train": met_tr["rank_median"],
            "Top1_train": met_tr["Top1"],
            "Top10_train": met_tr["Top10"],
            "Top50_train": met_tr["Top50"],
            "Top100_train": met_tr["Top100"],

            "NLL_oos": met_te["NLL"],
            "MRR_oos": met_te["MRR"],
            "McFadden_R2_oos": met_te["McFadden_R2"],
            "rank_median_oos": met_te["rank_median"],
            "Top1_oos": met_te["Top1"],
            "Top10_oos": met_te["Top10"],
            "Top50_oos": met_te["Top50"],
            "Top100_oos": met_te["Top100"],
        }
        row.update(full_met)
        results.append(row)

        print("[SAMPLED TRAIN]",
              f"NLL={row['NLL_train']:.4f} MRR={row['MRR_train']:.4f} R2={row['McFadden_R2_train']:.4f} "
              f"Top1={row['Top1_train']:.3f} Top10={row['Top10_train']:.3f} rMed={row['rank_median_train']:.1f}")
        print("[SAMPLED OOS ]",
              f"NLL={row['NLL_oos']:.4f} MRR={row['MRR_oos']:.4f} R2={row['McFadden_R2_oos']:.4f} "
              f"Top1={row['Top1_oos']:.3f} Top10={row['Top10_oos']:.3f} rMed={row['rank_median_oos']:.1f}")

        if full_met:
            print("[FULL OOS   ]",
                  f"NLL_full={row['NLL_full']:.4f} MRR_full={row['MRR_full']:.4f} R2_full={row['McFadden_R2_full']:.4f} "
                  f"Top1_full={row['Top1_full']:.3f} Top10_full={row['Top10_full']:.3f} rMed_full={row['rank_median_full']:.1f}")
            print("[SHARES FULL ]",
                  f"spearman={row['share_spearman_full']:.4f} JS={row['share_JS_full']:.4f} "
                  f"N_full={row['N_full_eval']} J_full={row['J_full']}")

# -----------------------------
# Save outputs
# -----------------------------
res_df = pd.DataFrame(results).sort_values(["segment", "variant"]).reset_index(drop=True)

long_path = os.path.join(OUT_DIR, "log1p_comparison_results_long.csv")
res_df.to_csv(long_path, index=False)
print(f"\n[OK] wrote {long_path}")

value_cols = [c for c in res_df.columns if c not in ("variant", "segment")]
wide = res_df.pivot(index="segment", columns="variant", values=value_cols)
wide.columns = [f"{v}__{s}" for (v, s) in wide.columns]
wide = wide.reset_index()

wide_path = os.path.join(OUT_DIR, "log1p_comparison_results_wide.csv")
wide.to_csv(wide_path, index=False)
print(f"[OK] wrote {wide_path}")

# Deltas: RAW - LOG1P
delta = pd.DataFrame({"segment": wide["segment"]})
for col in value_cols:
    c_log = f"{col}__LOG1P"
    c_raw = f"{col}__RAW"
    if c_log in wide.columns and c_raw in wide.columns:
        delta[col + "__delta_RAW_minus_LOG1P"] = wide[c_raw] - wide[c_log]

delta_path = os.path.join(OUT_DIR, "log1p_comparison_deltas_RAW_minus_LOG1P.csv")
delta.to_csv(delta_path, index=False)
print(f"[OK] wrote {delta_path}")

print("\n[DONE] Outputs in:", OUT_DIR)

## Z Standardization

In [ ]:
# ============================================================
# ROBUSTNESS: z-standardization ON vs OFF (LOG1P vars, LIE, byPerson)
# FIXED (hygienic + deterministic):
#   - LIE everywhere
#   - byPerson trip split
#   - LOG1P destination variables
#   - K = 1000
#   - 50 epochs
#   - uniform sampling (chosen + K non-chosen)
#   - deterministic seeds per segment
#
# Variants:
#   A) ZON  : global z-standardize destination vars on FULL TZ universe
#   B) ZOFF : no standardization (use raw LOG1P values as-is)
#
# For each variant and segment:
#   Sampled TRAIN metrics: NLL, MRR, McFadden_R2, rank_median, Top1/10/50/100
#   Sampled OOS   metrics: NLL, MRR, McFadden_R2, rank_median, Top1/10/50/100
#   FULL-choice-set metrics on OOS subset (all zones as alternatives):
#       NLL_full, MRR_full, McFadden_R2_full, rank_median_full
#       Top1/10/50/100_full
#       share_spearman_full, share_JS_full
#
# Outputs:
#   ModelRuns/ZStandardization/zstd_comparison_results_long.csv
#   ModelRuns/ZStandardization/zstd_comparison_results_wide.csv
#   ModelRuns/ZStandardization/zstd_comparison_deltas_ZON_minus_ZOFF.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path


# -----------------------------
# CONFIG (LIE + byPerson + LOG1P)
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TRIPS_DIR = "Trips/byPerson"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT   = "utility_emu"
MAP_NAME  = "NO"

OUT_DIR = "ModelRuns/ZStandardization"
os.makedirs(OUT_DIR, exist_ok=True)

K = 1000
BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

FULL_EVAL_MAX = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256
TOPKS = (1, 10, 50, 100)

LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",         
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

VARIANTS = {
    "ZON":  {"apply_z": True,  "suffix": "_z"},
    "ZOFF": {"apply_z": False, "suffix": ""},  # keep original LOG1P scale
}


# -----------------------------
# Small helpers
# -----------------------------
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300); q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)


# -----------------------------
# IO helpers
# -----------------------------
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df

def load_emu_matrix_and_mapping(omx_path: Path, emu_mat: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if emu_mat not in f.list_matrices():
        raise ValueError(f"Matrix '{emu_mat}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[emu_mat]
    print(f"[OMX] Loading {emu_mat} into RAM... shape={M.shape} (float32)")
    emu = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return emu, zone_to_idx, idx_to_zone


# -----------------------------
# Preprocess + design helpers
# -----------------------------
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts
    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_used: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_used]

    N = len(df)
    Pdim = len(feat_cols_used)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_used):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)  # chosen always first

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2
    )


# -----------------------------
# Model
# -----------------------------
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(1,10,50,100)):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = int(V.shape[1])
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

        out = {
            "NLL": nll,
            "MRR": mrr,
            "McFadden_R2": float(r2),
            "rank_median": rank_median,
            "J": J,
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))
    return out


# -----------------------------
# FULL-choice-set evaluation
# -----------------------------
def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    feat_cols_used: list[str],   # NOTE: here we pass either z-cols (ZON) or LOG1P raw (ZOFF)
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(1,10,50,100),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    X_all = tz_feat_seg[feat_cols_used].to_numpy(dtype=np.float32)  # (J, P)
    Aj = X_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    ll_model = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    top_num = {k: 0.0 for k in topKs}

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        nll_num += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    out = {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    return out


# ============================================================
# MAIN
# ============================================================

print("[LOAD] EMU matrix + mapping...")
emu_mat, zone_to_idx, idx_to_zone = load_emu_matrix_and_mapping(UTIL_OMX, EMU_MAT, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

print("[LOAD] TZ features ...")
tz0 = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)

need_cols = ["npvm_id"] + LOG1P_COLS
miss = [c for c in need_cols if c not in tz0.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

tz0 = tz0[["npvm_id"] + LOG1P_COLS + ["geometry"]].copy()
tz0["npvm_id"] = pd.to_numeric(tz0["npvm_id"], errors="coerce")
tz0 = tz0.dropna(subset=["npvm_id"]).copy()
tz0["npvm_id"] = tz0["npvm_id"].astype(int)

results = []

for variant_name, cfg in VARIANTS.items():
    apply_z = cfg["apply_z"]
    var_offset = 0 if variant_name == "ZON" else 9999

    print("\n" + "#" * 120)
    print(f"# VARIANT = {variant_name} | apply_z={apply_z} | K={K} | epochs={EPOCHS} | byPerson + LOG1P")
    print("#" * 120)

    tz_feat = tz0.copy()

    if apply_z:
        print("[PREP] Global z-standardize on full TZ universe...")
        tz_feat, z_cols, mu_all, sd_all = standardize_all_zones(tz_feat, LOG1P_COLS, suffix="_z")
        feat_cols_used = z_cols
    else:
        print("[PREP] No standardization (use LOG1P values as-is)...")
        feat_cols_used = LOG1P_COLS

    for seg in SEGMENTS:
        print("\n" + "=" * 100)
        print(f"Variant={variant_name} | Segment={seg} | K={K}")
        print("=" * 100)

        train_df = load_segment_csv(TRIPS_DIR, seg, "train")
        oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

        seed_tr = BASE_SEED + var_offset + 1000 * SEG_SEED[seg] + 10
        seed_te = BASE_SEED + var_offset + 1000 * SEG_SEED[seg] + 20

        emu_tr, X_tr, y_tr, w_tr, train_df2 = build_design(
            train_df, tz_feat, feat_cols_used, emu_mat, zone_to_idx, k=K, seed=seed_tr
        )
        emu_te, X_te, y_te, w_te, oos_df2 = build_design(
            oos_df, tz_feat, feat_cols_used, emu_mat, zone_to_idx, k=K, seed=seed_te
        )

        print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
        print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

        alpha_hat, beta_hat = train_mnl(
            emu_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
        )

        met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS)
        met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS)

        max_n = FULL_EVAL_MAX.get(seg, 8000)
        seed_full = FULL_EVAL_SEED + var_offset + 1000 * SEG_SEED[seg]
        full_met = full_choice_eval_oos_subset(
            oos_df=oos_df2,
            tz_feat_seg=tz_feat,
            feat_cols_used=feat_cols_used,
            emu_mat=emu_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=max_n,
            seed=seed_full,
            chunk=FULL_EVAL_CHUNK,
            topKs=TOPKS,
        )

        row = {
            "variant": variant_name,
            "segment": seg,
            "K": K,
            "alpha_emu": float(alpha_hat),

            "NLL_train": met_tr["NLL"],
            "MRR_train": met_tr["MRR"],
            "McFadden_R2_train": met_tr["McFadden_R2"],
            "rank_median_train": met_tr["rank_median"],
            "Top1_train": met_tr["Top1"],
            "Top10_train": met_tr["Top10"],
            "Top50_train": met_tr["Top50"],
            "Top100_train": met_tr["Top100"],

            "NLL_oos": met_te["NLL"],
            "MRR_oos": met_te["MRR"],
            "McFadden_R2_oos": met_te["McFadden_R2"],
            "rank_median_oos": met_te["rank_median"],
            "Top1_oos": met_te["Top1"],
            "Top10_oos": met_te["Top10"],
            "Top50_oos": met_te["Top50"],
            "Top100_oos": met_te["Top100"],
        }
        row.update(full_met)
        results.append(row)

        print("[SAMPLED TRAIN]",
              f"NLL={row['NLL_train']:.4f} MRR={row['MRR_train']:.4f} R2={row['McFadden_R2_train']:.4f} "
              f"Top1={row['Top1_train']:.3f} Top10={row['Top10_train']:.3f} rMed={row['rank_median_train']:.1f}")
        print("[SAMPLED OOS ]",
              f"NLL={row['NLL_oos']:.4f} MRR={row['MRR_oos']:.4f} R2={row['McFadden_R2_oos']:.4f} "
              f"Top1={row['Top1_oos']:.3f} Top10={row['Top10_oos']:.3f} rMed={row['rank_median_oos']:.1f}")

        if full_met:
            print("[FULL OOS   ]",
                  f"NLL_full={row['NLL_full']:.4f} MRR_full={row['MRR_full']:.4f} R2_full={row['McFadden_R2_full']:.4f} "
                  f"Top1_full={row['Top1_full']:.3f} Top10_full={row['Top10_full']:.3f} rMed_full={row['rank_median_full']:.1f}")
            print("[SHARES FULL ]",
                  f"spearman={row['share_spearman_full']:.4f} JS={row['share_JS_full']:.4f} "
                  f"N_full={row['N_full_eval']} J_full={row['J_full']}")

# -----------------------------
# Save outputs
# -----------------------------
res_df = pd.DataFrame(results).sort_values(["segment", "variant"]).reset_index(drop=True)

long_path = os.path.join(OUT_DIR, "zstd_comparison_results_long.csv")
res_df.to_csv(long_path, index=False)
print(f"\n[OK] wrote {long_path}")

value_cols = [c for c in res_df.columns if c not in ("variant", "segment")]
wide = res_df.pivot(index="segment", columns="variant", values=value_cols)
wide.columns = [f"{v}__{s}" for (v, s) in wide.columns]
wide = wide.reset_index()

wide_path = os.path.join(OUT_DIR, "zstd_comparison_results_wide.csv")
wide.to_csv(wide_path, index=False)
print(f"[OK] wrote {wide_path}")

# Deltas: ZON - ZOFF
delta = pd.DataFrame({"segment": wide["segment"]})
for col in value_cols:
    c_on = f"{col}__ZON"
    c_off = f"{col}__ZOFF"
    if c_on in wide.columns and c_off in wide.columns:
        delta[col + "__delta_ZON_minus_ZOFF"] = wide[c_on] - wide[c_off]

delta_path = os.path.join(OUT_DIR, "zstd_comparison_deltas_ZON_minus_ZOFF.csv")
delta.to_csv(delta_path, index=False)
print(f"[OK] wrote {delta_path}")

print("\n[DONE] Outputs in:", OUT_DIR)

## Density Based Variables

In [ ]:
# ============================================================
# ROBUSTNESS: Feature set comparison only (BASE vs DENS_SWAP)
# byPerson | with LIE | EMU | global Z
#
# Compare:
#   - BASE_LOG1P_COLS (13 vars)
#   - DENS_SWAP_COLS  (12 vars, count->density swap, drops POI_urban_dens)
#
# Fixed:
#   - K = 1000
#   - 50 epochs
#   - uniform sampling (chosen + K non-chosen)
#   - global z-standardization on FULL TZ universe (per feature set)
#   - sampled TRAIN/OOS metrics + FULL OOS metrics (subset)
#
# Outputs:
#   ModelRuns/FeatureRobustness/feature_set_compare_long.csv
#   ModelRuns/FeatureRobustness/feature_set_compare_wide.csv
#   ModelRuns/FeatureRobustness/feature_set_compare_deltas_DENS_minus_BASE.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path


# -----------------------------
# CONFIG (byPerson, with LIE)
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TRIPS_DIR = "Trips/byPerson"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT   = "utility_emu"
MAP_NAME  = "NO"

OUT_DIR = "ModelRuns/FeatureRobustness"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_SEED = 123
DEVICE = "cpu"
DTYPE  = torch.float32

# Fixed settings
K = 1000
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

TOPKS = (1, 10, 50, 100)

# FULL evaluation subsets (OOS)
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256
FULL_EVAL_MAX = {"YS": 100000, "OS": 100000, "YL": 100000, "OL": 100000}


# -----------------------------
# Feature lists
# -----------------------------
BASE_LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

DENS_SWAP_COLS = [
    "F8_gastr_count_dens",
    "F2_pop_total_log1p",
    "F8_cult_count_dens",
    "F8_sport_count_dens",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F8_others_count_dens",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
]


# -----------------------------
# Small helpers
# -----------------------------
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)


# -----------------------------
# IO helpers
# -----------------------------
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df

def load_emu_matrix_and_mapping(omx_path: Path, emu_mat: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if emu_mat not in f.list_matrices():
        raise ValueError(f"Matrix '{emu_mat}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[emu_mat]
    print(f"[OMX] Loading {emu_mat} into RAM... shape={M.shape} (float32)")
    emu = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return emu, zone_to_idx, idx_to_zone


# -----------------------------
# Preprocess + design helpers
# -----------------------------
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts
    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_used: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_used]

    N = len(df)
    Pdim = len(feat_cols_used)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_used):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2
    )


# -----------------------------
# Model
# -----------------------------
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(1,10,50,100)):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = int(V.shape[1])
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

        out = {
            "NLL": nll,
            "MRR": mrr,
            "McFadden_R2": float(r2),
            "rank_median": rank_median,
            "J": J,
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))
    return out


# -----------------------------
# FULL-choice-set evaluation
# -----------------------------
def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    feat_cols_used: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(1,10,50,100),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    X_all = tz_feat_seg[feat_cols_used].to_numpy(dtype=np.float32)
    Aj = X_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    ll_model = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    top_num = {k: 0.0 for k in topKs}

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        nll_num += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    out = {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    return out


# -----------------------------
# Runner
# -----------------------------
def run_spec(
    spec_name: str,
    base_tz: pd.DataFrame,
    feat_cols: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    K: int,
    EPOCHS: int,
    full_eval_max: dict,
    spec_seed_offset: int,
):
    tz_feat = base_tz.copy()

    miss = [c for c in (["npvm_id"] + feat_cols) if c not in tz_feat.columns]
    if miss:
        raise ValueError(f"[{spec_name}] TZ layer missing columns: {miss}")

    print(f"[{spec_name}] PREP: global z-standardize on full TZ universe...")
    tz_feat, z_cols, _, _ = standardize_all_zones(tz_feat, feat_cols, suffix="_z")

    rows = []
    for seg in SEGMENTS:
        print("\n" + "=" * 100)
        print(f"[{spec_name}] Segment={seg} | K={K} | epochs={EPOCHS}")
        print("=" * 100)

        train_df = load_segment_csv(TRIPS_DIR, seg, "train")
        oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

        seed_tr = BASE_SEED + spec_seed_offset + 1000 * SEG_SEED[seg] + 10
        seed_te = BASE_SEED + spec_seed_offset + 1000 * SEG_SEED[seg] + 20

        emu_tr, X_tr, y_tr, w_tr, train_df2 = build_design(
            train_df, tz_feat, z_cols, emu_mat, zone_to_idx, k=K, seed=seed_tr
        )
        emu_te, X_te, y_te, w_te, oos_df2 = build_design(
            oos_df, tz_feat, z_cols, emu_mat, zone_to_idx, k=K, seed=seed_te
        )

        alpha_hat, beta_hat = train_mnl(
            emu_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
        )

        met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS)
        met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS)

        max_n = full_eval_max.get(seg, None)
        seed_full = FULL_EVAL_SEED + spec_seed_offset + 1000 * SEG_SEED[seg]

        full_met = full_choice_eval_oos_subset(
            oos_df=oos_df2,
            tz_feat_seg=tz_feat,
            feat_cols_used=z_cols,
            emu_mat=emu_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=max_n,
            seed=seed_full,
            chunk=FULL_EVAL_CHUNK,
            topKs=TOPKS,
        )

        row = {
            "spec": spec_name,
            "segment": seg,
            "K": K,
            "P": len(feat_cols),
            "alpha_emu": float(alpha_hat),
            "NLL_train": met_tr["NLL"],
            "MRR_train": met_tr["MRR"],
            "McFadden_R2_train": met_tr["McFadden_R2"],
            "rank_median_train": met_tr["rank_median"],
            "Top1_train": met_tr["Top1"],
            "Top10_train": met_tr["Top10"],
            "Top50_train": met_tr["Top50"],
            "Top100_train": met_tr["Top100"],
            "NLL_oos": met_te["NLL"],
            "MRR_oos": met_te["MRR"],
            "McFadden_R2_oos": met_te["McFadden_R2"],
            "rank_median_oos": met_te["rank_median"],
            "Top1_oos": met_te["Top1"],
            "Top10_oos": met_te["Top10"],
            "Top50_oos": met_te["Top50"],
            "Top100_oos": met_te["Top100"],
        }
        row.update(full_met)
        rows.append(row)

    return rows


# ============================================================
# MAIN
# ============================================================

print("[LOAD] EMU matrix + mapping...")
emu_mat, zone_to_idx, idx_to_zone = load_emu_matrix_and_mapping(UTIL_OMX, EMU_MAT, MAP_NAME)

print("[LOAD] TZ features...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
if "npvm_id" not in tz.columns:
    raise ValueError("TZ layer missing 'npvm_id'")

tz["npvm_id"] = pd.to_numeric(tz["npvm_id"], errors="coerce")
tz = tz.dropna(subset=["npvm_id"]).copy()
tz["npvm_id"] = tz["npvm_id"].astype(int)

base_tz = tz.copy()

# ------------------------------------------------------------
# Run only BASE vs DENS_SWAP
# ------------------------------------------------------------
print("\n" + "#" * 120)
print("# FEATURE-SET COMPARISON ONLY (BASE vs DENS_SWAP)")
print("#" * 120)

rows_compare = []
rows_compare += run_spec(
    spec_name="BASE",
    base_tz=base_tz,
    feat_cols=BASE_LOG1P_COLS,
    emu_mat=emu_mat,
    zone_to_idx=zone_to_idx,
    idx_to_zone=idx_to_zone,
    K=K,
    EPOCHS=EPOCHS,
    full_eval_max=FULL_EVAL_MAX,
    spec_seed_offset=0,
)
rows_compare += run_spec(
    spec_name="DENS_SWAP",
    base_tz=base_tz,
    feat_cols=DENS_SWAP_COLS,
    emu_mat=emu_mat,
    zone_to_idx=zone_to_idx,
    idx_to_zone=idx_to_zone,
    K=K,
    EPOCHS=EPOCHS,
    full_eval_max=FULL_EVAL_MAX,
    spec_seed_offset=50000,
)

df_cmp = pd.DataFrame(rows_compare).sort_values(["segment", "spec"]).reset_index(drop=True)

cmp_long_path = os.path.join(OUT_DIR, "feature_set_compare_long.csv")
df_cmp.to_csv(cmp_long_path, index=False)
print(f"\n[OK] wrote {cmp_long_path}")

value_cols = [c for c in df_cmp.columns if c not in ("spec", "segment")]
wide = df_cmp.pivot(index="segment", columns="spec", values=value_cols)
wide.columns = [f"{v}__{s}" for (v, s) in wide.columns]
wide = wide.reset_index()

cmp_wide_path = os.path.join(OUT_DIR, "feature_set_compare_wide.csv")
wide.to_csv(cmp_wide_path, index=False)
print(f"[OK] wrote {cmp_wide_path}")

delta = pd.DataFrame({"segment": wide["segment"]})
for col in value_cols:
    a = f"{col}__DENS_SWAP"
    b = f"{col}__BASE"
    if a in wide.columns and b in wide.columns:
        delta[col + "__delta_DENS_minus_BASE"] = wide[a] - wide[b]

cmp_delta_path = os.path.join(OUT_DIR, "feature_set_compare_deltas_DENS_minus_BASE.csv")
delta.to_csv(cmp_delta_path, index=False)
print(f"[OK] wrote {cmp_delta_path}")

print("\n[DONE] Outputs in:", OUT_DIR)

## Only 4 variables

In [ ]:
# ============================================================
# BASELINE ESTIMATION + FULL REPORT + VCOV + COEFFICIENT TESTS
# (withLIE, byPerson, log1p, global z, EMU)
#
# ADDITIONS IN THIS VERSION:
#   - computes full Hessian-based approximate variance-covariance matrix per segment
#   - exports vcov matrix per segment
#   - exports coefficient table as before
#   - exports tests of coefficient differences:
#       * across segments, same parameter
#       * within segment, pairwise parameter differences
#
# IMPORTANT:
#   - coefficients are STILL estimated exactly as before by weighted MNL training
#   - Hessian is only used post-estimation for approximate inference
#   - across-segment difference tests assume segment-specific estimates are independent
# ============================================================

import os
import math
import random
import itertools
import numpy as np
import pandas as pd
import geopandas as gpd
import tables as tb

import torch
import openmatrix as omx
from pathlib import Path

import matplotlib.pyplot as plt


# -----------------------------
# CONFIG (withLIE + byPerson)
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"
DIST_COL  = "dist_km"

TRIPS_DIR = "Trips/byPerson"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

UTIL_OMX      = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME  = "utility_emu"

DIST_OMX      = READY_DIR / "distance_avg_2023_ready.omx"
DIST_MAT_NAME = "avg_distances_ready"

MAP_NAME   = "NO"

OUT_DIR = "ModelRuns/BaselineFinal_less_var"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DIST_OOS_DIR = os.path.join(OUT_DIR, "distance_oos_triplevel")
os.makedirs(DIST_OOS_DIR, exist_ok=True)

RESULT_DIR = os.path.join(OUT_DIR, "Result")
os.makedirs(RESULT_DIR, exist_ok=True)

# NEW OUTPUTS
VCOV_DIR = os.path.join(OUT_DIR, "vcov")
os.makedirs(VCOV_DIR, exist_ok=True)

TEST_DIR = os.path.join(OUT_DIR, "coef_tests")
os.makedirs(TEST_DIR, exist_ok=True)

ATTR_CSV_OUT  = os.path.join(RESULT_DIR, "Attractivity.csv")
ATTR_GPKG_OUT = os.path.join(RESULT_DIR, "Attractivity.gpkg")
UTIL_SEG_OMX  = os.path.join(RESULT_DIR, "utilities_by_segment.omx")

K = 1000
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

FULL_EVAL_MAX = {
    "YS": 20000,
    "OS": 8000,
    "YL": 8000,
    "OL": 8000,
}
FULL_EVAL_SEED  = 777
FULL_EVAL_CHUNK = 256

SE_MAX_TRAIN = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
SE_SEED = 2024

BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

TOPKS_SAMPLED = (5, 10)
TOPKS_FULL    = (5, 10)

BASE_COLS = [
    "F1_gastr_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F10_superinfra_log1p",
]

PARAM_NAMES = ["alpha_emu"] + BASE_COLS


# ============================================================
# Global reproducibility helper
# ============================================================
def set_all_seeds(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Small helpers
# ============================================================
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300); q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def p_value_two_sided_z(z):
    return 2.0 * (1.0 - norm_cdf(abs(float(z))))

def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.1:
        return "."
    return ""

def safe_sqrt(x):
    return float(np.sqrt(max(float(x), 0.0)))


# ============================================================
# IO helpers
# ============================================================
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()

    if DIST_COL in df.columns:
        df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")

    return df

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


# ============================================================
# Preprocess helpers
# ============================================================
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0]  = c
        dest_set[i, 1:] = alts
    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]
    dest_set2 = dest_set[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
        dest_set2,
        orig_idx2,
        dest_idx2,
    )


# ============================================================
# Model
# ============================================================
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)

            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum()

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted-avg NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(5,10), want_ff=True):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        P = torch.exp(logP)

        chosen_logp = logP[:, 0]
        chosen_p    = P[:, 0]
        sum_w = float(w.sum().cpu())

        nll_sum = float((-(w * chosen_logp)).sum().cpu())
        nll_avg = nll_sum / (sum_w + 1e-9)

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))

        out = {
            "NLL_sum": nll_sum,
            "NLL_avg": nll_avg,
            "MRR": mrr,
            "sum_WP": sum_w,
            "N": int(V.shape[0]),
            "J": int(V.shape[1]),
        }

        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))

        if want_ff:
            out["FF"] = float((w * chosen_p).sum().cpu() / (sum_w + 1e-9))

    return out


# ============================================================
# FULL-choice-set evaluation on an OOS subset
# ============================================================
def full_eval_and_shares_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    dist_mat: np.ndarray | None,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(5,10),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w    = df[WP_COL].to_numpy(dtype=np.float64)

    has_dist = (DIST_COL in df.columns) and df[DIST_COL].notna().any()
    obs_dist = df[DIST_COL].to_numpy(dtype=np.float64) if has_dist else None

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    if has_dist:
        good = good & np.isfinite(obs_dist) & (obs_dist >= 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w        = w[good]
    if has_dist:
        obs_dist = obs_dist[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}, None, (None, None), None

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass  = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    ll_model = 0.0
    mrr_num = 0.0
    ff_num = 0.0
    top_num = {k: 0.0 for k in topKs}
    ranks   = np.empty(len(df), dtype=np.int64)

    exp_dist = np.zeros(len(df), dtype=np.float64) if (dist_mat is not None and has_dist) else None

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse
        p_chosen = math.exp(logp_chosen)

        nll_num  += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)
        ff_num   += wn * p_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        if exp_dist is not None:
            expd = 0.0

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

            if exp_dist is not None:
                dist_blk = dist_mat[o, j0:j1].astype(np.float64)
                expd += float(np.sum(Pblk * dist_blk))

        if exp_dist is not None:
            exp_dist[n] = expd

    nll_full = float(nll_num)
    nll_full_avg = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    ff_full = float(ff_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs  = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)

    pear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="pearson")) if len(S_obs) > 2 else np.nan
    spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    mae  = float(np.mean(np.abs(S_obs - S_pred)))
    rmse = float(np.sqrt(np.mean((S_obs - S_pred) ** 2)))
    js   = float(js_div(S_obs, S_pred))

    dist_summary = {}
    if exp_dist is not None:
        wnorm = w / (sum_w + 1e-12)
        obs_mean = float(np.sum(wnorm * obs_dist))
        pred_mean = float(np.sum(wnorm * exp_dist))
        dist_summary = {
            "dist_obs_mean_km": obs_mean,
            "dist_pred_mean_km": pred_mean,
            "dist_mean_diff_km": float(pred_mean - obs_mean),
            "dist_obs_p50_km": float(np.quantile(obs_dist, 0.50)),
            "dist_pred_p50_km": float(np.quantile(exp_dist, 0.50)),
            "dist_obs_p90_km": float(np.quantile(obs_dist, 0.90)),
            "dist_pred_p90_km": float(np.quantile(exp_dist, 0.90)),
        }

    out = {
        "NLL_full_sum": nll_full,
        "NLL_full_avg": nll_full_avg,
        "MRR_full": mrr_full,
        "FF_full": ff_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Pearson_share": pear,
        "Spearman_share": spear,
        "MAE_share": mae,
        "RMSE_share": rmse,
        "JS_share": js,
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    out.update(dist_summary)

    trip_distance_df = None
    if exp_dist is not None:
        trip_distance_df = pd.DataFrame({
            "orig_zone": df[ORIG_COL].to_numpy(dtype=int),
            "dest_zone_obs": df[DEST_COL].to_numpy(dtype=int),
            "WP": w.astype(np.float64),
            "dist_obs_km": obs_dist.astype(np.float64),
            "dist_exp_km": exp_dist.astype(np.float64),
        })

    return out, {
        "tz_ids": all_zone_ids,
        "S_obs": S_obs,
        "S_pred": S_pred,
        "Aj": Aj.astype(np.float64),
        "orig_idx": orig_idx,
        "w": w,
    }, (obs_dist, exp_dist), trip_distance_df


# ============================================================
# Hessian-based approximate inference
# ============================================================
def approx_hessian_inference(
    emu_set: torch.Tensor,
    X_set: torch.Tensor,
    y: torch.Tensor,
    w: torch.Tensor,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int,
    seed: int,
    ridge: float = 1e-8,
):
    """
    Returns:
      vcov_sum : approximate covariance matrix of theta_hat
      se       : sqrt(diag(vcov_sum))
      H_sum    : Hessian of weighted SUM objective
      used_n   : number of observations used
      used_sum_w : total weight used in Hessian approximation

    theta = [alpha_emu, beta_1, ..., beta_P]
    """
    N = emu_set.shape[0]
    rng = np.random.default_rng(seed)

    if N > max_n:
        idx = rng.choice(N, size=max_n, replace=False)
        idx = torch.tensor(idx, dtype=torch.long, device=DEVICE)
        emu = emu_set[idx]
        X   = X_set[idx]
        yy  = y[idx]
        ww  = w[idx]
    else:
        emu, X, yy, ww = emu_set, X_set, y, w

    used_n = int(emu.shape[0])
    used_sum_w = float(ww.sum().detach().cpu())

    Pdim = X.shape[2]

    alpha = torch.tensor([alpha_hat], dtype=DTYPE, device=DEVICE, requires_grad=True)
    beta  = torch.tensor(beta_hat, dtype=DTYPE, device=DEVICE, requires_grad=True)

    def loss_avg(a, b):
        V = a * emu + torch.einsum("bjp,p->bj", X, b)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        ll = logP[torch.arange(V.shape[0], device=DEVICE), yy]
        return (-(ww * ll)).sum() / (ww.sum() + 1e-12)

    L = loss_avg(alpha, beta)
    g = torch.autograd.grad(L, [alpha, beta], create_graph=True)
    g_vec = torch.cat([g[0].reshape(1), g[1].reshape(-1)], dim=0)

    H_avg = torch.zeros((1 + Pdim, 1 + Pdim), dtype=torch.float64, device=DEVICE)
    for i in range(1 + Pdim):
        gi = g_vec[i]
        hi = torch.autograd.grad(gi, [alpha, beta], retain_graph=True)
        hi_vec = torch.cat([hi[0].reshape(1), hi[1].reshape(-1)], dim=0)
        H_avg[i, :] = hi_vec.detach().to(torch.float64)

    H_avg = H_avg.cpu().numpy()
    H_sum = used_sum_w * H_avg

    H_sum = 0.5 * (H_sum + H_sum.T)
    H_sum = H_sum + ridge * np.eye(H_sum.shape[0], dtype=np.float64)

    vcov = np.linalg.pinv(H_sum)
    vcov = 0.5 * (vcov + vcov.T)

    se = np.sqrt(np.clip(np.diag(vcov), 0.0, np.inf))
    return vcov, se, H_sum, used_n, used_sum_w


# ============================================================
# Coefficient-difference tests
# ============================================================
def build_within_segment_diff_tests(segment, params, coef_vec, vcov):
    rows = []
    for i in range(len(params)):
        for j in range(i + 1, len(params)):
            p1 = params[i]
            p2 = params[j]
            delta = float(coef_vec[i] - coef_vec[j])
            var_delta = float(vcov[i, i] + vcov[j, j] - 2.0 * vcov[i, j])
            se_delta = safe_sqrt(var_delta)
            z_delta = delta / (se_delta + 1e-12)
            p_delta = p_value_two_sided_z(z_delta)
            rows.append({
                "segment": segment,
                "param_1": p1,
                "param_2": p2,
                "coef_1": float(coef_vec[i]),
                "coef_2": float(coef_vec[j]),
                "delta_1_minus_2": delta,
                "se_delta": se_delta,
                "z_delta": float(z_delta),
                "p_delta": float(p_delta),
                "stars_delta": stars(p_delta),
            })
    return rows

def build_across_segment_sameparam_tests(seg_to_coef, seg_to_vcov, params):
    rows = []
    seg_pairs = list(itertools.combinations(sorted(seg_to_coef.keys()), 2))
    for param_idx, param in enumerate(params):
        for s1, s2 in seg_pairs:
            b1 = float(seg_to_coef[s1][param_idx])
            b2 = float(seg_to_coef[s2][param_idx])

            # assuming independence across segment-specific estimates
            var_delta = float(seg_to_vcov[s1][param_idx, param_idx] + seg_to_vcov[s2][param_idx, param_idx])
            se_delta = safe_sqrt(var_delta)
            delta = b1 - b2
            z_delta = delta / (se_delta + 1e-12)
            p_delta = p_value_two_sided_z(z_delta)

            rows.append({
                "param": param,
                "segment_1": s1,
                "segment_2": s2,
                "coef_1": b1,
                "coef_2": b2,
                "delta_1_minus_2": float(delta),
                "se_delta": se_delta,
                "z_delta": float(z_delta),
                "p_delta": float(p_delta),
                "stars_delta": stars(p_delta),
                "assumption": "independent_segment_estimates",
            })
    return rows


# ============================================================
# Destination-level summaries and plots
# ============================================================
def incoming_accessibility_logsum(
    tz_ids: np.ndarray,
    tz_pos: dict,
    emu_mat: np.ndarray,
    alpha: float,
    orig_idx: np.ndarray,
    w: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    chunk: int = 256,
):
    Msize = emu_mat.shape[0]
    w_by_o = np.zeros(Msize, dtype=np.float64)
    for o, wn in zip(orig_idx, w):
        w_by_o[int(o)] += float(wn)

    origins = np.where(w_by_o > 0)[0]
    w_o = w_by_o[origins]
    w_o = w_o / (w_o.sum() + 1e-12)

    L = np.full(len(tz_ids), np.nan, dtype=np.float64)

    tz_omx = np.array([zone_to_idx.get(int(z), -1) for z in tz_ids], dtype=int)
    good = (tz_omx >= 0)
    tz_omx_good = tz_omx[good]

    good_pos = np.where(good)[0]
    for local_idx, j in enumerate(tz_omx_good):
        vals = alpha * emu_mat[origins, j].astype(np.float64)
        m = np.max(vals)
        s = np.sum(w_o * np.exp(vals - m))
        L[good_pos[local_idx]] = float(m + np.log(s + 1e-300))

    return L

def plot_distance_plausibility(obs_dist, exp_dist, seg, out_png):
    obs_dist = np.asarray(obs_dist, dtype=np.float64)
    exp_dist = np.asarray(exp_dist, dtype=np.float64)

    finite = np.isfinite(obs_dist) & np.isfinite(exp_dist)
    obs_dist = obs_dist[finite]
    exp_dist = exp_dist[finite]

    if len(obs_dist) == 0:
        return

    obs_mean = float(np.mean(obs_dist))
    exp_mean = float(np.mean(exp_dist))

    xmax = float(max(np.max(obs_dist), np.max(exp_dist)))
    bins = np.linspace(0.0, xmax, 51)

    plt.figure()
    plt.hist(obs_dist, bins=bins, density=True, alpha=0.6,
             label=f"Observed dist_km (mean={obs_mean:.2f})")
    plt.hist(exp_dist, bins=bins, density=True, alpha=0.6,
             label=f"Predicted E[distance] (mean={exp_mean:.2f})")
    plt.xlabel("Distance (km)")
    plt.ylabel("Density")
    plt.title(f"Distance plausibility | {seg} | obs mean={obs_mean:.2f}, pred mean={exp_mean:.2f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_attr_vs_access(Aj, L_in, S_pred, seg, out_png):
    plt.figure()
    s = 20 + 4000 * (S_pred / (S_pred.max() + 1e-12))
    plt.scatter(L_in, Aj, s=s, alpha=0.5)
    plt.xlabel("Incoming accessibility logsum (from origins, using EMU)")
    plt.ylabel("Attractivity index A_j = X_j beta (z-scale)")
    plt.title(f"Attractivity vs incoming accessibility | {seg}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_share_maps(tz_gdf, col, seg, out_png, title):
    plt.figure()
    ax = tz_gdf.plot(column=col, legend=True)
    ax.set_axis_off()
    plt.title(f"{title} | {seg}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


# ============================================================
# MAIN
# ============================================================
set_all_seeds(BASE_SEED)

print("[LOAD] TZ features (withLIE)...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
if "npvm_id" not in tz.columns:
    raise ValueError("TZ layer missing 'npvm_id'")
tz["npvm_id"] = pd.to_numeric(tz["npvm_id"], errors="coerce")
tz = tz.dropna(subset=["npvm_id"]).copy()
tz["npvm_id"] = tz["npvm_id"].astype(int)

need_cols = ["npvm_id"] + BASE_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

print("[PREP] Global z-standardize destination variables on FULL TZ universe...")
tz_feat = tz.copy()
tz_feat, Z_COLS, MU_ALL, SD_ALL = standardize_all_zones(tz_feat, BASE_COLS, suffix="_z")

print("[LOAD] EMU matrix + mapping (withLIE)...")
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

print("[LOAD] Average distance matrix (for distance plausibility)...")
dist_mat, zone_to_idx_dist, idx_to_zone_dist = load_matrix_and_mapping(DIST_OMX, DIST_MAT_NAME, MAP_NAME)
if len(idx_to_zone_dist) != len(idx_to_zone) or not np.all(idx_to_zone_dist == idx_to_zone):
    print("[WARN] Distance OMX mapping differs from utilities OMX mapping. Distance plausibility may be misaligned.")

rows_sampled = []
rows_full    = []
rows_coef    = []

seg_alpha = {}
seg_beta  = {}
seg_Aj_tz = {}
seg_Aj_by_omx = {}

# NEW
seg_vcov = {}
seg_se   = {}
seg_coef_vec = {}
seg_hsum = {}
seg_inference_meta = {}

tz_ids_master = np.array(sorted(tz_feat["npvm_id"].astype(int).unique().tolist()), dtype=int)

GPKG_OUT = os.path.join(OUT_DIR, "tz_outputs.gpkg")
if os.path.exists(GPKG_OUT):
    os.remove(GPKG_OUT)

for seg in SEGMENTS:
    print("\n" + "#" * 120)
    print(f"# SEGMENT = {seg} | K={K} | epochs={EPOCHS}")
    print("#" * 120)

    set_all_seeds(BASE_SEED + 1000 * SEG_SEED[seg])

    train_df = load_segment_csv(TRIPS_DIR, seg, "train")
    oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

    seed_tr = BASE_SEED + 1000 * SEG_SEED[seg] + 10
    seed_te = BASE_SEED + 1000 * SEG_SEED[seg] + 20
    seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg]

    emu_tr, X_tr, y_tr, w_tr, train_df2, destset_tr, origidx_tr, destidx_tr = build_design(
        train_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_tr
    )
    emu_te, X_te, y_te, w_te, oos_df2, destset_te, origidx_te, destidx_te = build_design(
        oos_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_te
    )

    print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
    print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

    alpha_hat, beta_hat = train_mnl(
        emu_tr, X_tr, y_tr, w_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
    )

    seg_alpha[seg] = float(alpha_hat)
    seg_beta[seg]  = beta_hat.copy()

    coef_vec = np.concatenate([[float(alpha_hat)], beta_hat.astype(np.float64)])
    seg_coef_vec[seg] = coef_vec.copy()

    Xz_all_master = tz_feat.set_index("npvm_id").loc[tz_ids_master, Z_COLS].to_numpy(dtype=np.float32)
    Aj_master = Xz_all_master @ beta_hat.astype(np.float32)
    seg_Aj_tz[seg] = Aj_master.astype(np.float64)

    Msize = emu_mat.shape[0]
    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(tz_ids_master, Aj_master):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)
    seg_Aj_by_omx[seg] = Aj_by_omxidx

    met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)
    met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)

    rows_sampled.append({
        "segment": seg,
        "split": "train",
        "alpha_emu": float(alpha_hat),
        "K": K,
        **met_tr,
    })
    rows_sampled.append({
        "segment": seg,
        "split": "oos",
        "alpha_emu": float(alpha_hat),
        "K": K,
        **met_te,
    })

    print("[SAMPLED OOS ]",
          f"NLL_sum={met_te['NLL_sum']:.2f} NLL_avg={met_te['NLL_avg']:.4f} "
          f"MRR={met_te['MRR']:.4f} Top5={met_te['Top5']:.3f} Top10={met_te['Top10']:.3f} FF={met_te['FF']:.4f}")

    max_n = FULL_EVAL_MAX.get(seg, None)
    full_out, share_pack, dist_pack, trip_distance_df = full_eval_and_shares_oos_subset(
        oos_df=oos_df2,
        tz_feat=tz_feat,
        z_cols=Z_COLS,
        emu_mat=emu_mat,
        dist_mat=dist_mat,
        zone_to_idx=zone_to_idx,
        idx_to_zone=idx_to_zone,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=max_n,
        seed=seed_full,
        chunk=FULL_EVAL_CHUNK,
        topKs=TOPKS_FULL,
    )

    rows_full.append({
        "segment": seg,
        "alpha_emu": float(alpha_hat),
        "K": K,
        **full_out,
    })

    print("[FULL OOS   ]",
          f"NLL_full_avg={full_out['NLL_full_avg']:.4f} MRR_full={full_out['MRR_full']:.4f} "
          f"FF_full={full_out['FF_full']:.6f} "
          f"R2_full={full_out['McFadden_R2_full']:.4f} Top5_full={full_out['Top5_full']:.3f} Top10_full={full_out['Top10_full']:.3f}")
    print("[SHARES FULL]",
          f"Pear={full_out['Pearson_share']:.4f} Spear={full_out['Spearman_share']:.4f} "
          f"MAE={full_out['MAE_share']:.6f} RMSE={full_out['RMSE_share']:.6f} JS={full_out['JS_share']:.6f}")

    obs_dist, exp_dist = dist_pack
    if (obs_dist is not None) and (exp_dist is not None):
        png = os.path.join(PLOT_DIR, f"distance_plausibility_{seg}.png")
        plot_distance_plausibility(obs_dist, exp_dist, seg, png)
        print(f"[PLOT] wrote {png}")

        if trip_distance_df is not None:
            dist_csv = os.path.join(DIST_OOS_DIR, f"distance_triplevel_{seg}.csv")
            trip_distance_df.to_csv(dist_csv, index=False)
            print(f"[CSV ] wrote {dist_csv}")

    # =======================================================
    # Hessian-based full inference objects
    # =======================================================
    vcov_mat, se_vec, H_sum, used_n, used_sum_w = approx_hessian_inference(
        emu_set=emu_tr,
        X_set=X_tr,
        y=y_tr,
        w=w_tr,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=SE_MAX_TRAIN.get(seg, 20000),
        seed=SE_SEED + 1000 * SEG_SEED[seg],
        ridge=1e-8,
    )

    seg_vcov[seg] = vcov_mat.copy()
    seg_se[seg]   = se_vec.copy()
    seg_hsum[seg] = H_sum.copy()
    seg_inference_meta[seg] = {
        "segment": seg,
        "used_n_for_hessian": used_n,
        "used_sum_w_for_hessian": used_sum_w,
        "num_params": len(PARAM_NAMES),
    }

    # export vcov and hessian
    df_vcov = pd.DataFrame(vcov_mat, index=PARAM_NAMES, columns=PARAM_NAMES)
    df_hsum = pd.DataFrame(H_sum, index=PARAM_NAMES, columns=PARAM_NAMES)
    df_vcov.to_csv(os.path.join(VCOV_DIR, f"vcov_{seg}.csv"))
    df_hsum.to_csv(os.path.join(VCOV_DIR, f"hessian_sum_{seg}.csv"))

    # export per-segment diagonal summary
    df_diag = pd.DataFrame({
        "segment": seg,
        "param": PARAM_NAMES,
        "coef": coef_vec,
        "se": se_vec,
        "var": np.diag(vcov_mat),
    })
    df_diag.to_csv(os.path.join(VCOV_DIR, f"coef_se_diag_{seg}.csv"), index=False)

    # coefficient table rows
    for name, b, s in zip(PARAM_NAMES, coef_vec.tolist(), se_vec.tolist()):
        zval = b / (s + 1e-12)
        pval = p_value_two_sided_z(zval)
        rows_coef.append({
            "segment": seg,
            "param": name,
            "coef": float(b),
            "se": float(s),
            "z": float(zval),
            "p": float(pval),
            "stars": stars(pval),
        })

    if share_pack is not None:
        tz_ids  = share_pack["tz_ids"]
        S_obs   = share_pack["S_obs"]
        S_pred  = share_pack["S_pred"]
        Aj      = share_pack["Aj"]
        orig_i  = share_pack["orig_idx"]
        w_i     = share_pack["w"]

        tz_pos = {int(z): i for i, z in enumerate(tz_ids.tolist())}

        L_in = incoming_accessibility_logsum(
            tz_ids=tz_ids,
            tz_pos=tz_pos,
            emu_mat=emu_mat,
            alpha=float(alpha_hat),
            orig_idx=orig_i,
            w=w_i,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            chunk=FULL_EVAL_CHUNK,
        )

        png = os.path.join(PLOT_DIR, f"attr_vs_access_{seg}.png")
        plot_attr_vs_access(Aj, L_in, S_pred, seg, png)
        print(f"[PLOT] wrote {png}")

        tz_out = tz_feat[["npvm_id", "geometry"]].copy()
        tz_out = tz_out.set_index("npvm_id").loc[tz_ids].reset_index()

        tz_out[f"A_attr_{seg}"] = Aj
        tz_out[f"S_obs_{seg}"]  = S_obs
        tz_out[f"S_pred_{seg}"] = S_pred
        tz_out[f"L_in_{seg}"]   = L_in

        png1 = os.path.join(PLOT_DIR, f"map_S_pred_{seg}.png")
        png2 = os.path.join(PLOT_DIR, f"map_A_attr_{seg}.png")
        plot_share_maps(tz_out, f"S_pred_{seg}", seg, png1, "Predicted destination share (OOS subset, full denom)")
        plot_share_maps(tz_out, f"A_attr_{seg}", seg, png2, "Attractivity index A_j = X_j beta (z-scale)")
        print(f"[PLOT] wrote {png1}")
        print(f"[PLOT] wrote {png2}")

        layer_name = f"TZ_{seg}"
        tz_out.to_file(GPKG_OUT, layer=layer_name, driver="GPKG")
        print(f"[GPKG] wrote layer {layer_name} -> {GPKG_OUT}")


# ============================================================
# Save summary tables
# ============================================================
df_sampled = pd.DataFrame(rows_sampled)
df_full    = pd.DataFrame(rows_full)
df_coef    = pd.DataFrame(rows_coef)
df_infmeta = pd.DataFrame(list(seg_inference_meta.values()))

p1 = os.path.join(OUT_DIR, "metrics_sampled.csv")
p2 = os.path.join(OUT_DIR, "metrics_full_and_shares.csv")
p3 = os.path.join(OUT_DIR, "coef_table.csv")
p4 = os.path.join(OUT_DIR, "hessian_inference_meta.csv")

df_sampled.to_csv(p1, index=False)
df_full.to_csv(p2, index=False)
df_coef.to_csv(p3, index=False)
df_infmeta.to_csv(p4, index=False)

# ============================================================
# NEW: coefficient difference tests
# ============================================================
rows_within = []
for seg in SEGMENTS:
    rows_within.extend(
        build_within_segment_diff_tests(
            segment=seg,
            params=PARAM_NAMES,
            coef_vec=seg_coef_vec[seg],
            vcov=seg_vcov[seg],
        )
    )
df_within = pd.DataFrame(rows_within)
df_within.to_csv(os.path.join(TEST_DIR, "coef_diff_within_segment.csv"), index=False)

rows_across = build_across_segment_sameparam_tests(
    seg_to_coef=seg_coef_vec,
    seg_to_vcov=seg_vcov,
    params=PARAM_NAMES,
)
df_across = pd.DataFrame(rows_across)
df_across.to_csv(os.path.join(TEST_DIR, "coef_diff_across_segments.csv"), index=False)

# helpful wide summary for same parameter across all segments
wide_coef = df_coef.pivot(index="param", columns="segment", values="coef").reset_index()
wide_se   = df_coef.pivot(index="param", columns="segment", values="se").reset_index()
wide_coef.to_csv(os.path.join(TEST_DIR, "coef_wide_by_segment.csv"), index=False)
wide_se.to_csv(os.path.join(TEST_DIR, "se_wide_by_segment.csv"), index=False)

print("\n" + "=" * 110)
print("[OK] wrote", p1)
print("[OK] wrote", p2)
print("[OK] wrote", p3)
print("[OK] wrote", p4)
print("[OK] wrote", GPKG_OUT)
print("[OK] vcov matrices in", VCOV_DIR)
print("[OK] coefficient tests in", TEST_DIR)
print("[OK] plots in", PLOT_DIR)
print("[OK] trip-level distance CSVs in", DIST_OOS_DIR)
print("=" * 110)

print("\nSAMPLED OOS METRICS (weighted):")
print(df_sampled[df_sampled["split"] == "oos"][["segment","NLL_sum","NLL_avg","MRR","Top5","Top10","FF","N","J","sum_WP"]]
      .sort_values("segment").to_string(index=False))

print("\nFULL OOS METRICS + SHARE DIAGNOSTICS:")
keep_cols = [
    "segment","NLL_full_sum","NLL_full_avg","MRR_full","FF_full","McFadden_R2_full","Top5_full","Top10_full",
    "Pearson_share","Spearman_share","MAE_share","RMSE_share","JS_share","N_full_eval","WP_full_eval","J_full",
    "dist_obs_mean_km","dist_pred_mean_km","dist_mean_diff_km","dist_obs_p50_km","dist_pred_p50_km","dist_obs_p90_km","dist_pred_p90_km",
]
cols_exist = [c for c in keep_cols if c in df_full.columns]
print(df_full[cols_exist].sort_values("segment").to_string(index=False))

print("\nTOP OF coef_diff_across_segments.csv")
print(df_across.head(20).to_string(index=False))

# ============================================================
# EXPORT "Result" PACKAGE FOR SIMBA INTEGRATION
# ============================================================
print("\n" + "=" * 110)
print("[EXPORT] Building Result package (Attractivity + utilities omx) ...")

omx_idx_master = np.array([zone_to_idx.get(int(z), -1) for z in tz_ids_master], dtype=int)

attr_df = pd.DataFrame({
    "zone_id": tz_ids_master.astype(int),
    "omx_idx": omx_idx_master.astype(int),
    "Attr_YS": seg_Aj_tz["YS"].astype(np.float64),
    "Attr_OS": seg_Aj_tz["OS"].astype(np.float64),
    "Attr_YL": seg_Aj_tz["YL"].astype(np.float64),
    "Attr_OL": seg_Aj_tz["OL"].astype(np.float64),
})

attr_df.to_csv(ATTR_CSV_OUT, index=False)
print("[OK] wrote", ATTR_CSV_OUT)

if os.path.exists(ATTR_GPKG_OUT):
    os.remove(ATTR_GPKG_OUT)

geom = tz_feat[["npvm_id", "geometry"]].drop_duplicates().set_index("npvm_id").loc[tz_ids_master].reset_index()
geom = geom.rename(columns={"npvm_id": "zone_id"})
attr_gdf = gpd.GeoDataFrame(
    geom.merge(attr_df, on="zone_id", how="left"),
    geometry="geometry",
    crs=tz_feat.crs,
)
attr_gdf.to_file(ATTR_GPKG_OUT, driver="GPKG")
print("[OK] wrote", ATTR_GPKG_OUT)

if os.path.exists(UTIL_SEG_OMX):
    os.remove(UTIL_SEG_OMX)

Msize = emu_mat.shape[0]
print(f"[OMX-OUT] Creating {UTIL_SEG_OMX} with 4 matrices of shape {emu_mat.shape} (float32) ...")

fout = omx.open_file(UTIL_SEG_OMX, "w")
try:
    fout.create_mapping(MAP_NAME, idx_to_zone.astype(np.int32))
except Exception:
    fout.create_mapping(MAP_NAME, [int(x) for x in idx_to_zone.tolist()])

for seg in SEGMENTS:
    float_atom = tb.Atom.from_dtype(np.dtype("float32"))
    fout.create_matrix(f"utility_{seg}", atom=float_atom, shape=(Msize, Msize))

ROW_BLOCK = 512
for seg in SEGMENTS:
    alpha32 = np.float32(seg_alpha[seg])
    Aj_omx = seg_Aj_by_omx[seg].astype(np.float32)

    print(f"[OMX-OUT] Writing utility_{seg} ... alpha={float(seg_alpha[seg]):.6f}")
    mat = fout[f"utility_{seg}"]

    for i0 in range(0, Msize, ROW_BLOCK):
        i1 = min(Msize, i0 + ROW_BLOCK)
        emu_blk = emu_mat[i0:i1, :]
        util_blk = alpha32 * emu_blk + Aj_omx[None, :]
        mat[i0:i1, :] = util_blk.astype(np.float32)

fout.close()
print("[OK] wrote", UTIL_SEG_OMX)

print("[EXPORT] Done. Result files are in:", RESULT_DIR)
print("=" * 110)

# No visits

In [ ]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path


# ============================================================
# ROBUSTNESS: exclude leisure_cat == "Visits" BEFORE split,
# then estimate FULL vs NO_VISITS (byPerson only)
# ============================================================

# ============================================================
# CONFIG
# ============================================================
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

# raw EMU input tables
INPUT_GLOB = "Trips/destinations_*.csv"

# existing FULL split (already used in baseline)
FULL_SPLIT_DIR = "Trips/byPerson"

# new split to create after removing Visits
NOVISITS_SPLIT_DIR = "Trips/byPerson_noVisits"

# model inputs (same as baseline, with LIE)
TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT   = "utility_emu"
MAP_NAME  = "NO"

OUT_DIR = "ModelRuns/NoVisitsComparison"
os.makedirs(OUT_DIR, exist_ok=True)

# split settings
TARGET_OOS_SHARE = 0.20
SEED_PERSON = 43

# training settings
K = 1000
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

FULL_EVAL_MAX = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256
TOPKS = (1, 10, 50, 100)

# column candidates
ORIG_COL_CANDIDATES    = ["orig_zone"]
DEST_COL_CANDIDATES    = ["dest_zone"]
WEIGHT_COL_CANDIDATES  = ["WP"]
PERSON_COL_CANDIDATES  = ["HHNR"]
SURVEY_COL_CANDIDATES  = ["survey"]
LEISURE_COL_CANDIDATES = ["leisure_cat"]

# baseline destination-side variables
LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]


# ============================================================
# HELPERS
# ============================================================
def pick_col(df: pd.DataFrame, candidates: list[str], label: str) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(
        f"Could not find a '{label}' column. Tried: {candidates}. "
        f"Available: {list(df.columns)[:60]}"
    )

def get_weight_col(df: pd.DataFrame) -> str:
    wcol = pick_col(df, WEIGHT_COL_CANDIDATES, "weight (WP)")
    df[wcol] = pd.to_numeric(df[wcol], errors="coerce")
    if df[wcol].isna().all():
        raise ValueError(f"Weight column '{wcol}' is all-NaN after numeric conversion.")
    return wcol

def segment_from_filename(path: str) -> str:
    base = os.path.basename(path)
    stem = base.replace(".csv", "")
    parts = stem.split("_")
    return parts[-1]

def weighted_group_holdout(
    df: pd.DataFrame,
    group_cols: list[str],
    weight_col: str,
    target_w: float,
    seed: int,
):
    """
    Assign whole groups to OOS until cumulative group-weight reaches or exceeds target_w.
    """
    if len(df) == 0:
        return np.zeros(0, dtype=bool), 0.0

    g = (
        df.groupby(group_cols, dropna=False)[weight_col]
          .sum()
          .reset_index(name="group_w")
    )
    total_w = float(g["group_w"].sum())
    if total_w <= 0:
        raise ValueError("Total weight <= 0. Check WP column.")

    rng = np.random.default_rng(seed)
    g = g.iloc[rng.permutation(len(g))].reset_index(drop=True)

    cum = g["group_w"].cumsum().to_numpy()
    idx = int(np.searchsorted(cum, target_w, side="left"))
    idx = min(idx, len(g) - 1)

    oos_key = g.loc[:idx, group_cols].drop_duplicates()

    marked = (
        df[group_cols]
        .merge(oos_key, on=group_cols, how="left", indicator=True)["_merge"]
        .eq("both")
        .to_numpy()
    )

    achieved_w = float(df.loc[marked, weight_col].sum())
    return marked, achieved_w

def balanced_oos_split_by_wave(
    df: pd.DataFrame,
    survey_col: str,
    weight_col: str,
    group_cols: list[str],
    target_oos_share: float,
    seed: int,
    waves=(2015, 2021),
):
    """
    Wave-balanced OOS by WP across waves.
    """
    df = df.copy().reset_index(drop=True)
    if len(df) == 0:
        return np.zeros(0, dtype=bool), 0.0, 0.0, {}

    df[survey_col] = pd.to_numeric(df[survey_col], errors="coerce")

    total_w = float(df[weight_col].sum())
    if total_w <= 0:
        raise ValueError("Total weight <= 0. Check WP column.")

    target_total_oos_w = target_oos_share * total_w
    present_waves = [w for w in waves if (df[survey_col] == w).any()]

    if len(present_waves) == 0:
        raise ValueError(f"No rows found for waves {waves} in column '{survey_col}'.")

    # one wave only
    if len(present_waves) == 1:
        w = present_waves[0]
        pos = np.flatnonzero(df[survey_col].to_numpy() == w)
        df_w = df.iloc[pos].copy().reset_index(drop=True)

        mask_w, achieved_w = weighted_group_holdout(
            df_w,
            group_cols=group_cols,
            weight_col=weight_col,
            target_w=target_total_oos_w,
            seed=seed,
        )

        full_mask = np.zeros(len(df), dtype=bool)
        full_mask[pos] = mask_w
        return full_mask, achieved_w, target_total_oos_w, {w: achieved_w}

    # two waves -> split target 50/50 in weighted terms
    per_wave_target = target_total_oos_w / 2.0
    full_mask = np.zeros(len(df), dtype=bool)
    achieved_by_wave = {}

    for k, w in enumerate(present_waves[:2]):
        pos = np.flatnonzero(df[survey_col].to_numpy() == w)
        df_w = df.iloc[pos].copy().reset_index(drop=True)

        mask_w, achieved_w = weighted_group_holdout(
            df_w,
            group_cols=group_cols,
            weight_col=weight_col,
            target_w=per_wave_target,
            seed=seed + 1000 * k + int(w),
        )

        full_mask[pos] = mask_w
        achieved_by_wave[w] = achieved_w

    achieved_total = float(sum(achieved_by_wave.values()))
    return full_mask, achieved_total, target_total_oos_w, achieved_by_wave

def write_split(out_dir: str, seg: str, train_df: pd.DataFrame, oos_df: pd.DataFrame):
    os.makedirs(out_dir, exist_ok=True)
    train_path = os.path.join(out_dir, f"destinations_{seg}_train.csv")
    oos_path   = os.path.join(out_dir, f"destinations_{seg}_oos.csv")
    train_df.to_csv(train_path, index=False)
    oos_df.to_csv(oos_path, index=False)
    print(f"  -> wrote {train_path} ({len(train_df):,} rows)")
    print(f"  -> wrote {oos_path}   ({len(oos_df):,} rows)")

def normalize_leisure_cat(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()

def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)


# ============================================================
# PART 1 — BUILD byPerson_noVisits SPLIT
# ============================================================
def build_no_visits_split():
    files = sorted(glob.glob(INPUT_GLOB))
    if not files:
        raise FileNotFoundError(f"No files found matching: {INPUT_GLOB}")

    # clean target dir to avoid stale files
    if os.path.exists(NOVISITS_SPLIT_DIR):
        shutil.rmtree(NOVISITS_SPLIT_DIR)
    os.makedirs(NOVISITS_SPLIT_DIR, exist_ok=True)

    print("\n" + "#" * 120)
    print("# PART 1 — BUILD Trips/byPerson_noVisits")
    print("# Filter leisure_cat == 'Visits' FIRST, then split byPerson")
    print("#" * 120)

    for fpath in files:
        seg = segment_from_filename(fpath)
        print(f"\n=== Processing segment {seg} ({fpath}) ===")

        df0 = pd.read_csv(fpath, low_memory=False)

        wcol = get_weight_col(df0)
        ocol = pick_col(df0, ORIG_COL_CANDIDATES, "origin")
        dcol = pick_col(df0, DEST_COL_CANDIDATES, "destination")
        pcol = pick_col(df0, PERSON_COL_CANDIDATES, "person (HHNR)")
        scol = pick_col(df0, SURVEY_COL_CANDIDATES, "survey/year")
        lcol = pick_col(df0, LEISURE_COL_CANDIDATES, "leisure category")

        # numeric cleaning
        df0[wcol] = pd.to_numeric(df0[wcol], errors="coerce")
        df0[ocol] = pd.to_numeric(df0[ocol], errors="coerce")
        df0[dcol] = pd.to_numeric(df0[dcol], errors="coerce")
        df0[scol] = pd.to_numeric(df0[scol], errors="coerce")

        df0 = df0.dropna(subset=[wcol, ocol, dcol, scol]).copy()
        df0 = df0[df0[wcol] > 0].copy()

        df0[ocol] = df0[ocol].astype(int)
        df0[dcol] = df0[dcol].astype(int)
        df0[scol] = df0[scol].astype(int)

        total_rows_full = len(df0)
        total_wp_full = float(df0[wcol].sum())

        # remove Visits BEFORE split
        leisure_norm = normalize_leisure_cat(df0[lcol])
        keep_mask = leisure_norm != "visits"
        df = df0.loc[keep_mask].copy().reset_index(drop=True)

        total_rows_filtered = len(df)
        total_wp_filtered = float(df[wcol].sum())

        print(f"[FULL raw]      rows={total_rows_full:,} | WP={total_wp_full:.2f}")
        print(f"[NO_VISITS raw] rows={total_rows_filtered:,} | WP={total_wp_filtered:.2f}")
        print(f"[REMOVED]       rows={total_rows_full-total_rows_filtered:,} | WP={total_wp_full-total_wp_filtered:.2f}")

        if len(df) == 0:
            raise ValueError(f"{seg}: all rows removed after filtering Visits.")

        # leak-proof person id = household-by-wave
        df["person_uid"] = df[pcol].astype(str) + "_" + df[scol].astype("Int64").astype(str)

        mask_oos, achieved_oos_w, target_oos_w, by_wave = balanced_oos_split_by_wave(
            df=df,
            survey_col=scol,
            weight_col=wcol,
            group_cols=["person_uid"],
            target_oos_share=TARGET_OOS_SHARE,
            seed=SEED_PERSON,
            waves=(2015, 2021),
        )

        oos_df = df.loc[mask_oos].copy()
        train_df = df.loc[~mask_oos].copy()

        achieved_share = float(oos_df[wcol].sum() / (train_df[wcol].sum() + oos_df[wcol].sum() + 1e-12))
        print(
            f"[byPerson NO_VISITS] OOS weighted share achieved: {achieved_share:.3f} "
            f"(target {TARGET_OOS_SHARE:.2f}) | target_oos_w={target_oos_w:.2f} achieved_oos_w={achieved_oos_w:.2f}"
        )
        print(f"[byPerson NO_VISITS] OOS by wave (WP): " + ", ".join([f"{k}: {v:.2f}" for k, v in by_wave.items()]))

        write_split(
            NOVISITS_SPLIT_DIR,
            seg,
            train_df.drop(columns=["person_uid"], errors="ignore"),
            oos_df.drop(columns=["person_uid"], errors="ignore"),
        )


# ============================================================
# PART 2 — MODEL COMPARISON FULL vs NO_VISITS
# ============================================================
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing split file: {path}")

    df = pd.read_csv(path, low_memory=False)

    for c in ["orig_zone", "dest_zone", "WP"]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:60]}")

    df = df.dropna(subset=["orig_zone", "dest_zone", "WP"]).copy()
    df["orig_zone"] = pd.to_numeric(df["orig_zone"], errors="coerce")
    df["dest_zone"] = pd.to_numeric(df["dest_zone"], errors="coerce")
    df["WP"] = pd.to_numeric(df["WP"], errors="coerce")
    df = df.dropna(subset=["orig_zone", "dest_zone", "WP"]).copy()

    df["orig_zone"] = df["orig_zone"].astype(int)
    df["dest_zone"] = df["dest_zone"].astype(int)
    df = df[df["WP"] > 0].copy()
    return df

def load_emu_matrix_and_mapping(omx_path: Path, emu_mat: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if emu_mat not in f.list_matrices():
        raise ValueError(f"Matrix '{emu_mat}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[emu_mat]
    print(f"[OMX] Loading {emu_mat} into RAM... shape={M.shape} (float32)")
    emu = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return emu, zone_to_idx, idx_to_zone

def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray, k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)

    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts

    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df["dest_zone"].to_numpy(dtype=int)
    orig   = df["orig_zone"].to_numpy(dtype=int)
    w      = df["WP"].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)  # chosen always first

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2
    )

def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(1, 10, 50, 100)):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = int(V.shape[1])
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

        out = {
            "NLL": nll,
            "MRR": mrr,
            "McFadden_R2": float(r2),
            "rank_median": rank_median,
            "J": J,
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))
    return out

def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(1, 10, 50, 100),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df["orig_zone"].to_numpy(dtype=int)
    dest = df["dest_zone"].to_numpy(dtype=int)
    w = df["WP"].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass = (
        df.groupby("dest_zone")["WP"].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    ll_model = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    top_num = {k: 0.0 for k in topKs}

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        nll_num += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    out = {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    return out

def run_model_comparison():
    print("\n" + "#" * 120)
    print("# PART 2 — Estimate FULL vs NO_VISITS")
    print("#" * 120)

    # sanity check
    for seg in SEGMENTS:
        p_full_tr = os.path.join(FULL_SPLIT_DIR, f"destinations_{seg}_train.csv")
        p_full_te = os.path.join(FULL_SPLIT_DIR, f"destinations_{seg}_oos.csv")
        p_nov_tr  = os.path.join(NOVISITS_SPLIT_DIR, f"destinations_{seg}_train.csv")
        p_nov_te  = os.path.join(NOVISITS_SPLIT_DIR, f"destinations_{seg}_oos.csv")

        for p in [p_full_tr, p_full_te, p_nov_tr, p_nov_te]:
            if not os.path.exists(p):
                raise FileNotFoundError(p)

    print("[LOAD] EMU matrix + mapping...")
    emu_mat, zone_to_idx, idx_to_zone = load_emu_matrix_and_mapping(UTIL_OMX, EMU_MAT, MAP_NAME)
    print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

    print("[LOAD] TZ features...")
    tz0 = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)

    need_cols = ["npvm_id"] + LOG1P_COLS
    miss = [c for c in need_cols if c not in tz0.columns]
    if miss:
        raise ValueError(f"TZ layer missing {miss}")

    geom_cols = ["geometry"] if "geometry" in tz0.columns else []
    tz0 = tz0[["npvm_id"] + LOG1P_COLS + geom_cols].copy()
    tz0["npvm_id"] = pd.to_numeric(tz0["npvm_id"], errors="coerce")
    tz0 = tz0.dropna(subset=["npvm_id"]).copy()
    tz0["npvm_id"] = tz0["npvm_id"].astype(int)

    print("[PREP] Global z-standardize destination vars on full TZ universe...")
    tz_feat = tz0.copy()
    tz_feat, z_cols, _, _ = standardize_all_zones(tz_feat, LOG1P_COLS, suffix="_z")

    variants = {
        "FULL": FULL_SPLIT_DIR,
        "NO_VISITS": NOVISITS_SPLIT_DIR,
    }

    results = []

    for variant_name, trips_dir in variants.items():
        print("\n" + "#" * 120)
        print(f"# VARIANT = {variant_name}")
        print(f"# trips_dir = {trips_dir}")
        print("#" * 120)

        var_offset = 0 if variant_name == "FULL" else 9999

        for seg in SEGMENTS:
            print("\n" + "=" * 100)
            print(f"Variant={variant_name} | Segment={seg} | K={K} | epochs={EPOCHS}")
            print("=" * 100)

            train_df = load_segment_csv(trips_dir, seg, "train")
            oos_df   = load_segment_csv(trips_dir, seg, "oos")

            print(f"[DATA] rows train={len(train_df):,} | oos={len(oos_df):,}")
            print(f"[DATA] WP   train={train_df['WP'].sum():.2f} | oos={oos_df['WP'].sum():.2f}")

            seed_tr = BASE_SEED + var_offset + 1000 * SEG_SEED[seg] + 10
            seed_te = BASE_SEED + var_offset + 1000 * SEG_SEED[seg] + 20

            emu_tr, X_tr, y_tr, w_tr, train_df2 = build_design(
                train_df, tz_feat, z_cols, emu_mat, zone_to_idx, k=K, seed=seed_tr
            )
            emu_te, X_te, y_te, w_te, oos_df2 = build_design(
                oos_df, tz_feat, z_cols, emu_mat, zone_to_idx, k=K, seed=seed_te
            )

            alpha_hat, beta_hat = train_mnl(
                emu_tr, X_tr, y_tr, w_tr,
                epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
            )

            met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS)
            met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS)

            max_n = FULL_EVAL_MAX.get(seg, 8000)
            seed_full = FULL_EVAL_SEED + var_offset + 1000 * SEG_SEED[seg]

            full_met = full_choice_eval_oos_subset(
                oos_df=oos_df2,
                tz_feat_seg=tz_feat,
                z_cols=z_cols,
                emu_mat=emu_mat,
                zone_to_idx=zone_to_idx,
                idx_to_zone=idx_to_zone,
                alpha_hat=alpha_hat,
                beta_hat=beta_hat,
                max_n=max_n,
                seed=seed_full,
                chunk=FULL_EVAL_CHUNK,
                topKs=TOPKS,
            )

            row = {
                "variant": variant_name,
                "segment": seg,
                "K": K,
                "alpha_emu": float(alpha_hat),

                "N_train": len(train_df),
                "N_oos": len(oos_df),
                "WP_train": float(train_df["WP"].sum()),
                "WP_oos": float(oos_df["WP"].sum()),

                "NLL_train": met_tr["NLL"],
                "MRR_train": met_tr["MRR"],
                "McFadden_R2_train": met_tr["McFadden_R2"],
                "rank_median_train": met_tr["rank_median"],
                "Top1_train": met_tr["Top1"],
                "Top10_train": met_tr["Top10"],
                "Top50_train": met_tr["Top50"],
                "Top100_train": met_tr["Top100"],

                "NLL_oos": met_te["NLL"],
                "MRR_oos": met_te["MRR"],
                "McFadden_R2_oos": met_te["McFadden_R2"],
                "rank_median_oos": met_te["rank_median"],
                "Top1_oos": met_te["Top1"],
                "Top10_oos": met_te["Top10"],
                "Top50_oos": met_te["Top50"],
                "Top100_oos": met_te["Top100"],
            }

            # save betas
            for name, val in zip(LOG1P_COLS, beta_hat):
                row[f"beta__{name}"] = float(val)

            row.update(full_met)
            results.append(row)

            print(
                "[SAMPLED OOS ]",
                f"NLL={row['NLL_oos']:.4f} MRR={row['MRR_oos']:.4f} "
                f"R2={row['McFadden_R2_oos']:.4f} Top10={row['Top10_oos']:.3f} Top50={row['Top50_oos']:.3f}"
            )

            print(
                "[FULL OOS   ]",
                f"NLL_full={row['NLL_full']:.4f} MRR_full={row['MRR_full']:.4f} "
                f"R2_full={row['McFadden_R2_full']:.4f} Top10_full={row['Top10_full']:.3f}"
            )

            print(
                "[SHARES FULL ]",
                f"spearman={row['share_spearman_full']:.4f} JS={row['share_JS_full']:.4f}"
            )

    # save outputs
    res_df = pd.DataFrame(results).sort_values(["segment", "variant"]).reset_index(drop=True)

    long_path = os.path.join(OUT_DIR, "no_visits_comparison_results_long.csv")
    res_df.to_csv(long_path, index=False)
    print(f"\n[OK] wrote {long_path}")

    value_cols = [c for c in res_df.columns if c not in ("variant", "segment")]
    wide = res_df.pivot(index="segment", columns="variant", values=value_cols)
    wide.columns = [f"{v}__{s}" for (v, s) in wide.columns]
    wide = wide.reset_index()

    wide_path = os.path.join(OUT_DIR, "no_visits_comparison_results_wide.csv")
    wide.to_csv(wide_path, index=False)
    print(f"[OK] wrote {wide_path}")

    # delta = NO_VISITS - FULL
    delta = pd.DataFrame({"segment": wide["segment"]})
    for col in value_cols:
        c_full = f"{col}__FULL"
        c_nov  = f"{col}__NO_VISITS"
        if c_full in wide.columns and c_nov in wide.columns:
            delta[col + "__delta_NOVISITS_minus_FULL"] = wide[c_nov] - wide[c_full]

    delta_path = os.path.join(OUT_DIR, "no_visits_comparison_deltas_NOVISITS_minus_FULL.csv")
    delta.to_csv(delta_path, index=False)
    print(f"[OK] wrote {delta_path}")

    # coefficients only: long
    beta_cols = ["alpha_emu"] + [c for c in res_df.columns if c.startswith("beta__")]
    coef_long = res_df[["variant", "segment"] + beta_cols].copy()
    coef_long_path = os.path.join(OUT_DIR, "no_visits_coefficients_long.csv")
    coef_long.to_csv(coef_long_path, index=False)
    print(f"[OK] wrote {coef_long_path}")

    # coefficients only: wide
    coef_wide = coef_long.pivot(index="segment", columns="variant", values=beta_cols)
    coef_wide.columns = [f"{v}__{s}" for (v, s) in coef_wide.columns]
    coef_wide = coef_wide.reset_index()
    coef_wide_path = os.path.join(OUT_DIR, "no_visits_coefficients_wide.csv")
    coef_wide.to_csv(coef_wide_path, index=False)
    print(f"[OK] wrote {coef_wide_path}")

    # coefficients delta only
    coef_delta = pd.DataFrame({"segment": coef_wide["segment"]})
    for col in beta_cols:
        c_full = f"{col}__FULL"
        c_nov  = f"{col}__NO_VISITS"
        if c_full in coef_wide.columns and c_nov in coef_wide.columns:
            coef_delta[col + "__delta_NOVISITS_minus_FULL"] = coef_wide[c_nov] - coef_wide[c_full]

    coef_delta_path = os.path.join(OUT_DIR, "no_visits_coefficients_deltas_NOVISITS_minus_FULL.csv")
    coef_delta.to_csv(coef_delta_path, index=False)
    print(f"[OK] wrote {coef_delta_path}")

    print("\n[DONE] Outputs in:", OUT_DIR)


# ============================================================
# RUN ALL
# ============================================================
if __name__ == "__main__":
    build_no_visits_split()
    run_model_comparison()

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
COEF_DELTA_CSV = "ModelRuns/NoVisitsComparison/no_visits_coefficients_deltas_NOVISITS_minus_FULL.csv"
METRIC_DELTA_CSV = "ModelRuns/NoVisitsComparison/no_visits_comparison_deltas_NOVISITS_minus_FULL.csv"

SEGMENTS = ["YS", "OS", "YL", "OL"]

PRETTY_NAMES = {
    "alpha_emu": "EMU utility",
    "F1_gastr_count_log1p": "Gastronomy count",
    "F2_pop_total_log1p": "Population",
    "F4_cult_count_log1p": "Cultural count",
    "F5_sport_count_log1p": "Sport count",
    "F5_sport_out_length_log1p": "Sport outdoor length",
    "F8_outdoor_lake_raw_dens_log1p": "Lake outdoor density",
    "F3_outdoor_hard_count_log1p": "Outdoor hard count",
    "F3_outdoor_soft_count_log1p": "Outdoor soft count",
    "F3_outdoor_LUmix": "Outdoor land-use mix",
    "F6_others_count_log1p": "Other count",
    "F10_superinfra_log1p": "Super-infrastructure",
    "F7_urban_sum_log1p": "Urban sum",
    "F8_POI_urban_dens_log1p": "POI urban density",
}

# ============================================================
# LOAD
# ============================================================
coef = pd.read_csv(COEF_DELTA_CSV)
met = pd.read_csv(METRIC_DELTA_CSV)

coef = coef.copy()
met = met.copy()

# keep only expected segment order if possible
coef["segment"] = pd.Categorical(coef["segment"], categories=SEGMENTS, ordered=True)
met["segment"] = pd.Categorical(met["segment"], categories=SEGMENTS, ordered=True)
coef = coef.sort_values("segment").reset_index(drop=True)
met = met.sort_values("segment").reset_index(drop=True)

# ============================================================
# HELPERS
# ============================================================
def clean_var_name(col: str) -> str:
    x = col.replace("beta__", "")
    x = x.replace("__delta_NOVISITS_minus_FULL", "")
    return x

def metric_pretty(col: str) -> str:
    return (
        col.replace("__delta_NOVISITS_minus_FULL", "")
           .replace("_full", "")
    )

def sign_word(x: float) -> str:
    if x > 0:
        return "more positive"
    elif x < 0:
        return "more negative"
    return "unchanged"

def print_separator():
    print("-" * 100)

# ============================================================
# 1) COEFFICIENT DELTAS TABLE
# ============================================================
beta_cols = [c for c in coef.columns if c.endswith("__delta_NOVISITS_minus_FULL")]

coef_long = []
for _, row in coef.iterrows():
    seg = row["segment"]
    for col in beta_cols:
        raw_var = clean_var_name(col)
        pretty = PRETTY_NAMES.get(raw_var, raw_var)
        coef_long.append({
            "segment": seg,
            "variable": raw_var,
            "pretty_variable": pretty,
            "delta": row[col],
            "abs_delta": abs(row[col]),
        })

coef_long = pd.DataFrame(coef_long)

print("\n" + "=" * 100)
print("COEFFICIENT DELTAS: NO_VISITS - FULL")
print("=" * 100)

pivot_pretty = coef_long.pivot(index="pretty_variable", columns="segment", values="delta")
print(pivot_pretty.round(4).fillna(""))

# ============================================================
# 2) TOP CHANGES BY SEGMENT
# ============================================================
print("\n" + "=" * 100)
print("TOP 5 LARGEST COEFFICIENT CHANGES BY SEGMENT")
print("=" * 100)

for seg in SEGMENTS:
    print(f"\n### {seg}")
    sub = (
        coef_long[coef_long["segment"] == seg]
        .sort_values("abs_delta", ascending=False)
        .head(5)
        .copy()
    )
    for _, r in sub.iterrows():
        print(f"{r['pretty_variable']:<28} delta = {r['delta']:+.4f}")

# ============================================================
# 3) POPULATION INTERPRETATION
# ============================================================
pop_col = "beta__F2_pop_total_log1p__delta_NOVISITS_minus_FULL"

print("\n" + "=" * 100)
print("POPULATION CHECK")
print("=" * 100)

all_negative = True
for _, row in coef.iterrows():
    seg = row["segment"]
    val = row[pop_col]
    print(f"{seg}: delta population = {val:+.4f} -> {sign_word(val)} under NO_VISITS")
    if not (val < 0):
        all_negative = False

if all_negative:
    print("\nInterpretation:")
    print(
        "Population becomes more negative in all segments after excluding Visits. "
        "This is consistent with the idea that, in the full sample, population partly "
        "captures socially motivated destinations; once Visits are removed, the remaining "
        "leisure trips are less attracted by generic population concentration."
    )
else:
    print("\nInterpretation:")
    print(
        "Population does not move uniformly across all segments, so the robustness story "
        "should be stated more cautiously."
    )

# ============================================================
# 4) PERFORMANCE DELTAS
# ============================================================
metric_cols = [c for c in met.columns if c != "segment"]

print("\n" + "=" * 100)
print("PERFORMANCE DELTAS: NO_VISITS - FULL")
print("=" * 100)

met_print = met.copy()
for c in metric_cols:
    met_print[c] = met_print[c].round(4)
print(met_print.to_string(index=False))

print("\n" + "=" * 100)
print("PERFORMANCE SUMMARY")
print("=" * 100)

for _, row in met.iterrows():
    seg = row["segment"]
    dnll = row.get("NLL_full__delta_NOVISITS_minus_FULL", np.nan)
    dmrr = row.get("MRR_full__delta_NOVISITS_minus_FULL", np.nan)
    dr2  = row.get("McFadden_R2_full__delta_NOVISITS_minus_FULL", np.nan)
    dspr = row.get("share_spearman_full__delta_NOVISITS_minus_FULL", np.nan)
    djs  = row.get("share_JS_full__delta_NOVISITS_minus_FULL", np.nan)

    print(f"\n### {seg}")
    print(f"NLL:      {dnll:+.4f} {'(better)' if dnll < 0 else '(worse)'}")
    print(f"MRR:      {dmrr:+.4f} {'(better)' if dmrr > 0 else '(worse)'}")
    print(f"R2:       {dr2:+.4f} {'(better)' if dr2 > 0 else '(worse)'}")
    print(f"Spearman: {dspr:+.4f} {'(better)' if dspr > 0 else '(worse)'}")
    print(f"JS:       {djs:+.4f} {'(better)' if djs < 0 else '(worse)'}")

# ============================================================
# 5) OPTIONAL EXPORT OF A CLEAN SUMMARY
# ============================================================
summary_rows = []

for seg in SEGMENTS:
    coef_seg = coef_long[coef_long["segment"] == seg].copy()
    top3 = coef_seg.sort_values("abs_delta", ascending=False).head(3)

    perf_row = met[met["segment"] == seg].iloc[0]

    summary_rows.append({
        "segment": seg,
        "population_delta": coef.loc[coef["segment"] == seg, pop_col].iloc[0],
        "top_change_1": top3.iloc[0]["pretty_variable"] if len(top3) > 0 else "",
        "top_change_1_delta": top3.iloc[0]["delta"] if len(top3) > 0 else np.nan,
        "top_change_2": top3.iloc[1]["pretty_variable"] if len(top3) > 1 else "",
        "top_change_2_delta": top3.iloc[1]["delta"] if len(top3) > 1 else np.nan,
        "top_change_3": top3.iloc[2]["pretty_variable"] if len(top3) > 2 else "",
        "top_change_3_delta": top3.iloc[2]["delta"] if len(top3) > 2 else np.nan,
        "delta_NLL_full": perf_row.get("NLL_full__delta_NOVISITS_minus_FULL", np.nan),
        "delta_MRR_full": perf_row.get("MRR_full__delta_NOVISITS_minus_FULL", np.nan),
        "delta_R2_full": perf_row.get("McFadden_R2_full__delta_NOVISITS_minus_FULL", np.nan),
        "delta_Spearman_full": perf_row.get("share_spearman_full__delta_NOVISITS_minus_FULL", np.nan),
        "delta_JS_full": perf_row.get("share_JS_full__delta_NOVISITS_minus_FULL", np.nan),
    })

summary_df = pd.DataFrame(summary_rows)
out_path = "ModelRuns/NoVisitsComparison/no_visits_interpretation_summary.csv"
summary_df.to_csv(out_path, index=False)

print("\n" + "=" * 100)
print(f"[OK] wrote {out_path}")
print("=" * 100)

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
BASELINE_CSV = "ModelRuns/BaselineFinal2/metrics_full_and_shares.csv"
NEW_SPEC_CSV = "ModelRuns/BaselineFinal_less_var/metrics_full_and_shares.csv"
OUT_CSV      = "ModelRuns/BaselineFinal_less_var/deltas_vs_BaselineFinal2.csv"

SEGMENTS = ["YS", "OS", "YL", "OL"]

# solo le metriche che ti servono davvero per la robustness table
METRICS = [
    "NLL_full_avg",
    "MRR_full",
    "McFadden_R2_full",
    "Spearman_share",
    "JS_share",
]

# se vuoi tenerne anche altre per controllo, aggiungile qui
OPTIONAL_METRICS = [
    "FF_full",
    "Top5_full",
    "Top10_full",
    "Pearson_share",   # togli pure se non ti serve
    "MAE_share",
    "RMSE_share",
]

# ============================================================
# LOAD
# ============================================================
base = pd.read_csv(BASELINE_CSV)
new  = pd.read_csv(NEW_SPEC_CSV)

base = base[base["segment"].isin(SEGMENTS)].copy()
new  = new[new["segment"].isin(SEGMENTS)].copy()

# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================
required_base = ["segment"] + METRICS
required_new  = ["segment"] + METRICS

missing_base = [c for c in required_base if c not in base.columns]
missing_new  = [c for c in required_new  if c not in new.columns]

if missing_base:
    raise ValueError(f"Missing columns in BASELINE_CSV: {missing_base}")
if missing_new:
    raise ValueError(f"Missing columns in NEW_SPEC_CSV: {missing_new}")

# opzionali: tengo solo quelle presenti in entrambi
extra_metrics = [c for c in OPTIONAL_METRICS if c in base.columns and c in new.columns]
all_metrics = METRICS + extra_metrics

# ============================================================
# KEEP ONE ROW PER SEGMENT
# ============================================================
base_tbl = base[["segment"] + all_metrics].drop_duplicates(subset=["segment"]).copy()
new_tbl  = new[["segment"]  + all_metrics].drop_duplicates(subset=["segment"]).copy()

base_tbl = base_tbl.rename(columns={m: f"{m}_baseline" for m in all_metrics})
new_tbl  = new_tbl.rename(columns={m: f"{m}_newspec"  for m in all_metrics})

# ============================================================
# MERGE
# ============================================================
df = new_tbl.merge(base_tbl, on="segment", how="left", validate="one_to_one")

# ============================================================
# DELTAS = NEW SPEC - BASELINE
# ============================================================
for m in all_metrics:
    df[f"d{m}"] = df[f"{m}_newspec"] - df[f"{m}_baseline"]

# ============================================================
# NICE COLUMN ORDER
# ============================================================
ordered_cols = ["segment"]

for m in METRICS:
    ordered_cols += [
        f"{m}_newspec",
        f"{m}_baseline",
        f"d{m}",
    ]

for m in extra_metrics:
    ordered_cols += [
        f"{m}_newspec",
        f"{m}_baseline",
        f"d{m}",
    ]

remaining = [c for c in df.columns if c not in ordered_cols]
df = df[ordered_cols + remaining]

# ============================================================
# SAVE
# ============================================================
Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False)

print(f"[OK] wrote {OUT_CSV}")
print(df.to_string(index=False))

## Finer Age

In [ ]:
# ============================================================
# SHORT-only extraction -> TripsTest/byPersonAgeBins/
#
# CORRECTED LEISURE FILTER:
#   - keep ONLY short leisure trips with:
#       wzweck3 == 8
#       wzweck2 == 1   (Hinweg / outward only)
#
# - Age bins: 6-16, 16-30, 30-50, 50-65, 65-80, 80+
# - Zones: TZ_fixed npvm_id (matches OMX)
# - Split: train/oos by person = (HHNR, survey)  [no leakage]
# - Prints N stats per bin and split
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

# -----------------------------
# PATHS
# -----------------------------
PATH15  = "MTMZ/MZMV2015_mit_Geo/4_DB_csv/CH_CSV"
PATH21  = "MTMZ/MZMV2021_mit_Geo/4_DB_csv/4_DB_csv"
TZ_PATH = "TZ/TZ.gpkg"          # layer="TZ_fixed" has npvm_id
OUT_DIR = "TripsTest/byPersonAgeBins"

os.makedirs(OUT_DIR, exist_ok=True)

CRS_WGS84  = 4326
CRS_TARGET = 2056

# -----------------------------
# SPLIT SETTINGS
# -----------------------------
OOS_FRAC   = 0.20
SPLIT_SEED = 2024

# deterministic per-bin seeds
BIN_SEEDS = {
    "6-16":  6106,
    "16-30": 1630,
    "30-50": 3050,
    "50-65": 5065,
    "65-80": 6580,
    "80+":   8000,
}

# -----------------------------
# Leisure category mapper
# -----------------------------
def map_wegeinland_leisure_activity(code: float) -> str:
    if not np.isfinite(code):
        return "NK"
    code = int(code)

    if code in (-99, -98, -97):
        return "NK"
    if code == 1:
        return "Visits"
    if code == 2:
        return "Gastronomy"
    if code in (4, 5, 7, 15, 17):
        return "Outdoor"
    if code == 9:
        return "Culture"
    if code == 3:
        return "Sport"
    if code in (6, 8, 10, 11, 12, 13, 14, 16, 18, 22, 90):
        return "Others"

    return "NK"

# -----------------------------
# Load WegeInland + Zielpersonen for a year
# -----------------------------
def load_wegeinland_year(path, year):
    sep = "," if year == 2015 else ";"

    wege = pd.read_csv(
        f"{path}/wegeinland.csv",
        sep=sep,
        encoding="latin1",
        low_memory=False
    )
    ziel = pd.read_csv(
        f"{path}/zielpersonen.csv",
        sep=sep,
        encoding="latin1",
        low_memory=False
    )

    df = wege.merge(
        ziel[["HHNR", "alter", "WP"]],
        on="HHNR",
        how="left",
        suffixes=("", "_ZP")
    )

    df["alter"] = pd.to_numeric(df["alter"], errors="coerce")
    df = df[df["alter"].notna()].copy()
    df["survey"] = int(year)

    return df

# -----------------------------
# Load TZ_fixed (npvm_id)
# -----------------------------
tz = gpd.read_file(TZ_PATH, layer="TZ_fixed")

if tz.crs is None:
    raise ValueError("TZ.gpkg has no CRS set. Please set it to EPSG:2056 (LV95).")

if tz.crs.to_epsg() != CRS_TARGET:
    tz = tz.to_crs(CRS_TARGET)

if "npvm_id" not in tz.columns:
    raise ValueError("TZ_fixed must contain 'npvm_id' aligned to OMX mapping.")

tz_zones = tz[["npvm_id", "geometry"]].copy()

# -----------------------------
# Assign zones (WGS84 point -> npvm_id)
# -----------------------------
def assign_zones_wgs84(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    tz_gdf: gpd.GeoDataFrame,
    out_col: str
) -> pd.DataFrame:
    sub = df[["trip_id", x_col, y_col]].copy()
    sub[x_col] = pd.to_numeric(sub[x_col], errors="coerce")
    sub[y_col] = pd.to_numeric(sub[y_col], errors="coerce")
    sub = sub[sub[x_col].notna() & sub[y_col].notna()].copy()

    if sub.empty:
        return pd.DataFrame(columns=["trip_id", out_col])

    gdf = gpd.GeoDataFrame(
        sub[["trip_id"]],
        geometry=gpd.points_from_xy(sub[x_col], sub[y_col]),
        crs=f"EPSG:{CRS_WGS84}"
    ).to_crs(CRS_TARGET)

    # first pass: within
    sj = gpd.sjoin(gdf, tz_gdf, how="left", predicate="within")
    matched = sj[sj["npvm_id"].notna()][["trip_id", "npvm_id"]].drop_duplicates("trip_id")

    # fallback: intersects
    if len(matched) < len(gdf):
        remaining = gdf[~gdf["trip_id"].isin(matched["trip_id"])].copy()
        if not remaining.empty:
            sj2 = gpd.sjoin(remaining, tz_gdf, how="left", predicate="intersects")
            matched2 = sj2[sj2["npvm_id"].notna()][["trip_id", "npvm_id"]].drop_duplicates("trip_id")
            matched = pd.concat([matched, matched2], ignore_index=True)

    return matched.rename(columns={"npvm_id": out_col})

# -----------------------------
# Age bins
# -----------------------------
AGE_BINS = [
    ("6-16",  6,  16),
    ("16-30", 16, 30),
    ("30-50", 30, 50),
    ("50-65", 50, 65),
    ("65-80", 65, 80),
    ("80+",   80, 200),
]

def agebin_label(age: float) -> str | None:
    if not np.isfinite(age):
        return None

    a = float(age)
    for lab, lo, hi in AGE_BINS:
        if lab != "80+":
            if (a >= lo) and (a < hi):
                return lab
        else:
            if a >= lo:
                return lab

    return None

# -----------------------------
# Build destination df for SHORT
# -----------------------------
def build_dest_df_short(df: pd.DataFrame, agebin_value: str) -> pd.DataFrame:
    cols = [
        "HHNR", "survey",
        "orig_zone_id", "dest_zone_id",
        "alter", "agebin",
        "leisure_cat",
        "start_lang", "dest_lang",
        "w_rdist", "dur_tmp",
        "WP",
    ]
    cols = [c for c in cols if c in df.columns]

    out = df[df["agebin"] == agebin_value][cols].copy()

    out = out.rename(columns={
        "orig_zone_id": "orig_zone",
        "dest_zone_id": "dest_zone",
        "w_rdist": "dist_km",
        "dur_tmp": "dur_min",
    }).reset_index(drop=True)

    out["HHNR"]      = pd.to_numeric(out["HHNR"], errors="coerce").astype("Int64")
    out["survey"]    = pd.to_numeric(out["survey"], errors="coerce").astype("Int64")
    out["orig_zone"] = pd.to_numeric(out["orig_zone"], errors="coerce").astype("Int64")
    out["dest_zone"] = pd.to_numeric(out["dest_zone"], errors="coerce").astype("Int64")

    out["start_lang"] = pd.to_numeric(out.get("start_lang"), errors="coerce").astype("Int64")
    out["dest_lang"]  = pd.to_numeric(out.get("dest_lang"), errors="coerce").astype("Int64")

    out["dist_km"] = pd.to_numeric(out.get("dist_km"), errors="coerce")
    out["dur_min"] = pd.to_numeric(out.get("dur_min"), errors="coerce")
    out["WP"]      = pd.to_numeric(out.get("WP"), errors="coerce")

    out.loc[out["dist_km"] < 0, "dist_km"] = np.nan
    out.loc[out["dur_min"] < 0, "dur_min"] = np.nan

    out = out.dropna(subset=["HHNR", "survey", "orig_zone", "dest_zone", "WP"]).copy()
    out = out[out["WP"] > 0].copy()

    return out

# -----------------------------
# Split by person (HHNR+survey)
# -----------------------------
def split_by_person(df: pd.DataFrame, oos_frac: float, seed: int):
    if df.empty:
        return df.copy(), df.copy()

    persons = df[["HHNR", "survey"]].drop_duplicates().copy()
    persons = persons.dropna().copy()
    persons["HHNR"] = persons["HHNR"].astype(int)
    persons["survey"] = persons["survey"].astype(int)

    rng = np.random.default_rng(seed)

    nP = len(persons)
    nO = int(np.round(oos_frac * nP))
    nO = max(1, nO) if nP > 1 else 0

    perm = rng.permutation(nP)
    oos_idx = perm[:nO]
    oos_persons = set(map(tuple, persons.iloc[oos_idx][["HHNR", "survey"]].to_numpy()))

    key = list(map(tuple, df[["HHNR", "survey"]].astype(int).to_numpy()))
    is_oos = np.array([k in oos_persons for k in key], dtype=bool)

    oos = df[is_oos].copy()
    tr  = df[~is_oos].copy()
    return tr, oos

def print_stats(tag: str, df: pd.DataFrame):
    if df.empty:
        print(f"  {tag}: rows=0 | WP_sum=0 | persons=0 | orig==dest%=0")
        return

    persons = df[["HHNR", "survey"]].drop_duplicates().shape[0]
    wp_sum = float(df["WP"].sum())
    same = float((df["orig_zone"] == df["dest_zone"]).mean() * 100.0)

    print(f"  {tag}: rows={len(df):,} | WP_sum={wp_sum:.2f} | persons={persons:,} | orig==dest%={same:.2f}")

# ============================================================
# 1) LOAD 2015 + 2021
# ============================================================
wege_all = pd.concat(
    [load_wegeinland_year(PATH15, 2015), load_wegeinland_year(PATH21, 2021)],
    ignore_index=True
)

# ============================================================
# 2) KEEP ONLY CORRECT SHORT LEISURE:
#    wzweck3 == 8 AND wzweck2 == 1
# ============================================================
wege_all["wzweck3"] = pd.to_numeric(wege_all["wzweck3"], errors="coerce")
wege_all["wzweck2"] = pd.to_numeric(wege_all["wzweck2"], errors="coerce")

wege_short = wege_all[
    (wege_all["wzweck3"] == 8) &
    (wege_all["wzweck2"] == 1)
].copy()

# age bin
wege_short["agebin"] = wege_short["alter"].apply(agebin_label)
wege_short = wege_short[wege_short["agebin"].notna()].copy()

# leisure_cat
if "f51700_weg" in wege_short.columns:
    wege_short["f51700_weg"] = pd.to_numeric(wege_short["f51700_weg"], errors="coerce")
    wege_short["leisure_cat"] = wege_short["f51700_weg"].apply(map_wegeinland_leisure_activity)
else:
    wege_short["leisure_cat"] = "NK"

# language codes
wege_short["start_lang"] = pd.to_numeric(wege_short.get("S_SPRACHE"), errors="coerce")
wege_short["dest_lang"]  = pd.to_numeric(wege_short.get("Z_SPRACHE"), errors="coerce")

# distance/duration
if "w_rdist" in wege_short.columns:
    wege_short["w_rdist"] = pd.to_numeric(wege_short["w_rdist"], errors="coerce")
else:
    wege_short["w_rdist"] = np.nan

if "dauer1" in wege_short.columns:
    wege_short["dur_tmp"] = pd.to_numeric(wege_short["dauer1"], errors="coerce")
elif "dauer2" in wege_short.columns:
    wege_short["dur_tmp"] = pd.to_numeric(wege_short["dauer2"], errors="coerce")
else:
    wege_short["dur_tmp"] = np.nan

# trip_id
wege_short = wege_short.reset_index(drop=True)
wege_short["trip_id"] = np.arange(len(wege_short))

# ============================================================
# 3) Assign zones -> npvm_id
# ============================================================
orig_z = assign_zones_wgs84(wege_short, "S_X", "S_Y", tz_zones, "orig_zone_id")
dest_z = assign_zones_wgs84(wege_short, "Z_X", "Z_Y", tz_zones, "dest_zone_id")

wege_short = wege_short.merge(orig_z, on="trip_id", how="left")
wege_short = wege_short.merge(dest_z, on="trip_id", how="left")
wege_short = wege_short.dropna(subset=["orig_zone_id", "dest_zone_id"]).copy()

# ============================================================
# 4) Build, split by person, save per agebin
# ============================================================
print("\n=== SHORT leisure age-bins: build + split train/oos by person (HHNR+survey) ===")
print("Filter used: wzweck3 == 8 AND wzweck2 == 1")

for lab, _, _ in AGE_BINS:
    df_lab = build_dest_df_short(wege_short, lab)

    tr, oos = split_by_person(
        df_lab,
        oos_frac=OOS_FRAC,
        seed=SPLIT_SEED + BIN_SEEDS[lab]
    )

    p_tr  = os.path.join(OUT_DIR, f"destinations_S_{lab}_train.csv")
    p_oos = os.path.join(OUT_DIR, f"destinations_S_{lab}_oos.csv")

    tr.to_csv(p_tr, index=False)
    oos.to_csv(p_oos, index=False)

    print(f"\n[{lab}]")
    print_stats("TRAIN", tr)
    print_stats("OOS  ", oos)

print("\nSaved to:", OUT_DIR)

In [ ]:
# ============================================================
# AGE-BINS SHORT TRIPS - BASELINE ESTIMATION + FULL REPORT
# (UNIFORMED TO FINAL BASELINE STYLE)
#
# Utility:
#   V_ij = alpha * EMU_ij + X_j beta
#
# What this script does (per age bin):
#   1) Train baseline MNL on sampled choice sets
#      (chosen + K non-chosen), uniform sampling.
#   2) Report OOS sampled-set metrics:
#        - weighted NLL (sum WP * -log P_chosen)
#        - Top5 / Top10 accuracy
#        - MRR
#        - FF
#   3) FULL-choice-set diagnostics on an OOS subset (all zones in denominator):
#        - NLL_full, MRR_full, FF_full, McFadden_R2_full, Top5/10_full
#        - destination shares S_obs vs S_pred:
#            Pearson, Spearman, MAE, RMSE, JS
#   4) Distance plausibility:
#        - compares observed dist_km to predicted expected distance
#          using average car-network distance skim
#        - saves plots + summary stats
#        - saves trip-level OOS distance outputs
#   5) Coefficient table with Hessian-based significance on a train subset:
#        - coef, se, z, p, stars
#   6) Destination-level outputs:
#        - attractivity index A_j = X_j(beta) (on z features)
#        - predicted shares map-ready table
#        - incoming accessibility logsum proxy from origins:
#              L_j = log Σ_i w_i exp(alpha * EMU_ij)
#        - plots: attractivity vs incoming accessibility and maps
#
# Outputs (all under OUT_DIR):
#   - metrics_sampled.csv
#   - metrics_full_and_shares.csv
#   - metrics_weighted_averages.csv
#   - coef_table.csv
#   - tz_outputs.gpkg
#   - distance_oos_triplevel/*.csv
#   - plots/*.png
#
# IMPORTANT:
#   - Uses global z-standardization on FULL TZ universe
#   - Deterministic seeds per age bin
#   - FULL-choice-set evaluation is chunked and optionally capped
#   - Coefficients are estimated exactly as before
#   - Only SE / z / p / stars are Hessian-based post-estimation values
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path
import matplotlib.pyplot as plt


# -----------------------------
# CONFIG
# -----------------------------
AGE_BINS = ["6-16", "16-30", "30-50", "50-65", "65-80", "80+"]
BIN_SEED = {b: i + 1 for i, b in enumerate(AGE_BINS)}

ORIG_COL = "orig_zone"
DEST_COL = "dest_zone"
WP_COL   = "WP"
DIST_COL = "dist_km"   # observed trip distance; if missing, distance checks are skipped

TRIPS_DIR = "TripsTest/byPersonAgeBins"

TZ_GPKG  = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER = "TZ_first_sel_log1p"
TZ_KEY   = "npvm_id"

BASE_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

# Impedance (EMU)
UTIL_OMX     = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME = "utility_emu"

# Distance proxy for plausibility: average car-network distance
DIST_OMX     = READY_DIR / "distance_avg_2023_ready.omx"
DIST_MAT_NAME = "avg_distances_ready"

MAP_NAME = "NO"

OUT_DIR  = "ModelRuns/AgeBinsShortBaselineFinal"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DIST_OOS_DIR = os.path.join(OUT_DIR, "distance_oos_triplevel")
os.makedirs(DIST_OOS_DIR, exist_ok=True)

# Core baseline decisions
K = 1000
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# Full-choice-set evaluation subset sizes (optional cap per age bin)
# set to None or use dict values to keep runtime reasonable
FULL_EVAL_MAX = {
    "6-16": None,
    "16-30": None,
    "30-50": None,
    "50-65": None,
    "65-80": None,
    "80+": None,
}
FULL_EVAL_SEED  = 777
FULL_EVAL_CHUNK = 256

# Hessian SE subset sizes
SE_MAX_TRAIN = {
    "6-16": 12000,
    "16-30": 20000,
    "30-50": 20000,
    "50-65": 16000,
    "65-80": 12000,
    "80+": 8000,
}
SE_SEED = 2024

BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

TOPKS_SAMPLED = (5, 10)
TOPKS_FULL    = (5, 10)


# ============================================================
# Reproducibility helper
# ============================================================
def set_all_seeds(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Small helpers
# ============================================================
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def p_value_two_sided_z(z):
    return 2.0 * (1.0 - norm_cdf(abs(float(z))))

def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.1:
        return "."
    return ""


# ============================================================
# IO helpers
# ============================================================
def load_bin_csv(trips_dir: str, bin_name: str, split: str) -> pd.DataFrame:
    """
    Expected filenames:
      destinations_S_<BIN>_train.csv
      destinations_S_<BIN>_oos.csv
    """
    path = os.path.join(trips_dir, f"destinations_S_{bin_name}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:60]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()

    if DIST_COL in df.columns:
        df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")

    return df

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


# ============================================================
# Preprocess helpers
# ============================================================
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    """
    Z-standardize on the FULL TZ universe.
    """
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd


# ============================================================
# Sampling + design builder
# ============================================================
def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool) + 1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts
    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat[TZ_KEY].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index(TZ_KEY)[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]
    dest_set2 = dest_set[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
        dest_set2,
        orig_idx2,
        dest_idx2,
    )


# ============================================================
# Model
# ============================================================
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)

            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum()

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted-avg NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(5, 10), want_ff=True):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        P = torch.exp(logP)

        chosen_logp = logP[:, 0]
        chosen_p    = P[:, 0]
        sum_w = float(w.sum().cpu())

        nll_sum = float((-(w * chosen_logp)).sum().cpu())
        nll_avg = nll_sum / (sum_w + 1e-9)

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))

        out = {
            "NLL_sum": nll_sum,
            "NLL_avg": nll_avg,
            "MRR": mrr,
            "sum_WP": sum_w,
            "N": int(V.shape[0]),
            "J": int(V.shape[1]),
        }

        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))

        if want_ff:
            out["FF"] = float((w * chosen_p).sum().cpu() / (sum_w + 1e-9))

    return out


# ============================================================
# FULL-choice-set evaluation on OOS subset
# + shares + distance plausibility
# ============================================================
def full_eval_and_shares_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    dist_mat: np.ndarray | None,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(5, 10),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w    = df[WP_COL].to_numpy(dtype=np.float64)

    has_dist = (DIST_COL in df.columns) and df[DIST_COL].notna().any()
    obs_dist = df[DIST_COL].to_numpy(dtype=np.float64) if has_dist else None

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    if has_dist:
        good = good & np.isfinite(obs_dist) & (obs_dist >= 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w        = w[good]
    if has_dist:
        obs_dist = obs_dist[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}, None, (None, None), None

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat[TZ_KEY].to_numpy(dtype=int)
    Xz_all = tz_feat[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    ll_model = 0.0
    mrr_num = 0.0
    ff_num = 0.0
    top_num = {k: 0.0 for k in topKs}
    ranks = np.empty(len(df), dtype=np.int64)

    exp_dist = np.zeros(len(df), dtype=np.float64) if (dist_mat is not None and has_dist) else None

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        # pass 1: max V
        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        # pass 2: sumexp + rank
        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse
        p_chosen = math.exp(logp_chosen)

        nll_num += wn * (-logp_chosen)
        ll_model += wn * logp_chosen
        ff_num += wn * p_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        if exp_dist is not None:
            expd = 0.0

        # pass 3: shares + expected distance
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

            if exp_dist is not None:
                dist_blk = dist_mat[o, j0:j1].astype(np.float64)
                expd += float(np.sum(Pblk * dist_blk))

        if exp_dist is not None:
            exp_dist[n] = expd

    nll_full = float(nll_num)
    nll_full_avg = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    ff_full = float(ff_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)

    pear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="pearson")) if len(S_obs) > 2 else np.nan
    spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    mae = float(np.mean(np.abs(S_obs - S_pred)))
    rmse = float(np.sqrt(np.mean((S_obs - S_pred) ** 2)))
    js = float(js_div(S_obs, S_pred))

    dist_summary = {}
    if exp_dist is not None:
        wnorm = w / (sum_w + 1e-12)
        obs_mean = float(np.sum(wnorm * obs_dist))
        pred_mean = float(np.sum(wnorm * exp_dist))
        dist_summary = {
            "dist_obs_mean_km": obs_mean,
            "dist_pred_mean_km": pred_mean,
            "dist_mean_diff_km": float(pred_mean - obs_mean),
            "dist_obs_p50_km": float(np.quantile(obs_dist, 0.50)),
            "dist_pred_p50_km": float(np.quantile(exp_dist, 0.50)),
            "dist_obs_p90_km": float(np.quantile(obs_dist, 0.90)),
            "dist_pred_p90_km": float(np.quantile(exp_dist, 0.90)),
        }

    out = {
        "NLL_full_sum": nll_full,
        "NLL_full_avg": nll_full_avg,
        "MRR_full": mrr_full,
        "FF_full": ff_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Pearson_share": pear,
        "Spearman_share": spear,
        "MAE_share": mae,
        "RMSE_share": rmse,
        "JS_share": js,
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    out.update(dist_summary)

    trip_distance_df = None
    if exp_dist is not None:
        trip_distance_df = pd.DataFrame({
            "orig_zone": df[ORIG_COL].to_numpy(dtype=int),
            "dest_zone_obs": df[DEST_COL].to_numpy(dtype=int),
            "WP": w.astype(np.float64),
            "dist_obs_km": obs_dist.astype(np.float64),
            "dist_exp_km": exp_dist.astype(np.float64),
        })

    return out, {
        "tz_ids": all_zone_ids,
        "S_obs": S_obs,
        "S_pred": S_pred,
        "Aj": Aj.astype(np.float64),
        "orig_idx": orig_idx,
        "w": w,
    }, (obs_dist, exp_dist), trip_distance_df


# ============================================================
# Hessian-based SEs (post-estimation; no re-estimation)
# ============================================================
def approx_hessian_se(
    emu_set: torch.Tensor,
    X_set: torch.Tensor,
    y: torch.Tensor,
    w: torch.Tensor,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int,
    seed: int,
    ridge: float = 1e-8,
):
    N = emu_set.shape[0]
    rng = np.random.default_rng(seed)

    if (max_n is not None) and (N > max_n):
        idx = rng.choice(N, size=max_n, replace=False)
        idx = torch.tensor(idx, dtype=torch.long, device=DEVICE)
        emu = emu_set[idx]
        X   = X_set[idx]
        yy  = y[idx]
        ww  = w[idx]
    else:
        emu, X, yy, ww = emu_set, X_set, y, w

    sum_w = float(ww.sum().detach().cpu())
    Pdim = X.shape[2]

    alpha = torch.tensor([alpha_hat], dtype=DTYPE, device=DEVICE, requires_grad=True)
    beta  = torch.tensor(beta_hat, dtype=DTYPE, device=DEVICE, requires_grad=True)

    def loss_avg(a, b):
        V = a * emu + torch.einsum("bjp,p->bj", X, b)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        ll = logP[torch.arange(V.shape[0], device=DEVICE), yy]
        return (-(ww * ll)).sum() / (ww.sum() + 1e-12)

    L = loss_avg(alpha, beta)
    g = torch.autograd.grad(L, [alpha, beta], create_graph=True)
    g_vec = torch.cat([g[0].reshape(1), g[1].reshape(-1)], dim=0)

    H_avg = torch.zeros((1 + Pdim, 1 + Pdim), dtype=torch.float64, device=DEVICE)
    for i in range(1 + Pdim):
        gi = g_vec[i]
        hi = torch.autograd.grad(gi, [alpha, beta], retain_graph=True)
        hi_vec = torch.cat([hi[0].reshape(1), hi[1].reshape(-1)], dim=0)
        H_avg[i, :] = hi_vec.detach().to(torch.float64)

    H_avg = H_avg.cpu().numpy()
    H_sum = sum_w * H_avg

    H_sum = 0.5 * (H_sum + H_sum.T)
    H_sum = H_sum + ridge * np.eye(H_sum.shape[0], dtype=np.float64)

    Vcov = np.linalg.pinv(H_sum)
    se = np.sqrt(np.clip(np.diag(Vcov), 0.0, np.inf))
    return se


# ============================================================
# Destination-level summaries and plots
# ============================================================
def incoming_accessibility_logsum(
    tz_ids: np.ndarray,
    emu_mat: np.ndarray,
    alpha: float,
    orig_idx: np.ndarray,
    w: np.ndarray,
    zone_to_idx: dict,
):
    Msize = emu_mat.shape[0]
    w_by_o = np.zeros(Msize, dtype=np.float64)
    for o, wn in zip(orig_idx, w):
        w_by_o[int(o)] += float(wn)

    origins = np.where(w_by_o > 0)[0]
    w_o = w_by_o[origins]
    w_o = w_o / (w_o.sum() + 1e-12)

    L = np.full(len(tz_ids), np.nan, dtype=np.float64)
    tz_omx = np.array([zone_to_idx.get(int(z), -1) for z in tz_ids], dtype=int)
    good = (tz_omx >= 0)

    for pos in np.where(good)[0]:
        j = tz_omx[pos]
        vals = alpha * emu_mat[origins, j].astype(np.float64)
        m = np.max(vals)
        s = np.sum(w_o * np.exp(vals - m))
        L[pos] = float(m + np.log(s + 1e-300))

    return L

def plot_distance_plausibility(obs_dist, exp_dist, bin_name, out_png):
    obs_dist = np.asarray(obs_dist, dtype=np.float64)
    exp_dist = np.asarray(exp_dist, dtype=np.float64)

    finite = np.isfinite(obs_dist) & np.isfinite(exp_dist)
    obs_dist = obs_dist[finite]
    exp_dist = exp_dist[finite]

    if len(obs_dist) == 0:
        return

    obs_mean = float(np.mean(obs_dist))
    exp_mean = float(np.mean(exp_dist))

    xmax = float(max(np.max(obs_dist), np.max(exp_dist)))
    bins = np.linspace(0.0, xmax, 51)

    plt.figure()
    plt.hist(obs_dist, bins=bins, density=True, alpha=0.6,
             label=f"Observed dist_km (mean={obs_mean:.2f})")
    plt.hist(exp_dist, bins=bins, density=True, alpha=0.6,
             label=f"Predicted E[distance] (mean={exp_mean:.2f})")
    plt.xlabel("Distance (km)")
    plt.ylabel("Density")
    plt.title(f"Distance plausibility | bin={bin_name} | obs mean={obs_mean:.2f}, pred mean={exp_mean:.2f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_attr_vs_access(Aj, L_in, S_pred, bin_name, out_png):
    plt.figure()
    s = 20 + 4000 * (S_pred / (S_pred.max() + 1e-12))
    plt.scatter(L_in, Aj, s=s, alpha=0.5)
    plt.xlabel("Incoming accessibility logsum (from origins, using EMU)")
    plt.ylabel("Attractivity index A_j = X_j beta (z-scale)")
    plt.title(f"Attractivity vs incoming accessibility | bin={bin_name}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_share_maps(tz_gdf, col, bin_name, out_png, title):
    plt.figure()
    ax = tz_gdf.plot(column=col, legend=True)
    ax.set_axis_off()
    plt.title(f"{title} | bin={bin_name}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


# ============================================================
# MAIN
# ============================================================
set_all_seeds(BASE_SEED)

print("[LOAD] TZ features...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
if TZ_KEY not in tz.columns:
    raise ValueError(f"TZ layer missing '{TZ_KEY}'")

tz[TZ_KEY] = pd.to_numeric(tz[TZ_KEY], errors="coerce")
tz = tz.dropna(subset=[TZ_KEY]).copy()
tz[TZ_KEY] = tz[TZ_KEY].astype(int)

need_cols = [TZ_KEY] + BASE_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

for c in BASE_COLS:
    tz[c] = pd.to_numeric(tz[c], errors="coerce")

print("[PREP] Global z-standardize destination variables on FULL TZ universe...")
tz_feat = tz[[TZ_KEY, "geometry"] + BASE_COLS].copy()
tz_feat, Z_COLS, MU_ALL, SD_ALL = standardize_all_zones(tz_feat, BASE_COLS, suffix="_z")

print("[LOAD] EMU matrix + mapping...")
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

print("[LOAD] Average distance matrix (for distance plausibility)...")
dist_mat, zone_to_idx_dist, idx_to_zone_dist = load_matrix_and_mapping(DIST_OMX, DIST_MAT_NAME, MAP_NAME)
if len(idx_to_zone_dist) != len(idx_to_zone) or not np.all(idx_to_zone_dist == idx_to_zone):
    print("[WARN] Distance OMX mapping differs from utilities OMX mapping. Distance plausibility may be misaligned.")

rows_sampled = []
rows_full    = []
rows_coef    = []

GPKG_OUT = os.path.join(OUT_DIR, "tz_outputs.gpkg")
if os.path.exists(GPKG_OUT):
    os.remove(GPKG_OUT)

for bin_name in AGE_BINS:
    print("\n" + "#" * 120)
    print(f"# AGE BIN = {bin_name} | SHORT | K={K} | epochs={EPOCHS}")
    print("#" * 120)

    set_all_seeds(BASE_SEED + 1000 * BIN_SEED[bin_name])

    train_df = load_bin_csv(TRIPS_DIR, bin_name, "train")
    oos_df   = load_bin_csv(TRIPS_DIR, bin_name, "oos")

    seed_tr   = BASE_SEED + 1000 * BIN_SEED[bin_name] + 10
    seed_te   = BASE_SEED + 1000 * BIN_SEED[bin_name] + 20
    seed_full = FULL_EVAL_SEED + 1000 * BIN_SEED[bin_name]

    emu_tr, X_tr, y_tr, w_tr, train_df2, destset_tr, origidx_tr, destidx_tr = build_design(
        train_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_tr
    )
    emu_te, X_te, y_te, w_te, oos_df2, destset_te, origidx_te, destidx_te = build_design(
        oos_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_te
    )

    print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
    print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

    alpha_hat, beta_hat = train_mnl(
        emu_tr, X_tr, y_tr, w_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
    )

    met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)
    met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)

    rows_sampled.append({
        "bin": bin_name,
        "split": "train",
        "alpha_emu": float(alpha_hat),
        "K": K,
        **met_tr,
    })
    rows_sampled.append({
        "bin": bin_name,
        "split": "oos",
        "alpha_emu": float(alpha_hat),
        "K": K,
        **met_te,
    })

    print("[SAMPLED OOS ]",
          f"NLL_sum={met_te['NLL_sum']:.2f} NLL_avg={met_te['NLL_avg']:.4f} "
          f"MRR={met_te['MRR']:.4f} Top5={met_te['Top5']:.3f} Top10={met_te['Top10']:.3f} FF={met_te['FF']:.4f}")

    max_n = FULL_EVAL_MAX.get(bin_name, None)
    full_out, share_pack, dist_pack, trip_distance_df = full_eval_and_shares_oos_subset(
        oos_df=oos_df2,
        tz_feat=tz_feat,
        z_cols=Z_COLS,
        emu_mat=emu_mat,
        dist_mat=dist_mat,
        zone_to_idx=zone_to_idx,
        idx_to_zone=idx_to_zone,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=max_n,
        seed=seed_full,
        chunk=FULL_EVAL_CHUNK,
        topKs=TOPKS_FULL,
    )

    rows_full.append({
        "bin": bin_name,
        "alpha_emu": float(alpha_hat),
        "K": K,
        **full_out,
    })

    print("[FULL OOS   ]",
          f"NLL_full_avg={full_out['NLL_full_avg']:.4f} MRR_full={full_out['MRR_full']:.4f} "
          f"FF_full={full_out['FF_full']:.6f} "
          f"R2_full={full_out['McFadden_R2_full']:.4f} Top5_full={full_out['Top5_full']:.3f} Top10_full={full_out['Top10_full']:.3f}")
    print("[SHARES FULL]",
          f"Pear={full_out['Pearson_share']:.4f} Spear={full_out['Spearman_share']:.4f} "
          f"MAE={full_out['MAE_share']:.6f} RMSE={full_out['RMSE_share']:.6f} JS={full_out['JS_share']:.6f}")

    obs_dist, exp_dist = dist_pack
    if (obs_dist is not None) and (exp_dist is not None):
        png = os.path.join(PLOT_DIR, f"distance_plausibility_{bin_name.replace('+', 'plus').replace('-', '_')}.png")
        plot_distance_plausibility(obs_dist, exp_dist, bin_name, png)
        print(f"[PLOT] wrote {png}")

        if trip_distance_df is not None:
            dist_csv = os.path.join(
                DIST_OOS_DIR,
                f"distance_triplevel_{bin_name.replace('+', 'plus').replace('-', '_')}.csv"
            )
            trip_distance_df.to_csv(dist_csv, index=False)
            print(f"[CSV ] wrote {dist_csv}")

    # Hessian-based SEs
    se_vec = approx_hessian_se(
        emu_set=emu_tr,
        X_set=X_tr,
        y=y_tr,
        w=w_tr,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=SE_MAX_TRAIN.get(bin_name, 12000),
        seed=SE_SEED + 1000 * BIN_SEED[bin_name],
        ridge=1e-8,
    )

    se_alpha = float(se_vec[0])
    se_beta  = se_vec[1:]

    zval = alpha_hat / (se_alpha + 1e-12)
    pval = p_value_two_sided_z(zval)
    rows_coef.append({
        "bin": bin_name,
        "param": "alpha_emu",
        "coef": float(alpha_hat),
        "se": float(se_alpha),
        "z": float(zval),
        "p": float(pval),
        "stars": stars(pval),
    })

    for name, b, s in zip(BASE_COLS, beta_hat.tolist(), se_beta.tolist()):
        zval = b / (s + 1e-12)
        pval = p_value_two_sided_z(zval)
        rows_coef.append({
            "bin": bin_name,
            "param": name,
            "coef": float(b),
            "se": float(s),
            "z": float(zval),
            "p": float(pval),
            "stars": stars(pval),
        })

    if share_pack is not None:
        tz_ids = share_pack["tz_ids"]
        S_obs  = share_pack["S_obs"]
        S_pred = share_pack["S_pred"]
        Aj     = share_pack["Aj"]
        orig_i = share_pack["orig_idx"]
        w_i    = share_pack["w"]

        L_in = incoming_accessibility_logsum(
            tz_ids=tz_ids,
            emu_mat=emu_mat,
            alpha=float(alpha_hat),
            orig_idx=orig_i,
            w=w_i,
            zone_to_idx=zone_to_idx,
        )

        safe_bin = bin_name.replace("+", "plus").replace("-", "_")

        png = os.path.join(PLOT_DIR, f"attr_vs_access_{safe_bin}.png")
        plot_attr_vs_access(Aj, L_in, S_pred, bin_name, png)
        print(f"[PLOT] wrote {png}")

        tz_out = tz_feat[[TZ_KEY, "geometry"]].copy()
        tz_out = tz_out.set_index(TZ_KEY).loc[tz_ids].reset_index()

        tz_out[f"A_attr_{safe_bin}"] = Aj
        tz_out[f"S_obs_{safe_bin}"]  = S_obs
        tz_out[f"S_pred_{safe_bin}"] = S_pred
        tz_out[f"L_in_{safe_bin}"]   = L_in

        png1 = os.path.join(PLOT_DIR, f"map_S_pred_{safe_bin}.png")
        png2 = os.path.join(PLOT_DIR, f"map_A_attr_{safe_bin}.png")

        plot_share_maps(tz_out, f"S_pred_{safe_bin}", bin_name, png1, "Predicted destination share (OOS subset, full denom)")
        plot_share_maps(tz_out, f"A_attr_{safe_bin}", bin_name, png2, "Attractivity index A_j = X_j beta (z-scale)")
        print(f"[PLOT] wrote {png1}")
        print(f"[PLOT] wrote {png2}")

        layer_name = f"TZ_{safe_bin}"
        tz_out.to_file(GPKG_OUT, layer=layer_name, driver="GPKG")
        print(f"[GPKG] wrote layer {layer_name} -> {GPKG_OUT}")


# ============================================================
# Save summary tables
# ============================================================
df_sampled = pd.DataFrame(rows_sampled)
df_full    = pd.DataFrame(rows_full)
df_coef    = pd.DataFrame(rows_coef)

p1 = os.path.join(OUT_DIR, "metrics_sampled.csv")
p2 = os.path.join(OUT_DIR, "metrics_full_and_shares.csv")
p3 = os.path.join(OUT_DIR, "coef_table.csv")

df_sampled.to_csv(p1, index=False)
df_full.to_csv(p2, index=False)
df_coef.to_csv(p3, index=False)


# ============================================================
# Weighted averages (OOS, WP-weighted) across all age bins
#   - FULL metrics: weight by WP_full_eval
#   - sampled metrics: weight by sum_WP (OOS)
# ============================================================
def wp_weighted_avg(df, weight_col, metric_cols):
    w = df[weight_col].to_numpy(dtype=np.float64)
    out = {}
    W = float(np.sum(w))
    out["WP_sum"] = W
    for c in metric_cols:
        x = df[c].to_numpy(dtype=np.float64)
        out[c] = float(np.sum(w * x) / (W + 1e-12))
    return out

full_oos = df_full.copy()
full_metrics = [
    "NLL_full_avg", "MRR_full", "FF_full", "McFadden_R2_full", "Top5_full", "Top10_full",
    "Pearson_share", "Spearman_share", "MAE_share", "RMSE_share", "JS_share",
    "dist_obs_mean_km", "dist_pred_mean_km", "dist_mean_diff_km",
]
full_metrics = [c for c in full_metrics if c in full_oos.columns]

full_avg = wp_weighted_avg(full_oos, "WP_full_eval", full_metrics)
full_avg["group"] = "ALL_BINS_SHORT"
full_avg["N_full_sum"] = int(full_oos["N_full_eval"].sum())
full_avg["NLL_full_sum"] = float(full_oos["NLL_full_sum"].sum())

samp_oos = df_sampled[df_sampled["split"] == "oos"].copy()
samp_metrics = ["NLL_avg", "MRR", "Top5", "Top10", "FF"]
samp_avg = wp_weighted_avg(samp_oos, "sum_WP", samp_metrics)
samp_avg["WP_sum_sampled"] = samp_avg.pop("WP_sum")
samp_avg["N_sampled_sum"] = int(samp_oos["N"].sum())
samp_avg["NLL_sum_sampled"] = float(samp_oos["NLL_sum"].sum())
samp_avg["group"] = "ALL_BINS_SHORT"

weighted = {**full_avg, **samp_avg}
df_weighted = pd.DataFrame([weighted])

p4 = os.path.join(OUT_DIR, "metrics_weighted_averages.csv")
df_weighted.to_csv(p4, index=False)


# ============================================================
# Final prints
# ============================================================
print("\n" + "=" * 110)
print("[OK] wrote", p1)
print("[OK] wrote", p2)
print("[OK] wrote", p3)
print("[OK] wrote", p4)
print("[OK] wrote", GPKG_OUT)
print("[OK] plots in", PLOT_DIR)
print("[OK] trip-level distance CSVs in", DIST_OOS_DIR)
print("=" * 110)

print("\nSAMPLED OOS METRICS (weighted):")
print(df_sampled[df_sampled["split"] == "oos"][["bin", "NLL_sum", "NLL_avg", "MRR", "Top5", "Top10", "FF", "N", "J", "sum_WP"]]
      .sort_values("bin").to_string(index=False))

print("\nFULL OOS METRICS + SHARE DIAGNOSTICS:")
keep_cols = [
    "bin", "NLL_full_sum", "NLL_full_avg", "MRR_full", "FF_full", "McFadden_R2_full", "Top5_full", "Top10_full",
    "Pearson_share", "Spearman_share", "MAE_share", "RMSE_share", "JS_share",
    "N_full_eval", "WP_full_eval", "J_full",
    "dist_obs_mean_km", "dist_pred_mean_km", "dist_mean_diff_km",
    "dist_obs_p50_km", "dist_pred_p50_km", "dist_obs_p90_km", "dist_pred_p90_km",
]
cols_exist = [c for c in keep_cols if c in df_full.columns]
print(df_full[cols_exist].sort_values("bin").to_string(index=False))

print("\nWEIGHTED AVERAGES across all bins (OOS, WP-weighted):")
print(df_weighted.to_string(index=False))

## Leave 1 variable out

In [ ]:
# ============================================================
# VARIABLE SENSITIVITY (LEAVE-ONE-ATTRACTIVITY-VARIABLE-OUT)
# BASELINE ESTIMATION + FULL REPORT
# (withLIE, byPerson, log1p, global z, EMU)
#
# For each variable v in BASE_COLS:
#   - remove v from attractivity specification
#   - re-estimate the baseline model on the 4 standard segments:
#         YS, OS, YL, OL
#   - keep everything else identical to the final baseline
#
# Outputs:
#   ModelRuns/variablesensitivity/
#       summary_variable_sensitivity.csv
#       drop_<varname>/
#           metrics_sampled.csv
#           metrics_full_and_shares.csv
#           coef_table.csv
#           tz_outputs.gpkg
#           plots/*.png
#           distance_oos_triplevel/*.csv
#           Result/
#               Attractivity.csv
#               Attractivity.gpkg
#               utilities_by_segment.omx
#
# IMPORTANT:
#   - Uses global z-standardization on FULL TZ universe
#   - Deterministic seeds
#   - FULL-choice-set evaluation is chunked and optionally capped
#   - Coefficients are estimated exactly as baseline
#   - SE / z / p / stars are Hessian-based post-estimation values
# ============================================================

import os
import math
import random
import re
import shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import tables as tb

import torch
import openmatrix as omx
from pathlib import Path

import matplotlib.pyplot as plt


# -----------------------------
# CONFIG (same as baseline final)
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"
DIST_COL  = "dist_km"

TRIPS_DIR = "Trips/byPerson"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

# Impedance (EMU)
UTIL_OMX      = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME  = "utility_emu"

# Distance proxy for plausibility (average car-network distance)
DIST_OMX      = READY_DIR / "distance_avg_2023_ready.omx"
DIST_MAT_NAME = "avg_distances_ready"

MAP_NAME = "NO"

ROOT_OUT_DIR = "ModelRuns/variablesensitivity"

# Core baseline decisions
K = 1000
EPOCHS = 30
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# Full-choice-set evaluation subset sizes
FULL_EVAL_MAX = {
    "YS": 20000,
    "OS": 8000,
    "YL": 8000,
    "OL": 8000,
}
FULL_EVAL_SEED  = 777
FULL_EVAL_CHUNK = 256

# Hessian SE subset sizes
SE_MAX_TRAIN = {
    "YS": 40000,
    "OS": 25000,
    "YL": 12000,
    "OL": 6000,
}
SE_SEED = 2024

BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

TOPKS_SAMPLED = (5, 10)
TOPKS_FULL    = (5, 10)

BASE_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]


# ============================================================
# Reproducibility helper
# ============================================================
def set_all_seeds(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Small helpers
# ============================================================
def safe_name(x: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_]+", "_", x)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300); q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def p_value_two_sided_z(z):
    return 2.0 * (1.0 - norm_cdf(abs(float(z))))

def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.1:
        return "."
    return ""


# ============================================================
# IO helpers
# ============================================================
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()

    if DIST_COL in df.columns:
        df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")

    return df

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


# ============================================================
# Preprocess helpers
# ============================================================
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd


# ============================================================
# Sampling + design builder
# ============================================================
def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0]  = c
        dest_set[i, 1:] = alts
    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]
    dest_set2 = dest_set[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
        dest_set2,
        orig_idx2,
        dest_idx2,
    )


# ============================================================
# Model
# ============================================================
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)

            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum()

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted-avg NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(5,10), want_ff=True):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        P = torch.exp(logP)

        chosen_logp = logP[:, 0]
        chosen_p    = P[:, 0]
        sum_w = float(w.sum().cpu())

        nll_sum = float((-(w * chosen_logp)).sum().cpu())
        nll_avg = nll_sum / (sum_w + 1e-9)

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))

        out = {
            "NLL_sum": nll_sum,
            "NLL_avg": nll_avg,
            "MRR": mrr,
            "sum_WP": sum_w,
            "N": int(V.shape[0]),
            "J": int(V.shape[1]),
        }

        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))

        if want_ff:
            out["FF"] = float((w * chosen_p).sum().cpu() / (sum_w + 1e-9))

    return out


# ============================================================
# FULL-choice-set evaluation
# ============================================================
def full_eval_and_shares_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    dist_mat: np.ndarray | None,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(5,10),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w    = df[WP_COL].to_numpy(dtype=np.float64)

    has_dist = (DIST_COL in df.columns) and df[DIST_COL].notna().any()
    obs_dist = df[DIST_COL].to_numpy(dtype=np.float64) if has_dist else None

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    if has_dist:
        good = good & np.isfinite(obs_dist) & (obs_dist >= 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w        = w[good]
    if has_dist:
        obs_dist = obs_dist[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}, None, (None, None), None

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass  = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    ll_model = 0.0
    mrr_num = 0.0
    ff_num = 0.0
    top_num = {k: 0.0 for k in topKs}
    ranks   = np.empty(len(df), dtype=np.int64)

    exp_dist = np.zeros(len(df), dtype=np.float64) if (dist_mat is not None and has_dist) else None

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse
        p_chosen = math.exp(logp_chosen)

        nll_num  += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)
        ff_num   += wn * p_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        if exp_dist is not None:
            expd = 0.0

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

            if exp_dist is not None:
                dist_blk = dist_mat[o, j0:j1].astype(np.float64)
                expd += float(np.sum(Pblk * dist_blk))

        if exp_dist is not None:
            exp_dist[n] = expd

    nll_full = float(nll_num)
    nll_full_avg = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    ff_full = float(ff_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs  = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)

    pear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="pearson")) if len(S_obs) > 2 else np.nan
    spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    mae  = float(np.mean(np.abs(S_obs - S_pred)))
    rmse = float(np.sqrt(np.mean((S_obs - S_pred) ** 2)))
    js   = float(js_div(S_obs, S_pred))

    dist_summary = {}
    if exp_dist is not None:
        wnorm = w / (sum_w + 1e-12)
        obs_mean = float(np.sum(wnorm * obs_dist))
        pred_mean = float(np.sum(wnorm * exp_dist))
        dist_summary = {
            "dist_obs_mean_km": obs_mean,
            "dist_pred_mean_km": pred_mean,
            "dist_mean_diff_km": float(pred_mean - obs_mean),
            "dist_obs_p50_km": float(np.quantile(obs_dist, 0.50)),
            "dist_pred_p50_km": float(np.quantile(exp_dist, 0.50)),
            "dist_obs_p90_km": float(np.quantile(obs_dist, 0.90)),
            "dist_pred_p90_km": float(np.quantile(exp_dist, 0.90)),
        }

    out = {
        "NLL_full_sum": nll_full,
        "NLL_full_avg": nll_full_avg,
        "MRR_full": mrr_full,
        "FF_full": ff_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Pearson_share": pear,
        "Spearman_share": spear,
        "MAE_share": mae,
        "RMSE_share": rmse,
        "JS_share": js,
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    out.update(dist_summary)

    trip_distance_df = None
    if exp_dist is not None:
        trip_distance_df = pd.DataFrame({
            "orig_zone": df[ORIG_COL].to_numpy(dtype=int),
            "dest_zone_obs": df[DEST_COL].to_numpy(dtype=int),
            "WP": w.astype(np.float64),
            "dist_obs_km": obs_dist.astype(np.float64),
            "dist_exp_km": exp_dist.astype(np.float64),
        })

    return out, {
        "tz_ids": all_zone_ids,
        "S_obs": S_obs,
        "S_pred": S_pred,
        "Aj": Aj.astype(np.float64),
        "orig_idx": orig_idx,
        "w": w,
    }, (obs_dist, exp_dist), trip_distance_df


# ============================================================
# Hessian-based SEs
# ============================================================
def approx_hessian_se(
    emu_set: torch.Tensor,
    X_set: torch.Tensor,
    y: torch.Tensor,
    w: torch.Tensor,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int,
    seed: int,
    ridge: float = 1e-8,
):
    N = emu_set.shape[0]
    rng = np.random.default_rng(seed)

    if N > max_n:
        idx = rng.choice(N, size=max_n, replace=False)
        idx = torch.tensor(idx, dtype=torch.long, device=DEVICE)
        emu = emu_set[idx]
        X   = X_set[idx]
        yy  = y[idx]
        ww  = w[idx]
    else:
        emu, X, yy, ww = emu_set, X_set, y, w

    sum_w = float(ww.sum().detach().cpu())
    Pdim = X.shape[2]

    alpha = torch.tensor([alpha_hat], dtype=DTYPE, device=DEVICE, requires_grad=True)
    beta  = torch.tensor(beta_hat, dtype=DTYPE, device=DEVICE, requires_grad=True)

    def loss_avg(a, b):
        V = a * emu + torch.einsum("bjp,p->bj", X, b)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        ll = logP[torch.arange(V.shape[0], device=DEVICE), yy]
        return (-(ww * ll)).sum() / (ww.sum() + 1e-12)

    L = loss_avg(alpha, beta)
    g = torch.autograd.grad(L, [alpha, beta], create_graph=True)
    g_vec = torch.cat([g[0].reshape(1), g[1].reshape(-1)], dim=0)

    H_avg = torch.zeros((1 + Pdim, 1 + Pdim), dtype=torch.float64, device=DEVICE)
    for i in range(1 + Pdim):
        gi = g_vec[i]
        hi = torch.autograd.grad(gi, [alpha, beta], retain_graph=True)
        hi_vec = torch.cat([hi[0].reshape(1), hi[1].reshape(-1)], dim=0)
        H_avg[i, :] = hi_vec.detach().to(torch.float64)

    H_avg = H_avg.cpu().numpy()
    H_sum = sum_w * H_avg

    H_sum = 0.5 * (H_sum + H_sum.T)
    H_sum = H_sum + ridge * np.eye(H_sum.shape[0], dtype=np.float64)

    Vcov = np.linalg.pinv(H_sum)
    se = np.sqrt(np.clip(np.diag(Vcov), 0.0, np.inf))
    return se


# ============================================================
# Destination-level outputs + plots
# ============================================================
def incoming_accessibility_logsum(
    tz_ids: np.ndarray,
    emu_mat: np.ndarray,
    alpha: float,
    orig_idx: np.ndarray,
    w: np.ndarray,
    zone_to_idx: dict,
):
    Msize = emu_mat.shape[0]
    w_by_o = np.zeros(Msize, dtype=np.float64)
    for o, wn in zip(orig_idx, w):
        w_by_o[int(o)] += float(wn)

    origins = np.where(w_by_o > 0)[0]
    w_o = w_by_o[origins]
    w_o = w_o / (w_o.sum() + 1e-12)

    L = np.full(len(tz_ids), np.nan, dtype=np.float64)

    tz_omx = np.array([zone_to_idx.get(int(z), -1) for z in tz_ids], dtype=int)
    good = (tz_omx >= 0)
    tz_omx_good = tz_omx[good]

    good_pos = np.where(good)[0]
    for local_idx, j in enumerate(tz_omx_good):
        vals = alpha * emu_mat[origins, j].astype(np.float64)
        m = np.max(vals)
        s = np.sum(w_o * np.exp(vals - m))
        L[good_pos[local_idx]] = float(m + np.log(s + 1e-300))

    return L

def plot_distance_plausibility(obs_dist, exp_dist, seg, out_png):
    obs_dist = np.asarray(obs_dist, dtype=np.float64)
    exp_dist = np.asarray(exp_dist, dtype=np.float64)

    finite = np.isfinite(obs_dist) & np.isfinite(exp_dist)
    obs_dist = obs_dist[finite]
    exp_dist = exp_dist[finite]

    if len(obs_dist) == 0:
        return

    obs_mean = float(np.mean(obs_dist))
    exp_mean = float(np.mean(exp_dist))

    xmax = float(max(np.max(obs_dist), np.max(exp_dist)))
    bins = np.linspace(0.0, xmax, 51)

    plt.figure()
    plt.hist(obs_dist, bins=bins, density=True, alpha=0.6,
             label=f"Observed dist_km (mean={obs_mean:.2f})")
    plt.hist(exp_dist, bins=bins, density=True, alpha=0.6,
             label=f"Predicted E[distance] (mean={exp_mean:.2f})")
    plt.xlabel("Distance (km)")
    plt.ylabel("Density")
    plt.title(f"Distance plausibility | {seg} | obs mean={obs_mean:.2f}, pred mean={exp_mean:.2f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_attr_vs_access(Aj, L_in, S_pred, seg, out_png):
    plt.figure()
    s = 20 + 4000 * (S_pred / (S_pred.max() + 1e-12))
    plt.scatter(L_in, Aj, s=s, alpha=0.5)
    plt.xlabel("Incoming accessibility logsum (from origins, using EMU)")
    plt.ylabel("Attractivity index A_j = X_j beta (z-scale)")
    plt.title(f"Attractivity vs incoming accessibility | {seg}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_share_maps(tz_gdf, col, seg, out_png, title):
    plt.figure()
    ax = tz_gdf.plot(column=col, legend=True)
    ax.set_axis_off()
    plt.title(f"{title} | {seg}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


# ============================================================
# One full leave-one-out run
# ============================================================
def run_leave_one_out(drop_col: str,
                      tz_full: gpd.GeoDataFrame,
                      emu_mat: np.ndarray,
                      dist_mat: np.ndarray,
                      zone_to_idx: dict,
                      idx_to_zone: np.ndarray):
    keep_cols = [c for c in BASE_COLS if c != drop_col]
    if len(keep_cols) == 0:
        raise ValueError("Cannot estimate a model with zero attractivity variables.")

    drop_tag = safe_name(drop_col)
    OUT_DIR = os.path.join(ROOT_OUT_DIR, f"drop_{drop_tag}")
    PLOT_DIR = os.path.join(OUT_DIR, "plots")
    DIST_OOS_DIR = os.path.join(OUT_DIR, "distance_oos_triplevel")
    RESULT_DIR = os.path.join(OUT_DIR, "Result")

    ATTR_CSV_OUT  = os.path.join(RESULT_DIR, "Attractivity.csv")
    ATTR_GPKG_OUT = os.path.join(RESULT_DIR, "Attractivity.gpkg")
    UTIL_SEG_OMX  = os.path.join(RESULT_DIR, "utilities_by_segment.omx")

    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)
    os.makedirs(DIST_OOS_DIR, exist_ok=True)
    os.makedirs(RESULT_DIR, exist_ok=True)

    print("\n" + "=" * 120)
    print(f"[VARIABLE SENSITIVITY] DROP = {drop_col}")
    print(f"[VARIABLE SENSITIVITY] KEEP = {keep_cols}")
    print(f"[OUT] {OUT_DIR}")
    print("=" * 120)

    # prepare TZ with only kept cols standardized
    tz_feat = tz_full[["npvm_id", "geometry"] + keep_cols].copy()
    tz_feat, Z_COLS, MU_ALL, SD_ALL = standardize_all_zones(tz_feat, keep_cols, suffix="_z")

    rows_sampled = []
    rows_full    = []
    rows_coef    = []

    seg_alpha = {}
    seg_beta  = {}
    seg_Aj_tz = {}
    seg_Aj_by_omx = {}

    tz_ids_master = np.array(sorted(tz_feat["npvm_id"].astype(int).unique().tolist()), dtype=int)

    GPKG_OUT = os.path.join(OUT_DIR, "tz_outputs.gpkg")
    if os.path.exists(GPKG_OUT):
        os.remove(GPKG_OUT)

    for seg in SEGMENTS:
        print("\n" + "#" * 120)
        print(f"# DROP = {drop_col} | SEGMENT = {seg} | K={K} | epochs={EPOCHS}")
        print("#" * 120)

        set_all_seeds(BASE_SEED + 1000 * SEG_SEED[seg])

        train_df = load_segment_csv(TRIPS_DIR, seg, "train")
        oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

        seed_tr = BASE_SEED + 1000 * SEG_SEED[seg] + 10
        seed_te = BASE_SEED + 1000 * SEG_SEED[seg] + 20
        seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg]

        emu_tr, X_tr, y_tr, w_tr, train_df2, destset_tr, origidx_tr, destidx_tr = build_design(
            train_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_tr
        )
        emu_te, X_te, y_te, w_te, oos_df2, destset_te, origidx_te, destidx_te = build_design(
            oos_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_te
        )

        print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
        print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

        alpha_hat, beta_hat = train_mnl(
            emu_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
        )

        seg_alpha[seg] = float(alpha_hat)
        seg_beta[seg]  = beta_hat.copy()

        Xz_all_master = tz_feat.set_index("npvm_id").loc[tz_ids_master, Z_COLS].to_numpy(dtype=np.float32)
        Aj_master = Xz_all_master @ beta_hat.astype(np.float32)
        seg_Aj_tz[seg] = Aj_master.astype(np.float64)

        Msize = emu_mat.shape[0]
        Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
        for zid, val in zip(tz_ids_master, Aj_master):
            jidx = zone_to_idx.get(int(zid), None)
            if jidx is not None and 0 <= jidx < Msize:
                Aj_by_omxidx[jidx] = float(val)
        seg_Aj_by_omx[seg] = Aj_by_omxidx

        met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)
        met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)

        rows_sampled.append({
            "dropped_variable": drop_col,
            "segment": seg,
            "split": "train",
            "alpha_emu": float(alpha_hat),
            "K": K,
            "n_attr_vars": len(keep_cols),
            **met_tr,
        })
        rows_sampled.append({
            "dropped_variable": drop_col,
            "segment": seg,
            "split": "oos",
            "alpha_emu": float(alpha_hat),
            "K": K,
            "n_attr_vars": len(keep_cols),
            **met_te,
        })

        print("[SAMPLED OOS ]",
              f"NLL_sum={met_te['NLL_sum']:.2f} NLL_avg={met_te['NLL_avg']:.4f} "
              f"MRR={met_te['MRR']:.4f} Top5={met_te['Top5']:.3f} Top10={met_te['Top10']:.3f} FF={met_te['FF']:.4f}")

        max_n = FULL_EVAL_MAX.get(seg, None)
        full_out, share_pack, dist_pack, trip_distance_df = full_eval_and_shares_oos_subset(
            oos_df=oos_df2,
            tz_feat=tz_feat,
            z_cols=Z_COLS,
            emu_mat=emu_mat,
            dist_mat=dist_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=max_n,
            seed=seed_full,
            chunk=FULL_EVAL_CHUNK,
            topKs=TOPKS_FULL,
        )

        rows_full.append({
            "dropped_variable": drop_col,
            "segment": seg,
            "alpha_emu": float(alpha_hat),
            "K": K,
            "n_attr_vars": len(keep_cols),
            **full_out,
        })

        print("[FULL OOS   ]",
              f"NLL_full_avg={full_out['NLL_full_avg']:.4f} MRR_full={full_out['MRR_full']:.4f} "
              f"FF_full={full_out['FF_full']:.6f} "
              f"R2_full={full_out['McFadden_R2_full']:.4f} Top5_full={full_out['Top5_full']:.3f} Top10_full={full_out['Top10_full']:.3f}")
        print("[SHARES FULL]",
              f"Pear={full_out['Pearson_share']:.4f} Spear={full_out['Spearman_share']:.4f} "
              f"MAE={full_out['MAE_share']:.6f} RMSE={full_out['RMSE_share']:.6f} JS={full_out['JS_share']:.6f}")

        obs_dist, exp_dist = dist_pack
        if (obs_dist is not None) and (exp_dist is not None):
            png = os.path.join(PLOT_DIR, f"distance_plausibility_{seg}.png")
            plot_distance_plausibility(obs_dist, exp_dist, seg, png)
            print(f"[PLOT] wrote {png}")

            if trip_distance_df is not None:
                dist_csv = os.path.join(DIST_OOS_DIR, f"distance_triplevel_{seg}.csv")
                trip_distance_df.to_csv(dist_csv, index=False)
                print(f"[CSV ] wrote {dist_csv}")

        se_vec = approx_hessian_se(
            emu_set=emu_tr,
            X_set=X_tr,
            y=y_tr,
            w=w_tr,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=SE_MAX_TRAIN.get(seg, 20000),
            seed=SE_SEED + 1000 * SEG_SEED[seg],
            ridge=1e-8,
        )

        se_alpha = float(se_vec[0])
        se_beta  = se_vec[1:]

        zval = alpha_hat / (se_alpha + 1e-12)
        pval = p_value_two_sided_z(zval)
        rows_coef.append({
            "dropped_variable": drop_col,
            "segment": seg,
            "param": "alpha_emu",
            "coef": float(alpha_hat),
            "se": float(se_alpha),
            "z": float(zval),
            "p": float(pval),
            "stars": stars(pval),
        })

        for name, b, s in zip(keep_cols, beta_hat.tolist(), se_beta.tolist()):
            zval = b / (s + 1e-12)
            pval = p_value_two_sided_z(zval)
            rows_coef.append({
                "dropped_variable": drop_col,
                "segment": seg,
                "param": name,
                "coef": float(b),
                "se": float(s),
                "z": float(zval),
                "p": float(pval),
                "stars": stars(pval),
            })

        if share_pack is not None:
            tz_ids  = share_pack["tz_ids"]
            S_obs   = share_pack["S_obs"]
            S_pred  = share_pack["S_pred"]
            Aj      = share_pack["Aj"]
            orig_i  = share_pack["orig_idx"]
            w_i     = share_pack["w"]

            L_in = incoming_accessibility_logsum(
                tz_ids=tz_ids,
                emu_mat=emu_mat,
                alpha=float(alpha_hat),
                orig_idx=orig_i,
                w=w_i,
                zone_to_idx=zone_to_idx,
            )

            png = os.path.join(PLOT_DIR, f"attr_vs_access_{seg}.png")
            plot_attr_vs_access(Aj, L_in, S_pred, seg, png)
            print(f"[PLOT] wrote {png}")

            tz_out = tz_feat[["npvm_id", "geometry"]].copy()
            tz_out = tz_out.set_index("npvm_id").loc[tz_ids].reset_index()

            tz_out[f"A_attr_{seg}"] = Aj
            tz_out[f"S_obs_{seg}"]  = S_obs
            tz_out[f"S_pred_{seg}"] = S_pred
            tz_out[f"L_in_{seg}"]   = L_in

            png1 = os.path.join(PLOT_DIR, f"map_S_pred_{seg}.png")
            png2 = os.path.join(PLOT_DIR, f"map_A_attr_{seg}.png")
            plot_share_maps(tz_out, f"S_pred_{seg}", seg, png1, "Predicted destination share (OOS subset, full denom)")
            plot_share_maps(tz_out, f"A_attr_{seg}", seg, png2, "Attractivity index A_j = X_j beta (z-scale)")
            print(f"[PLOT] wrote {png1}")
            print(f"[PLOT] wrote {png2}")

            layer_name = f"TZ_{seg}"
            tz_out.to_file(GPKG_OUT, layer=layer_name, driver="GPKG")
            print(f"[GPKG] wrote layer {layer_name} -> {GPKG_OUT}")

    # save local tables
    df_sampled = pd.DataFrame(rows_sampled)
    df_full    = pd.DataFrame(rows_full)
    df_coef    = pd.DataFrame(rows_coef)

    p1 = os.path.join(OUT_DIR, "metrics_sampled.csv")
    p2 = os.path.join(OUT_DIR, "metrics_full_and_shares.csv")
    p3 = os.path.join(OUT_DIR, "coef_table.csv")

    df_sampled.to_csv(p1, index=False)
    df_full.to_csv(p2, index=False)
    df_coef.to_csv(p3, index=False)

    print("\n" + "=" * 110)
    print("[OK] wrote", p1)
    print("[OK] wrote", p2)
    print("[OK] wrote", p3)
    print("[OK] wrote", GPKG_OUT)
    print("[OK] plots in", PLOT_DIR)
    print("[OK] trip-level distance CSVs in", DIST_OOS_DIR)
    print("=" * 110)

    print("\nSAMPLED OOS METRICS (weighted):")
    print(df_sampled[df_sampled["split"] == "oos"][[
        "dropped_variable", "segment", "NLL_sum", "NLL_avg", "MRR", "Top5", "Top10", "FF", "N", "J", "sum_WP"
    ]].sort_values(["segment"]).to_string(index=False))

    print("\nFULL OOS METRICS + SHARE DIAGNOSTICS:")
    keep_print = [
        "dropped_variable", "segment", "NLL_full_sum", "NLL_full_avg", "MRR_full", "FF_full", "McFadden_R2_full",
        "Top5_full", "Top10_full", "Pearson_share", "Spearman_share", "MAE_share", "RMSE_share", "JS_share",
        "N_full_eval", "WP_full_eval", "J_full",
        "dist_obs_mean_km", "dist_pred_mean_km", "dist_mean_diff_km",
        "dist_obs_p50_km", "dist_pred_p50_km", "dist_obs_p90_km", "dist_pred_p90_km",
    ]
    cols_exist = [c for c in keep_print if c in df_full.columns]
    print(df_full[cols_exist].sort_values(["segment"]).to_string(index=False))

    # export Result package
    print("\n" + "=" * 110)
    print("[EXPORT] Building Result package (Attractivity + utilities omx) ...")

    attr_df = pd.DataFrame({
        "zone_id": tz_ids_master.astype(int),
        "omx_idx": np.array([zone_to_idx.get(int(z), -1) for z in tz_ids_master], dtype=int),
        "Attr_YS": seg_Aj_tz["YS"].astype(np.float64),
        "Attr_OS": seg_Aj_tz["OS"].astype(np.float64),
        "Attr_YL": seg_Aj_tz["YL"].astype(np.float64),
        "Attr_OL": seg_Aj_tz["OL"].astype(np.float64),
    })
    attr_df.to_csv(ATTR_CSV_OUT, index=False)
    print("[OK] wrote", ATTR_CSV_OUT)

    if os.path.exists(ATTR_GPKG_OUT):
        os.remove(ATTR_GPKG_OUT)

    geom = tz_feat[["npvm_id", "geometry"]].drop_duplicates().set_index("npvm_id").loc[tz_ids_master].reset_index()
    geom = geom.rename(columns={"npvm_id": "zone_id"})
    attr_gdf = gpd.GeoDataFrame(
        geom.merge(attr_df, on="zone_id", how="left"),
        geometry="geometry",
        crs=tz_feat.crs,
    )
    attr_gdf.to_file(ATTR_GPKG_OUT, driver="GPKG")
    print("[OK] wrote", ATTR_GPKG_OUT)

    if os.path.exists(UTIL_SEG_OMX):
        os.remove(UTIL_SEG_OMX)

    Msize = emu_mat.shape[0]
    print(f"[OMX-OUT] Creating {UTIL_SEG_OMX} with 4 matrices of shape {emu_mat.shape} (float32) ...")

    fout = omx.open_file(UTIL_SEG_OMX, "w")
    try:
        fout.create_mapping(MAP_NAME, idx_to_zone.astype(np.int32))
    except Exception:
        fout.create_mapping(MAP_NAME, [int(x) for x in idx_to_zone.tolist()])

    for seg in SEGMENTS:
        float_atom = tb.Atom.from_dtype(np.dtype("float32"))
        fout.create_matrix(f"utility_{seg}", atom=float_atom, shape=(Msize, Msize))

    ROW_BLOCK = 512
    for seg in SEGMENTS:
        alpha32 = np.float32(seg_alpha[seg])
        Aj_omx = seg_Aj_by_omx[seg].astype(np.float32)

        print(f"[OMX-OUT] Writing utility_{seg} ... alpha={float(seg_alpha[seg]):.6f}")
        mat = fout[f"utility_{seg}"]

        for i0 in range(0, Msize, ROW_BLOCK):
            i1 = min(Msize, i0 + ROW_BLOCK)
            emu_blk = emu_mat[i0:i1, :]
            util_blk = alpha32 * emu_blk + Aj_omx[None, :]
            mat[i0:i1, :] = util_blk.astype(np.float32)

    fout.close()
    print("[OK] wrote", UTIL_SEG_OMX)
    print("[EXPORT] Done. Result files are in:", RESULT_DIR)
    print("=" * 110)

    # return compact summary for global file
    df_samp_oos = df_sampled[df_sampled["split"] == "oos"].copy()
    df_full_oos = df_full.copy()

    merged = df_full_oos.merge(
        df_samp_oos[["segment", "NLL_sum", "NLL_avg", "MRR", "Top5", "Top10", "FF", "N", "J", "sum_WP"]],
        on="segment",
        how="left",
        suffixes=("_fulltbl", "_sampled")
    )

    merged["dropped_variable"] = drop_col
    merged["n_attr_vars"] = len(keep_cols)
    merged["out_dir"] = OUT_DIR
    return merged


# ============================================================
# MAIN
# ============================================================
def main():
    set_all_seeds(BASE_SEED)

    os.makedirs(ROOT_OUT_DIR, exist_ok=True)

    print("[LOAD] TZ features (withLIE)...")
    tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
    if "npvm_id" not in tz.columns:
        raise ValueError("TZ layer missing 'npvm_id'")
    tz["npvm_id"] = pd.to_numeric(tz["npvm_id"], errors="coerce")
    tz = tz.dropna(subset=["npvm_id"]).copy()
    tz["npvm_id"] = tz["npvm_id"].astype(int)

    need_cols = ["npvm_id"] + BASE_COLS
    miss = [c for c in need_cols if c not in tz.columns]
    if miss:
        raise ValueError(f"TZ layer missing {miss}")

    for c in BASE_COLS:
        tz[c] = pd.to_numeric(tz[c], errors="coerce")

    print("[LOAD] EMU matrix + mapping (withLIE)...")
    emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)
    print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

    print("[LOAD] Average distance matrix (for distance plausibility)...")
    dist_mat, zone_to_idx_dist, idx_to_zone_dist = load_matrix_and_mapping(DIST_OMX, DIST_MAT_NAME, MAP_NAME)
    if len(idx_to_zone_dist) != len(idx_to_zone) or not np.all(idx_to_zone_dist == idx_to_zone):
        print("[WARN] Distance OMX mapping differs from utilities OMX mapping. Distance plausibility may be misaligned.")

    summary_rows = []

    for drop_col in BASE_COLS:
        merged_summary = run_leave_one_out(
            drop_col=drop_col,
            tz_full=tz,
            emu_mat=emu_mat,
            dist_mat=dist_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
        )
        summary_rows.append(merged_summary)

    df_summary = pd.concat(summary_rows, axis=0, ignore_index=True)

    summary_path = os.path.join(ROOT_OUT_DIR, "summary_variable_sensitivity.csv")
    df_summary.to_csv(summary_path, index=False)

    print("\n" + "=" * 120)
    print("[DONE] Variable sensitivity finished.")
    print("[DONE] Global summary:", summary_path)
    print("=" * 120)

    print("\nSUMMARY PREVIEW:")
    preview_cols = [
        "dropped_variable", "segment",
        "NLL_avg", "MRR", "Top5", "Top10", "FF",
        "NLL_full_avg", "MRR_full", "FF_full", "McFadden_R2_full",
        "Top5_full", "Top10_full",
        "Pearson_share", "Spearman_share", "MAE_share", "RMSE_share", "JS_share",
        "dist_obs_mean_km", "dist_pred_mean_km", "dist_mean_diff_km",
        "out_dir"
    ]
    preview_cols = [c for c in preview_cols if c in df_summary.columns]
    print(df_summary[preview_cols].head(20).to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
BASELINE_CSV = "ModelRuns/BaselineFinal2/metrics_full_and_shares.csv"
DROP1_CSV    = "ModelRuns/variablesensitivity/summary_variable_sensitivity.csv"
OUT_CSV      = "ModelRuns/variablesensitivity/summary_variable_sensitivity_deltas_vs_baselinefinal2.csv"

METRICS = [
    "NLL_full_avg",
    "MRR_full",
    "McFadden_R2_full",
    "Pearson_share",
    "Spearman_share",
    "JS_share",
]

SEGMENTS = ["YS", "OS", "YL", "OL"]

# ============================================================
# LOAD
# ============================================================
base = pd.read_csv(BASELINE_CSV)
drop1 = pd.read_csv(DROP1_CSV)

base = base[base["segment"].isin(SEGMENTS)].copy()
drop1 = drop1[drop1["segment"].isin(SEGMENTS)].copy()

# check columns
missing_base = [c for c in ["segment"] + METRICS if c not in base.columns]
missing_drop = [c for c in ["segment", "dropped_variable"] + METRICS if c not in drop1.columns]

if missing_base:
    raise ValueError(f"Missing columns in baseline file: {missing_base}")
if missing_drop:
    raise ValueError(f"Missing columns in drop1 file: {missing_drop}")

# ============================================================
# BASELINE TABLE: one row per segment
# ============================================================
base_tbl = base[["segment"] + METRICS].copy()
base_tbl = base_tbl.drop_duplicates(subset=["segment"]).copy()

base_tbl = base_tbl.rename(columns={m: f"{m}_baseline" for m in METRICS})

# ============================================================
# DROP1 TABLE
# ============================================================
keep_cols = ["dropped_variable", "segment"]
if "out_dir" in drop1.columns:
    keep_cols.append("out_dir")

drop_tbl = drop1[keep_cols + METRICS].copy()
drop_tbl = drop_tbl.rename(columns={m: f"{m}_drop1" for m in METRICS})

# ============================================================
# MERGE
# ============================================================
df = drop_tbl.merge(base_tbl, on="segment", how="left", validate="many_to_one")

# ============================================================
# DELTAS = drop1 - baseline
# ============================================================
for m in METRICS:
    df[f"d{m}"] = df[f"{m}_drop1"] - df[f"{m}_baseline"]

# ============================================================
# FINAL COLUMN ORDER
# ============================================================
ordered_cols = ["dropped_variable", "segment"]
if "out_dir" in df.columns:
    ordered_cols.append("out_dir")

for m in METRICS:
    ordered_cols += [
        f"{m}_drop1",
        f"{m}_baseline",
        f"d{m}",
    ]

remaining = [c for c in df.columns if c not in ordered_cols]
df = df[ordered_cols + remaining]

# ============================================================
# SAVE
# ============================================================
Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False)

print(f"[OK] wrote {OUT_CSV}")
print(df.head(20).to_string(index=False))

# R2 Alternative Impedance

## Beeline

In [ ]:
# ============================================================
# ROBUSTNESS: Impedance proxy comparison (EMU vs Beeline)
# byPerson, BASE_LOG1P_COLS, global Z, uniform sampling
#
# Fixed:
#   - K = 1000
#   - 50 epochs
#   - sampled TRAIN + sampled OOS metrics
#   - FULL-choice-set metrics on OOS subset (all zones as alternatives)
#
# Two models differ ONLY by impedance matrix:
#   A) EMU (utilities_2023_ready.omx : "utility_emu")
#   B) Beeline (beeline_2023_ready.omx : "beeline_km_ready")
#
# Outputs:
#   ModelRuns/ImpedanceRobustness/impedance_compare_long.csv
#   ModelRuns/ImpedanceRobustness/impedance_compare_wide.csv
#   ModelRuns/ImpedanceRobustness/impedance_compare_deltas_BEE_minus_EMU.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path


# -----------------------------
# CONFIG
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TRIPS_DIR = "Trips/byPerson"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

# impedance matrices
UTIL_OMX      = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME  = "utility_emu"

BEE_OMX      = READY_DIR / "beeline_2023_ready_log1p.omx"
BEE_MAT_NAME = "beeline_km_ready_log1p"

MAP_NAME  = "NO"

OUT_DIR = "ModelRuns/ImpedanceRobustness"
os.makedirs(OUT_DIR, exist_ok=True)

# Fixed robustness decision
K = 1000
EPOCHS = 50
BASE_SEED = 123

# Torch
DEVICE = "cpu"
DTYPE  = torch.float32

# Optimization
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# FULL-choice-set evaluation subset sizes (OOS)
FULL_EVAL_MAX = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256

TOPKS = (1, 10, 50, 100)

# -----------------------------
# BASELINE feature list (13)
# -----------------------------
BASE_LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]


# -----------------------------
# Small helpers
# -----------------------------
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300); q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

# -----------------------------
# IO helpers
# -----------------------------
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone

# -----------------------------
# Preprocess helpers
# -----------------------------
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    """
    Z-standardize on the FULL TZ universe (all alternatives),
    i.e., a fixed transformation of alternative attributes.
    """
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)

    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts

    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 imp_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    imp_set = imp_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)  # chosen always first

    return (
        torch.tensor(imp_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2
    )

# -----------------------------
# Model: train/eval on sampled sets
# -----------------------------
def train_mnl(imp_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, J = imp_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            imp_b = imp_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * imp_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state  # (alpha_hat, beta_hat)

def eval_sampled_metrics(imp_set, X_set, w, alpha, beta, topKs=(1,10,50,100)):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * imp_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = int(V.shape[1])
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

        out = {
            "NLL": nll,
            "MRR": mrr,
            "McFadden_R2": float(r2),
            "rank_median": rank_median,
            "J": J,
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))

    return out

# -----------------------------
# FULL-choice-set evaluation on OOS subset (all zones available)
# -----------------------------
def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    imp_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(1,10,50,100),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)
    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = imp_mat.shape[0]
    alpha = float(alpha_hat)

    # Alternative attribute term Aj(j) aligned to OMX indices
    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    # shares on TZ universe
    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}
    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)

    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    ll_model = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    top_num = {k: 0.0 for k in topKs}

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(imp_mat[o, d]) + float(Aj_by_omxidx[d])

        # pass 1: max V
        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * imp_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        # pass 2: sumexp + rank
        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * imp_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        nll_num += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        # pass 3: shares accumulation
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * imp_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    out = {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    return out


# ============================================================
# MAIN
# ============================================================

print("[LOAD] TZ features...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)

need_cols = ["npvm_id"] + BASE_LOG1P_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

tz_feat0 = tz[["npvm_id"] + BASE_LOG1P_COLS + ["geometry"]].copy()
tz_feat0["npvm_id"] = pd.to_numeric(tz_feat0["npvm_id"], errors="coerce")
tz_feat0 = tz_feat0.dropna(subset=["npvm_id"]).copy()
tz_feat0["npvm_id"] = tz_feat0["npvm_id"].astype(int)

print("[PREP] Global z-standardize destination vars on full TZ universe...")
tz_feat0, z_cols, mu_all, sd_all = standardize_all_zones(tz_feat0, BASE_LOG1P_COLS, suffix="_z")

# Load EMU and Beeline, assert same mapping universe
print("[LOAD] EMU impedance...")
emu_mat, zone_to_idx_emu, idx_to_zone_emu = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)

print("[LOAD] Beeline impedance...")
bee_mat, zone_to_idx_bee, idx_to_zone_bee = load_matrix_and_mapping(BEE_OMX, BEE_MAT_NAME, MAP_NAME)

# Safety checks: mapping compatibility
if len(idx_to_zone_emu) != len(idx_to_zone_bee) or not np.all(idx_to_zone_emu == idx_to_zone_bee):
    raise ValueError("EMU and Beeline OMX mappings differ. Ensure both OMX files use identical MAP_NAME mapping order.")
if emu_mat.shape != bee_mat.shape:
    raise ValueError(f"Matrix shapes differ: EMU {emu_mat.shape} vs Beeline {bee_mat.shape}")

# We'll use one consistent mapping dict
zone_to_idx = zone_to_idx_emu
idx_to_zone = idx_to_zone_emu

RUNS = [
    {"name": "EMU", "mat": emu_mat},
    {"name": "BEE", "mat": bee_mat},
]

results = []

for run in RUNS:
    imp_name = run["name"]
    imp_mat  = run["mat"]

    print("\n" + "#" * 110)
    print(f"# IMPEDANCE = {imp_name} | K={K} | epochs={EPOCHS} | byPerson | BASE vars")
    print("#" * 110)

    for seg in SEGMENTS:
        print("\n" + "=" * 95)
        print(f"Segment {seg} | impedance={imp_name}")
        print("=" * 95)

        train_df = load_segment_csv(TRIPS_DIR, seg, "train")
        oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

        # Deterministic seeds (same sampled sets across impedance variants)
        seed_tr = BASE_SEED + 1000 * SEG_SEED[seg] + 10
        seed_te = BASE_SEED + 1000 * SEG_SEED[seg] + 20

        emu_tr, X_tr, y_tr, w_tr, train_df2 = build_design(
            train_df, tz_feat0, z_cols, imp_mat, zone_to_idx, k=K, seed=seed_tr
        )
        emu_te, X_te, y_te, w_te, oos_df2 = build_design(
            oos_df, tz_feat0, z_cols, imp_mat, zone_to_idx, k=K, seed=seed_te
        )

        print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
        print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

        alpha_hat, beta_hat = train_mnl(
            emu_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
        )

        met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS)
        met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS)

        max_n = FULL_EVAL_MAX.get(seg, 8000)
        seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg]  # same subset across impedance variants
        full_met = full_choice_eval_oos_subset(
            oos_df=oos_df2,
            tz_feat_seg=tz_feat0,
            z_cols=z_cols,
            imp_mat=imp_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=max_n,
            seed=seed_full,
            chunk=FULL_EVAL_CHUNK,
            topKs=TOPKS,
        )

        row = {
            "impedance": imp_name,
            "segment": seg,
            "K": K,
            "alpha_imp": float(alpha_hat),

            "NLL_train": met_tr["NLL"],
            "MRR_train": met_tr["MRR"],
            "McFadden_R2_train": met_tr["McFadden_R2"],
            "rank_median_train": met_tr["rank_median"],
            "Top1_train": met_tr["Top1"],
            "Top10_train": met_tr["Top10"],
            "Top50_train": met_tr["Top50"],
            "Top100_train": met_tr["Top100"],

            "NLL_oos": met_te["NLL"],
            "MRR_oos": met_te["MRR"],
            "McFadden_R2_oos": met_te["McFadden_R2"],
            "rank_median_oos": met_te["rank_median"],
            "Top1_oos": met_te["Top1"],
            "Top10_oos": met_te["Top10"],
            "Top50_oos": met_te["Top50"],
            "Top100_oos": met_te["Top100"],
        }
        row.update(full_met)
        results.append(row)

        print("[SAMPLED TRAIN]",
              f"NLL={row['NLL_train']:.4f} MRR={row['MRR_train']:.4f} R2={row['McFadden_R2_train']:.4f} "
              f"Top1={row['Top1_train']:.3f} Top10={row['Top10_train']:.3f} rMed={row['rank_median_train']:.1f}")
        print("[SAMPLED OOS ]",
              f"NLL={row['NLL_oos']:.4f} MRR={row['MRR_oos']:.4f} R2={row['McFadden_R2_oos']:.4f} "
              f"Top1={row['Top1_oos']:.3f} Top10={row['Top10_oos']:.3f} rMed={row['rank_median_oos']:.1f}")

        if full_met:
            print("[FULL OOS   ]",
                  f"NLL_full={row['NLL_full']:.4f} MRR_full={row['MRR_full']:.4f} R2_full={row['McFadden_R2_full']:.4f} "
                  f"Top1_full={row['Top1_full']:.3f} Top10_full={row['Top10_full']:.3f} rMed_full={row['rank_median_full']:.1f}")
            print("[SHARES FULL ]",
                  f"spearman={row['share_spearman_full']:.4f} JS={row['share_JS_full']:.4f} "
                  f"N_full={row['N_full_eval']} J_full={row['J_full']}")

# -----------------------------
# Save outputs
# -----------------------------
res_df = pd.DataFrame(results).sort_values(["segment", "impedance"]).reset_index(drop=True)

long_path = os.path.join(OUT_DIR, "impedance_compare_long.csv")
res_df.to_csv(long_path, index=False)
print(f"\n[OK] wrote {long_path}")

# Wide pivot
value_cols = [c for c in res_df.columns if c not in ("impedance", "segment")]
wide = res_df.pivot(index="segment", columns="impedance", values=value_cols)

wide.columns = [f"{v}__{s}" for (v, s) in wide.columns]
wide = wide.reset_index()

wide_path = os.path.join(OUT_DIR, "impedance_compare_wide.csv")
wide.to_csv(wide_path, index=False)
print(f"[OK] wrote {wide_path}")

# Deltas: BEE - EMU
delta = pd.DataFrame({"segment": wide["segment"]})
for col in value_cols:
    c_b = f"{col}__BEE"
    c_e = f"{col}__EMU"
    if c_b in wide.columns and c_e in wide.columns:
        delta[col + "__delta_BEE_minus_EMU"] = wide[c_b] - wide[c_e]

delta_path = os.path.join(OUT_DIR, "impedance_compare_deltas_BEE_minus_EMU.csv")
delta.to_csv(delta_path, index=False)
print(f"[OK] wrote {delta_path}")

print("\n[DONE] Outputs in:", OUT_DIR)

## Travel Distance

In [ ]:
# ============================================================
# ROBUSTNESS: Impedance proxy comparison (EMU vs Distance skim)
# byPerson | with LIE | BASE_LOG1P_COLS | global Z | uniform sampling
#
# Fixed:
#   - K = 1000
#   - 50 epochs
#   - sampled TRAIN + sampled OOS metrics
#   - FULL-choice-set metrics on OOS subset (all zones as alternatives)
#
# Two models differ ONLY by impedance matrix:
#   A) EMU       (utilities_2023_ready.omx      : "utility_emu")
#   B) Distance  (distance_avg_2023_ready.omx   : "avg_distances_ready")
#
# Outputs:
#   ModelRuns/ImpedanceRobustness_Distance/impedance_compare_long.csv
#   ModelRuns/ImpedanceRobustness_Distance/impedance_compare_wide.csv
#   ModelRuns/ImpedanceRobustness_Distance/impedance_compare_deltas_DIST_minus_EMU.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path


# -----------------------------
# CONFIG
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TRIPS_DIR = "Trips/byPerson"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

# impedance matrices
UTIL_OMX       = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME   = "utility_emu"

DIST_OMX      = READY_DIR / "distance_avg_2023_ready_log1p.omx"
DIST_MAT_NAME = "avg_distances_ready_log1p"

MAP_NAME  = "NO"

OUT_DIR = "ModelRuns/ImpedanceRobustness_Distance"
os.makedirs(OUT_DIR, exist_ok=True)

K = 1000
EPOCHS = 50
BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

FULL_EVAL_MAX = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256

TOPKS = (1, 10, 50, 100)

# -----------------------------
# BASELINE feature list (13)
# -----------------------------
BASE_LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]


# -----------------------------
# Small helpers
# -----------------------------
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)


# -----------------------------
# IO helpers
# -----------------------------
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


# -----------------------------
# Preprocess helpers
# -----------------------------
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)

    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts

    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 imp_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    imp_set = imp_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(imp_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2
    )


# -----------------------------
# Model
# -----------------------------
def train_mnl(imp_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, J = imp_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            imp_b = imp_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * imp_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(imp_set, X_set, w, alpha, beta, topKs=(1,10,50,100)):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * imp_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = int(V.shape[1])
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

        out = {
            "NLL": nll,
            "MRR": mrr,
            "McFadden_R2": float(r2),
            "rank_median": rank_median,
            "J": J,
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))

    return out


# -----------------------------
# FULL-choice-set evaluation
# -----------------------------
def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    imp_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(1,10,50,100),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)
    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = imp_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}
    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)

    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    ll_model = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    top_num = {k: 0.0 for k in topKs}

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(imp_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * imp_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * imp_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        nll_num += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * imp_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    out = {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    return out


# ============================================================
# MAIN
# ============================================================

print("[LOAD] TZ features...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)

need_cols = ["npvm_id"] + BASE_LOG1P_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

tz_feat0 = tz[["npvm_id"] + BASE_LOG1P_COLS + ["geometry"]].copy()
tz_feat0["npvm_id"] = pd.to_numeric(tz_feat0["npvm_id"], errors="coerce")
tz_feat0 = tz_feat0.dropna(subset=["npvm_id"]).copy()
tz_feat0["npvm_id"] = tz_feat0["npvm_id"].astype(int)

print("[PREP] Global z-standardize destination vars on full TZ universe...")
tz_feat0, z_cols, mu_all, sd_all = standardize_all_zones(tz_feat0, BASE_LOG1P_COLS, suffix="_z")

print("[LOAD] EMU impedance...")
emu_mat, zone_to_idx_emu, idx_to_zone_emu = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)

print("[LOAD] Distance skim impedance...")
dist_mat, zone_to_idx_dist, idx_to_zone_dist = load_matrix_and_mapping(DIST_OMX, DIST_MAT_NAME, MAP_NAME)

if len(idx_to_zone_emu) != len(idx_to_zone_dist) or not np.all(idx_to_zone_emu == idx_to_zone_dist):
    raise ValueError("EMU and Distance OMX mappings differ.")
if emu_mat.shape != dist_mat.shape:
    raise ValueError(f"Matrix shapes differ: EMU {emu_mat.shape} vs Distance {dist_mat.shape}")

zone_to_idx = zone_to_idx_emu
idx_to_zone = idx_to_zone_emu

RUNS = [
    {"name": "EMU",  "mat": emu_mat},
    {"name": "DIST", "mat": dist_mat},
]

results = []

for run in RUNS:
    imp_name = run["name"]
    imp_mat  = run["mat"]

    print("\n" + "#" * 110)
    print(f"# IMPEDANCE = {imp_name} | K={K} | epochs={EPOCHS} | byPerson | BASE vars")
    print("#" * 110)

    for seg in SEGMENTS:
        print("\n" + "=" * 95)
        print(f"Segment {seg} | impedance={imp_name}")
        print("=" * 95)

        train_df = load_segment_csv(TRIPS_DIR, seg, "train")
        oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

        seed_tr = BASE_SEED + 1000 * SEG_SEED[seg] + 10
        seed_te = BASE_SEED + 1000 * SEG_SEED[seg] + 20

        imp_tr, X_tr, y_tr, w_tr, train_df2 = build_design(
            train_df, tz_feat0, z_cols, imp_mat, zone_to_idx, k=K, seed=seed_tr
        )
        imp_te, X_te, y_te, w_te, oos_df2 = build_design(
            oos_df, tz_feat0, z_cols, imp_mat, zone_to_idx, k=K, seed=seed_te
        )

        print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
        print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

        alpha_hat, beta_hat = train_mnl(
            imp_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
        )

        met_tr = eval_sampled_metrics(imp_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS)
        met_te = eval_sampled_metrics(imp_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS)

        max_n = FULL_EVAL_MAX.get(seg, 8000)
        seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg]
        full_met = full_choice_eval_oos_subset(
            oos_df=oos_df2,
            tz_feat_seg=tz_feat0,
            z_cols=z_cols,
            imp_mat=imp_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=max_n,
            seed=seed_full,
            chunk=FULL_EVAL_CHUNK,
            topKs=TOPKS,
        )

        row = {
            "impedance": imp_name,
            "segment": seg,
            "K": K,
            "alpha_imp": float(alpha_hat),

            "NLL_train": met_tr["NLL"],
            "MRR_train": met_tr["MRR"],
            "McFadden_R2_train": met_tr["McFadden_R2"],
            "rank_median_train": met_tr["rank_median"],
            "Top1_train": met_tr["Top1"],
            "Top10_train": met_tr["Top10"],
            "Top50_train": met_tr["Top50"],
            "Top100_train": met_tr["Top100"],

            "NLL_oos": met_te["NLL"],
            "MRR_oos": met_te["MRR"],
            "McFadden_R2_oos": met_te["McFadden_R2"],
            "rank_median_oos": met_te["rank_median"],
            "Top1_oos": met_te["Top1"],
            "Top10_oos": met_te["Top10"],
            "Top50_oos": met_te["Top50"],
            "Top100_oos": met_te["Top100"],
        }
        row.update(full_met)
        results.append(row)

        print("[SAMPLED TRAIN]",
              f"NLL={row['NLL_train']:.4f} MRR={row['MRR_train']:.4f} R2={row['McFadden_R2_train']:.4f} "
              f"Top1={row['Top1_train']:.3f} Top10={row['Top10_train']:.3f} rMed={row['rank_median_train']:.1f}")
        print("[SAMPLED OOS ]",
              f"NLL={row['NLL_oos']:.4f} MRR={row['MRR_oos']:.4f} R2={row['McFadden_R2_oos']:.4f} "
              f"Top1={row['Top1_oos']:.3f} Top10={row['Top10_oos']:.3f} rMed={row['rank_median_oos']:.1f}")

        if full_met:
            print("[FULL OOS   ]",
                  f"NLL_full={row['NLL_full']:.4f} MRR_full={row['MRR_full']:.4f} R2_full={row['McFadden_R2_full']:.4f} "
                  f"Top1_full={row['Top1_full']:.3f} Top10_full={row['Top10_full']:.3f} rMed_full={row['rank_median_full']:.1f}")
            print("[SHARES FULL ]",
                  f"spearman={row['share_spearman_full']:.4f} JS={row['share_JS_full']:.4f} "
                  f"N_full={row['N_full_eval']} J_full={row['J_full']}")

# -----------------------------
# Save outputs
# -----------------------------
res_df = pd.DataFrame(results).sort_values(["segment", "impedance"]).reset_index(drop=True)

long_path = os.path.join(OUT_DIR, "impedance_compare_long.csv")
res_df.to_csv(long_path, index=False)
print(f"\n[OK] wrote {long_path}")

value_cols = [c for c in res_df.columns if c not in ("impedance", "segment")]
wide = res_df.pivot(index="segment", columns="impedance", values=value_cols)
wide.columns = [f"{v}__{s}" for (v, s) in wide.columns]
wide = wide.reset_index()

wide_path = os.path.join(OUT_DIR, "impedance_compare_wide.csv")
wide.to_csv(wide_path, index=False)
print(f"[OK] wrote {wide_path}")

delta = pd.DataFrame({"segment": wide["segment"]})
for col in value_cols:
    c_dist = f"{col}__DIST"
    c_emu  = f"{col}__EMU"
    if c_dist in wide.columns and c_emu in wide.columns:
        delta[col + "__delta_DIST_minus_EMU"] = wide[c_dist] - wide[c_emu]

delta_path = os.path.join(OUT_DIR, "impedance_compare_deltas_DIST_minus_EMU.csv")
delta.to_csv(delta_path, index=False)
print(f"[OK] wrote {delta_path}")

print("\n[DONE] Outputs in:", OUT_DIR)

# R3 Alternative Sampling

## Different K

In [ ]:
# ============================================================
# K-ROBUSTNESS EVALUATION (byPerson split + uniform sampling)
# ------------------------------------------------------------
# WHAT THIS VERSION DOES
#   - Trains SAME MNL model for K in:
#       [100, 300, 500, 800, 1000, 2000, 3000]
#   - Uses GLOBAL z-score computed once on ALL TZ zones
#   - Keeps everything else fixed
#
# ADDITIONS IN THIS VERSION
#   1) K=1000 is treated as the baseline K
#   2) For every K != 1000, computes metric deltas vs K=1000
#      separately by segment
#
# OUTPUTS
#   ModelRuns/K_evaluation/k_results_long.csv
#   ModelRuns/K_evaluation/k_results_wide.csv
#   ModelRuns/K_evaluation/k_deltas_vs_k1000.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path


# -----------------------------
# CONFIG
# -----------------------------
TRIPS_DIR = "Trips/byPerson"
SEGMENTS  = ["YS", "OS", "YL", "OL"]

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT   = "utility_emu"
MAP_NAME  = "NO"

OUT_DIR = "ModelRuns/K_evaluation"
os.makedirs(OUT_DIR, exist_ok=True)

K_LIST = [100, 300, 500, 800, 1000, 2000, 3000]
BASELINE_K = 1000
SEED = 123

# Torch
DEVICE = "cpu"
DTYPE  = torch.float32

# Optimization
EPOCHS = 30
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# FULL-choice-set evaluation subset sizes (OOS)
FULL_EVAL_MAX = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256

LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]


# -----------------------------
# Helpers
# -----------------------------
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)


# -----------------------------
# IO helpers
# -----------------------------
def load_segment_csv(seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(TRIPS_DIR, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    need = [ORIG_COL, DEST_COL, WP_COL]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df

def load_emu_matrix_and_mapping(omx_path: Path, emu_mat: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if emu_mat not in f.list_matrices():
        raise ValueError(f"Matrix '{emu_mat}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[emu_mat]
    print(f"[OMX] Loading {emu_mat} into RAM... shape={M.shape} (float32)")
    emu = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return emu, zone_to_idx, idx_to_zone


# -----------------------------
# Preprocess helpers
# -----------------------------
def standardize_global_allzones(tz_feat: pd.DataFrame, cols: list[str], suffix: str = "_z"):
    X = tz_feat[cols].replace([np.inf, -np.inf], np.nan)

    mu = X.mean(axis=0, skipna=True)
    sd = X.std(axis=0, skipna=True).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    if tz_feat[z_cols].isna().any().any():
        n_na = int(tz_feat[z_cols].isna().sum().sum())
        print(f"[WARN] z-score produced {n_na} NaNs.")

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)

    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts

    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
        dest_set[good],
    )


# -----------------------------
# Model
# -----------------------------
def train_mnl(emu_set, X_set, y, w, epochs=30, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, J = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]

            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    alpha_hat, beta_hat = best_state
    return alpha_hat, beta_hat

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        top1   = float((w * (rank <= 1).float()).sum().cpu() / (sum_w + 1e-9))
        top10  = float((w * (rank <= 10).float()).sum().cpu() / (sum_w + 1e-9))
        top50  = float((w * (rank <= 50).float()).sum().cpu() / (sum_w + 1e-9))
        top100 = float((w * (rank <= 100).float()).sum().cpu() / (sum_w + 1e-9))

        ll_model = float((w * chosen_logp).sum().cpu())
        J = V.shape[1]
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    return {
        "NLL": nll,
        "MRR": mrr,
        "McFadden_R2": float(r2),
        "rank_median": rank_median,
        "Top1": top1,
        "Top10": top10,
        "Top50": top50,
        "Top100": top100,
        "LL_model": ll_model,
        "LL0_uniform": ll0,
        "J": int(emu_set.shape[1]),
    }

def sampled_shares_metrics(df_obs, dest_set, emu_set, X_set, w, alpha, beta):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        P = torch.softmax(V, dim=1).detach().cpu().numpy()

    w_np = w.detach().cpu().numpy().astype(np.float64)
    chosen = df_obs[DEST_COL].to_numpy(dtype=int)

    obs_mass = pd.Series(w_np).groupby(chosen).sum()

    pred_mass = {}
    for n in range(dest_set.shape[0]):
        wn = float(w_np[n])
        for j in range(dest_set.shape[1]):
            zid = int(dest_set[n, j])
            pred_mass[zid] = pred_mass.get(zid, 0.0) + wn * float(P[n, j])

    pred_mass = pd.Series(pred_mass)

    all_ids = obs_mass.index.union(pred_mass.index)
    obs = obs_mass.reindex(all_ids, fill_value=0.0).to_numpy(dtype=np.float64)
    pred = pred_mass.reindex(all_ids, fill_value=0.0).to_numpy(dtype=np.float64)

    obs = obs / (obs.sum() + 1e-12)
    pred = pred / (pred.sum() + 1e-12)

    spear = float(pd.Series(obs).corr(pd.Series(pred), method="spearman")) if len(obs) > 2 else np.nan
    js = js_div(obs, pred)
    return spear, js


# -----------------------------
# FULL-choice-set evaluation
# -----------------------------
def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}
    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)

    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    top1_num = top10_num = top50_num = top100_num = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    ll_model = 0.0

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

        nll_num += wn * (-logp_chosen)
        ll_model += wn * logp_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))

        top1_num   += wn * (1.0 if rank <= 1 else 0.0)
        top10_num  += wn * (1.0 if rank <= 10 else 0.0)
        top50_num  += wn * (1.0 if rank <= 50 else 0.0)
        top100_num += wn * (1.0 if rank <= 100 else 0.0)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    top1_full   = float(top1_num / (sum_w + 1e-9))
    top10_full  = float(top10_num / (sum_w + 1e-9))
    top50_full  = float(top50_num / (sum_w + 1e-9))
    top100_full = float(top100_num / (sum_w + 1e-9))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    return {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Top1_full": top1_full,
        "Top10_full": top10_full,
        "Top50_full": top50_full,
        "Top100_full": top100_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }


# ============================================================
# MAIN
# ============================================================
print("[LOAD] TZ features...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
need_cols = ["npvm_id"] + LOG1P_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

tz_feat0 = tz[["npvm_id"] + LOG1P_COLS + ["geometry"]].copy()
tz_feat0["npvm_id"] = pd.to_numeric(tz_feat0["npvm_id"], errors="coerce")
tz_feat0 = tz_feat0.dropna(subset=["npvm_id"]).copy()
tz_feat0["npvm_id"] = tz_feat0["npvm_id"].astype(int)

tz_feat0, z_cols, mu_all, sd_all = standardize_global_allzones(tz_feat0, LOG1P_COLS, suffix="_z")
print("[Z] Global scaling computed once on ALL TZ zones.")

print("[LOAD] EMU matrix...")
emu_mat, zone_to_idx, idx_to_zone = load_emu_matrix_and_mapping(UTIL_OMX, EMU_MAT, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

results = []

for K in K_LIST:
    print("\n" + "#" * 110)
    print(f"# K = {K}")
    print("#" * 110)

    for seg in SEGMENTS:
        print("\n" + "=" * 95)
        print(f"Segment {seg} | K={K}")
        print("=" * 95)

        train_df = load_segment_csv(seg, "train")
        oos_df   = load_segment_csv(seg, "oos")
        tz_feat_seg = tz_feat0

        emu_tr, X_tr, y_tr, w_tr, train_df2, destset_tr = build_design(
            train_df, tz_feat_seg, z_cols, emu_mat, zone_to_idx, k=K, seed=SEED + 10 + K
        )
        emu_te, X_te, y_te, w_te, oos_df2, destset_te = build_design(
            oos_df, tz_feat_seg, z_cols, emu_mat, zone_to_idx, k=K, seed=SEED + 20 + K
        )

        print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
        print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

        alpha_hat, beta_hat = train_mnl(
            emu_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
        )

        met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat)
        met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat)

        share_spear_tr, share_js_tr = sampled_shares_metrics(
            df_obs=train_df2, dest_set=destset_tr,
            emu_set=emu_tr, X_set=X_tr, w=w_tr,
            alpha=alpha_hat, beta=beta_hat
        )
        share_spear_te, share_js_te = sampled_shares_metrics(
            df_obs=oos_df2, dest_set=destset_te,
            emu_set=emu_te, X_set=X_te, w=w_te,
            alpha=alpha_hat, beta=beta_hat
        )

        max_n = FULL_EVAL_MAX.get(seg, 8000)
        seed_full = FULL_EVAL_SEED + 1000 * SEGMENTS.index(seg) + K

        full_met = full_choice_eval_oos_subset(
            oos_df=oos_df2,
            tz_feat_seg=tz_feat_seg,
            z_cols=z_cols,
            emu_mat=emu_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=max_n,
            seed=seed_full,
            chunk=FULL_EVAL_CHUNK,
        )

        row = {
            "K": K,
            "segment": seg,
            "alpha_emu": float(alpha_hat),

            "NLL_train": met_tr["NLL"],
            "MRR_train": met_tr["MRR"],
            "McFadden_R2_train": met_tr["McFadden_R2"],
            "rank_median_train": met_tr["rank_median"],
            "Top1_train": met_tr["Top1"],
            "Top10_train": met_tr["Top10"],
            "Top50_train": met_tr["Top50"],
            "Top100_train": met_tr["Top100"],
            "share_spearman_train": float(share_spear_tr),
            "share_JS_train": float(share_js_tr),

            "NLL_oos": met_te["NLL"],
            "MRR_oos": met_te["MRR"],
            "McFadden_R2_oos": met_te["McFadden_R2"],
            "rank_median_oos": met_te["rank_median"],
            "Top1_oos": met_te["Top1"],
            "Top10_oos": met_te["Top10"],
            "Top50_oos": met_te["Top50"],
            "Top100_oos": met_te["Top100"],
            "share_spearman_oos": float(share_spear_te),
            "share_JS_oos": float(share_js_te),
        }
        row.update(full_met)
        results.append(row)

        print("[SAMPLED TRAIN] "
              f"NLL={row['NLL_train']:.4f} | MRR={row['MRR_train']:.4f} | "
              f"Top1={row['Top1_train']:.3f} Top10={row['Top10_train']:.3f} "
              f"Top50={row['Top50_train']:.3f} Top100={row['Top100_train']:.3f} | "
              f"shareSp={row['share_spearman_train']:.3f} JS={row['share_JS_train']:.3f}")

        print("[SAMPLED OOS  ] "
              f"NLL={row['NLL_oos']:.4f} | MRR={row['MRR_oos']:.4f} | "
              f"Top1={row['Top1_oos']:.3f} Top10={row['Top10_oos']:.3f} "
              f"Top50={row['Top50_oos']:.3f} Top100={row['Top100_oos']:.3f} | "
              f"shareSp={row['share_spearman_oos']:.3f} JS={row['share_JS_oos']:.3f}")

        if full_met:
            print("[FULL OOS SUB] "
                  f"NLL_full={row['NLL_full']:.4f} | MRR_full={row['MRR_full']:.4f} | "
                  f"Top1={row['Top1_full']:.3f} Top10={row['Top10_full']:.3f} "
                  f"Top50={row['Top50_full']:.3f} Top100={row['Top100_full']:.3f} | "
                  f"shareSp={row['share_spearman_full']:.3f} JS={row['share_JS_full']:.3f} | "
                  f"N_full={row.get('N_full_eval', np.nan)} J={row.get('J_full', np.nan)}")

# ------------------------------------------------------------
# SAVE MAIN LONG TABLE
# ------------------------------------------------------------
res_df = pd.DataFrame(results).sort_values(["K", "segment"]).reset_index(drop=True)

long_path = os.path.join(OUT_DIR, "k_results_long.csv")
res_df.to_csv(long_path, index=False)
print(f"\n[OK] wrote {long_path}")

# ------------------------------------------------------------
# SAVE WIDE TABLE
# ------------------------------------------------------------
value_cols = [c for c in res_df.columns if c not in ("K", "segment")]
wide = res_df.pivot(index="segment", columns="K", values=value_cols)
wide.columns = [f"{v}__K{k}" for (v, k) in wide.columns]
wide = wide.reset_index()

wide_path = os.path.join(OUT_DIR, "k_results_wide.csv")
wide.to_csv(wide_path, index=False)
print(f"[OK] wrote {wide_path}")

# ------------------------------------------------------------
# DELTAS VS K=1000
# ------------------------------------------------------------
metrics_for_delta = [
    "alpha_emu",
    "NLL_train", "MRR_train", "McFadden_R2_train", "rank_median_train",
    "Top1_train", "Top10_train", "Top50_train", "Top100_train",
    "share_spearman_train", "share_JS_train",
    "NLL_oos", "MRR_oos", "McFadden_R2_oos", "rank_median_oos",
    "Top1_oos", "Top10_oos", "Top50_oos", "Top100_oos",
    "share_spearman_oos", "share_JS_oos",
    "NLL_full", "MRR_full", "McFadden_R2_full", "rank_median_full",
    "Top1_full", "Top10_full", "Top50_full", "Top100_full",
    "share_spearman_full", "share_JS_full",
]

base_ref = res_df[res_df["K"] == BASELINE_K].set_index("segment")
delta_rows = []

for _, r in res_df.iterrows():
    K = int(r["K"])
    seg = r["segment"]

    if K == BASELINE_K:
        continue
    if seg not in base_ref.index:
        continue

    b = base_ref.loc[seg]
    out = {
        "segment": seg,
        "K": K,
        "baseline_K": BASELINE_K,
    }

    for m in metrics_for_delta:
        if m in r.index and m in b.index and pd.notnull(r[m]) and pd.notnull(b[m]):
            out[f"{m}__delta_vs_K{BASELINE_K}"] = float(r[m] - b[m])

    delta_rows.append(out)

df_delta = pd.DataFrame(delta_rows).sort_values(["K", "segment"]).reset_index(drop=True)
delta_path = os.path.join(OUT_DIR, "k_deltas_vs_k1000.csv")
df_delta.to_csv(delta_path, index=False)
print(f"[OK] wrote {delta_path}")

# ------------------------------------------------------------
# SCREEN SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("MAIN RESULTS (selected columns)")
print("=" * 110)

cols_show = [
    "K", "segment",
    "NLL_train", "NLL_oos", "NLL_full",
    "MRR_train", "MRR_oos", "MRR_full",
    "McFadden_R2_train", "McFadden_R2_oos", "McFadden_R2_full",
    "share_spearman_train", "share_spearman_oos", "share_spearman_full",
    "share_JS_train", "share_JS_oos", "share_JS_full",
]
cols_show = [c for c in cols_show if c in res_df.columns]
print(res_df[cols_show].to_string(index=False))

print("\n" + "=" * 110)
print(f"DELTAS VS K={BASELINE_K}")
print("=" * 110)
if len(df_delta) > 0:
    print(df_delta.head(20).to_string(index=False))
else:
    print("No delta rows produced.")

print("\n[DONE] Outputs in:", OUT_DIR)

## Sampling EMU guided

In [ ]:
# ============================================================
# ROBUSTNESS: sampled choice-set construction
# Compare:
#   1) uniform sampling (baseline loaded from baseline csv)
#   2) EMU-weighted sampling with approximate McFadden correction
#
# COHERENT WITH BASELINE:
#   - byPerson split
#   - with LIE
#   - same 13 destination-side variables
#   - log1p variables already in GPKG
#   - GLOBAL z-standardisation on full TZ universe
#   - K = 1000
#   - 50 epochs
#   - weighted MNL
#   - full-choice-set OOS evaluation
#
# IMPORTANT:
#   For weighted sampling, sampled-set likelihood uses:
#       V'_ij = V_ij - log(pi_ij)
#   where pi_ij is an approximate inclusion probability of alternative j
#   in the sampled set. Chosen alternative is always included => pi=1.
#
# OUTPUTS:
#   ModelRuns/SamplingStrategyRobustness/
#   - sampling_results_emu_weighted.csv
#   - sampling_compare_uniform_vs_weighted_long.csv
#   - sampling_compare_uniform_vs_weighted_wide.csv
#   - sampling_compare_deltas_weighted_minus_uniform.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path


# -----------------------------
# CONFIG
# -----------------------------
TRIPS_DIR = "Trips/byPerson"
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT   = "utility_emu"
MAP_NAME  = "NO"

BASELINE_CSV_CANDIDATES = [
    "ModelRuns/K_evaluation/k_results_long.csv",
    "ModelRuns/K_evaluation/k_results.csv",
    "/mnt/data/k_results_long.csv",
    "/mnt/data/k_results.csv",
]

OUT_DIR = "ModelRuns/SamplingStrategyRobustness"
os.makedirs(OUT_DIR, exist_ok=True)

K = 1000
BASELINE_K = 1000

BASE_SEED = 123
FULL_EVAL_SEED = 777

DEVICE = "cpu"
DTYPE  = torch.float32

EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

FULL_EVAL_MAX = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
FULL_EVAL_CHUNK = 256

TOPKS = (1, 10, 50, 100)

BASE_LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

# -----------------------------
# EMU-weighted sampling hyperparams
# -----------------------------
TOP_M = 6000    # must be >= K
TEMP  = 1.0


# ============================================================
# Helpers
# ============================================================
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)


def load_segment_csv(seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(TRIPS_DIR, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df


def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


def find_baseline_csv() -> str:
    for p in BASELINE_CSV_CANDIDATES:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not find baseline csv in: {BASELINE_CSV_CANDIDATES}")


def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd


# ============================================================
# Sampling helpers
# ============================================================
def sample_uniform_nonchosen(all_dest_zone_ids: np.ndarray,
                             chosen_zone_id: int,
                             k: int,
                             rng: np.random.Generator):
    pool = all_dest_zone_ids[all_dest_zone_ids != chosen_zone_id]
    if k > len(pool):
        raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")

    alts = rng.choice(pool, size=k, replace=False)

    # Uniform without replacement over the non-chosen universe
    pi_const = k / float(len(pool))
    log_pi = np.full(k, np.log(pi_const), dtype=np.float32)

    return alts.astype(np.int64), log_pi


def sample_emu_weighted_nonchosen(
    origin_zone_id: int,
    chosen_zone_id: int,
    k: int,
    rng: np.random.Generator,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    emu_mat: np.ndarray,
    all_dest_omx_idx: np.ndarray,
    top_m: int,
    temp: float,
):
    """
    Returns:
      alts_zone_ids (k,)
      log_pi_alts   (k,) approximate log inclusion probabilities
    """
    oi = zone_to_idx.get(int(origin_zone_id), -1)
    cj = zone_to_idx.get(int(chosen_zone_id), -1)
    if oi < 0:
        return None, None

    cand = all_dest_omx_idx
    if cj >= 0:
        cand = cand[cand != cj]

    if len(cand) < k:
        return None, None

    row = emu_mat[oi, cand].astype(np.float32)

    M = min(int(top_m), len(cand))
    M = max(M, k)

    if M < len(cand):
        top_idx_local = np.argpartition(row, -M)[-M:]
        cand_top = cand[top_idx_local]
        row_top = row[top_idx_local]
    else:
        cand_top = cand
        row_top = row

    m = float(np.max(row_top))
    logits = (row_top.astype(np.float64) - m) / max(float(temp), 1e-9)
    weights = np.exp(logits)
    sw = float(np.sum(weights))

    if (not np.isfinite(sw)) or sw <= 0:
        pick = rng.choice(cand_top, size=k, replace=False)
        alts = idx_to_zone[pick].astype(np.int64)
        pi_const = k / float(len(cand_top))
        log_pi = np.full(k, np.log(pi_const), dtype=np.float32)
        return alts, log_pi

    p = weights / (sw + 1e-300)

    pick = rng.choice(cand_top, size=k, replace=False, p=p)
    alts = idx_to_zone[pick].astype(np.int64)

    # Approximate inclusion probability:
    # pi_j ≈ 1 - (1 - p_j)^k
    pos = {int(cand_top[i]): i for i in range(len(cand_top))}
    pi_vals = []
    for j in pick:
        pj = float(p[pos[int(j)]])
        pi_j = 1.0 - (1.0 - pj) ** k
        pi_j = min(max(pi_j, 1e-300), 1.0)
        pi_vals.append(pi_j)

    log_pi = np.log(np.asarray(pi_vals, dtype=np.float64)).astype(np.float32)
    return alts, log_pi


def build_design(
    df: pd.DataFrame,
    tz_feat: pd.DataFrame,
    feat_cols_z: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    k: int,
    seed: int,
    strategy: str,
    top_m: int = 6000,
    temp: float = 1.0,
):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests_zone = tz_feat["npvm_id"].to_numpy(dtype=int)
    all_dests_omx_idx = np.array([zone_to_idx.get(int(z), -1) for z in all_dests_zone], dtype=int)
    good_dest = all_dests_omx_idx >= 0
    all_dests_zone = all_dests_zone[good_dest]
    all_dests_omx_idx = all_dests_omx_idx[good_dest]

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)

    dest_set = np.empty((N, k + 1), dtype=np.int64)
    log_pi   = np.zeros((N, k + 1), dtype=np.float32)  # chosen alternative has pi=1 => log_pi=0

    for i in range(N):
        c = int(chosen[i])
        o = int(orig[i])

        if strategy == "uniform":
            alts, logpi_alts = sample_uniform_nonchosen(all_dests_zone, c, k, rng)

        elif strategy == "emu_weighted":
            alts, logpi_alts = sample_emu_weighted_nonchosen(
                origin_zone_id=o,
                chosen_zone_id=c,
                k=k,
                rng=rng,
                zone_to_idx=zone_to_idx,
                idx_to_zone=idx_to_zone,
                emu_mat=emu_mat,
                all_dest_omx_idx=all_dests_omx_idx,
                top_m=top_m,
                temp=temp,
            )
            if alts is None:
                alts = np.full(k, -1, dtype=np.int64)
                logpi_alts = np.zeros(k, dtype=np.float32)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        dest_set[i, 0] = c
        dest_set[i, 1:] = alts
        log_pi[i, 1:] = logpi_alts

    X = np.empty((N, k + 1, Pdim), dtype=np.float32)
    flat = dest_set.ravel()

    for p, col in enumerate(feat_cols_z):
        vals = feat_map.reindex(flat)[col].to_numpy(dtype=np.float32)
        X[:, :, p] = vals.reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (
        (orig_idx >= 0)
        & (dest_idx.min(axis=1) >= 0)
        & np.isfinite(w) & (w > 0)
        & np.isfinite(X).all(axis=(1, 2))
    )

    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping/features/bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    logpi2 = log_pi[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(logpi2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
    )


# ============================================================
# Model
# ============================================================
def train_mnl(emu_set, X_set, log_pi, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            lp_b  = log_pi[b]
            y_b   = y[b]
            w_b   = w[b]

            # McFadden correction: V' = V - log(pi)
            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta) - lp_b
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]

            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state


def eval_sampled_metrics(emu_set, X_set, log_pi, w, alpha, beta, top_ks=(1, 10, 50, 100)):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t) - log_pi
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())
        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = int(V.shape[1])
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

        out = {
            "NLL": nll,
            "MRR": mrr,
            "McFadden_R2": float(r2),
            "rank_median": rank_median,
            "J": J,
        }
        for kk in top_ks:
            out[f"Top{kk}"] = float((w * (rank <= kk).float()).sum().cpu() / (sum_w + 1e-9))
        return out


# ============================================================
# FULL-choice-set evaluation
# ============================================================
def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}
    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)

    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    top1_num = top10_num = top50_num = top100_num = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    ll_model = 0.0

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

        nll_num += wn * (-logp_chosen)
        ll_model += wn * logp_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))

        top1_num   += wn * (1.0 if rank <= 1 else 0.0)
        top10_num  += wn * (1.0 if rank <= 10 else 0.0)
        top50_num  += wn * (1.0 if rank <= 50 else 0.0)
        top100_num += wn * (1.0 if rank <= 100 else 0.0)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    return {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Top1_full": float(top1_num / (sum_w + 1e-9)),
        "Top10_full": float(top10_num / (sum_w + 1e-9)),
        "Top50_full": float(top50_num / (sum_w + 1e-9)),
        "Top100_full": float(top100_num / (sum_w + 1e-9)),
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }


# ============================================================
# MAIN
# ============================================================
baseline_path = find_baseline_csv()
base_df = pd.read_csv(baseline_path)

if "K" not in base_df.columns or "segment" not in base_df.columns:
    raise ValueError(f"Baseline file missing required columns K/segment: {baseline_path}")

base_df = base_df[(base_df["K"] == BASELINE_K) & (base_df["segment"].isin(SEGMENTS))].copy()
print(f"[BASELINE] Loaded uniform baseline from: {baseline_path} | rows={len(base_df):,}")

print("[LOAD] TZ features...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)

need_cols = ["npvm_id"] + BASE_LOG1P_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

tz_feat0 = tz[["npvm_id"] + BASE_LOG1P_COLS].copy()
tz_feat0["npvm_id"] = pd.to_numeric(tz_feat0["npvm_id"], errors="coerce")
tz_feat0 = tz_feat0.dropna(subset=["npvm_id"]).copy()
tz_feat0["npvm_id"] = tz_feat0["npvm_id"].astype(int)

print("[PREP] Global z-standardize destination vars on full TZ universe...")
tz_feat0, z_cols, mu_all, sd_all = standardize_all_zones(tz_feat0, BASE_LOG1P_COLS, suffix="_z")

print("[LOAD] EMU matrix...")
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

# ------------------------------------------------------------
# Estimate EMU-weighted sampling strategy
# ------------------------------------------------------------
weighted_rows = []

for seg in SEGMENTS:
    print("\n" + "#" * 110)
    print(f"# SEGMENT = {seg} | strategy = emu_weighted | K={K} | TOP_M={TOP_M} | TEMP={TEMP}")
    print("#" * 110)

    train_df = load_segment_csv(seg, "train")
    oos_df   = load_segment_csv(seg, "oos")

    seed_tr = BASE_SEED + 1000 * SEG_SEED[seg] + 10
    seed_te = BASE_SEED + 1000 * SEG_SEED[seg] + 20
    seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg]

    emu_tr, X_tr, logpi_tr, y_tr, w_tr, train_df2 = build_design(
        train_df, tz_feat0, z_cols,
        emu_mat, zone_to_idx, idx_to_zone,
        k=K, seed=seed_tr,
        strategy="emu_weighted",
        top_m=TOP_M, temp=TEMP
    )
    emu_te, X_te, logpi_te, y_te, w_te, oos_df2 = build_design(
        oos_df, tz_feat0, z_cols,
        emu_mat, zone_to_idx, idx_to_zone,
        k=K, seed=seed_te,
        strategy="emu_weighted",
        top_m=TOP_M, temp=TEMP
    )

    print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
    print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

    alpha_hat, beta_hat = train_mnl(
        emu_tr, X_tr, logpi_tr, y_tr, w_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
    )

    met_tr = eval_sampled_metrics(emu_tr, X_tr, logpi_tr, w_tr, alpha_hat, beta_hat, top_ks=TOPKS)
    met_te = eval_sampled_metrics(emu_te, X_te, logpi_te, w_te, alpha_hat, beta_hat, top_ks=TOPKS)

    max_n = FULL_EVAL_MAX.get(seg, 8000)
    full_met = full_choice_eval_oos_subset(
        oos_df=oos_df2,
        tz_feat_seg=tz_feat0,
        z_cols=z_cols,
        emu_mat=emu_mat,
        zone_to_idx=zone_to_idx,
        idx_to_zone=idx_to_zone,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=max_n,
        seed=seed_full,
        chunk=FULL_EVAL_CHUNK,
    )

    row = {
        "K": K,
        "segment": seg,
        "strategy": "emu_weighted",
        "alpha_emu": float(alpha_hat),
        "TOP_M": TOP_M,
        "TEMP": TEMP,
    }

    for k_, v_ in met_tr.items():
        row[f"train_{k_}"] = v_
    for k_, v_ in met_te.items():
        row[f"oos_{k_}"] = v_
    row.update(full_met)
    weighted_rows.append(row)

    print("[SAMPLED TRAIN]",
          f"NLL={row['train_NLL']:.4f} "
          f"MRR={row['train_MRR']:.4f} "
          f"R2={row['train_McFadden_R2']:.4f} "
          f"Top10={row['train_Top10']:.3f}")

    print("[SAMPLED OOS ]",
          f"NLL={row['oos_NLL']:.4f} "
          f"MRR={row['oos_MRR']:.4f} "
          f"R2={row['oos_McFadden_R2']:.4f} "
          f"Top10={row['oos_Top10']:.3f}")

    print("[FULL OOS   ]",
          f"NLL_full={row['NLL_full']:.4f} "
          f"MRR_full={row['MRR_full']:.4f} "
          f"R2_full={row['McFadden_R2_full']:.4f} "
          f"shareSp={row['share_spearman_full']:.4f} "
          f"JS={row['share_JS_full']:.4f}")

weighted_df = pd.DataFrame(weighted_rows).sort_values(["segment"]).reset_index(drop=True)
weighted_csv = os.path.join(OUT_DIR, "sampling_results_emu_weighted.csv")
weighted_df.to_csv(weighted_csv, index=False)
print(f"\n[OK] wrote {weighted_csv}")

# ------------------------------------------------------------
# Standardize uniform baseline into same structure
# ------------------------------------------------------------
required_uniform_cols = [
    "K", "segment", "alpha_emu",
    "NLL_train", "MRR_train", "McFadden_R2_train", "rank_median_train",
    "Top1_train", "Top10_train", "Top50_train", "Top100_train",
    "NLL_oos", "MRR_oos", "McFadden_R2_oos", "rank_median_oos",
    "Top1_oos", "Top10_oos", "Top50_oos", "Top100_oos",
    "NLL_full", "MRR_full", "McFadden_R2_full", "rank_median_full",
    "Top1_full", "Top10_full", "Top50_full", "Top100_full",
    "share_spearman_full", "share_JS_full",
]

missing_uniform = [c for c in required_uniform_cols if c not in base_df.columns]
if missing_uniform:
    raise ValueError(f"Baseline file is missing columns needed for comparison: {missing_uniform}")

uniform_std = pd.DataFrame({
    "K": base_df["K"],
    "segment": base_df["segment"],
    "strategy": "uniform",
    "alpha_emu": base_df["alpha_emu"],

    "train_NLL": base_df["NLL_train"],
    "train_MRR": base_df["MRR_train"],
    "train_McFadden_R2": base_df["McFadden_R2_train"],
    "train_rank_median": base_df["rank_median_train"],
    "train_Top1": base_df["Top1_train"],
    "train_Top10": base_df["Top10_train"],
    "train_Top50": base_df["Top50_train"],
    "train_Top100": base_df["Top100_train"],

    "oos_NLL": base_df["NLL_oos"],
    "oos_MRR": base_df["MRR_oos"],
    "oos_McFadden_R2": base_df["McFadden_R2_oos"],
    "oos_rank_median": base_df["rank_median_oos"],
    "oos_Top1": base_df["Top1_oos"],
    "oos_Top10": base_df["Top10_oos"],
    "oos_Top50": base_df["Top50_oos"],
    "oos_Top100": base_df["Top100_oos"],

    "NLL_full": base_df["NLL_full"],
    "MRR_full": base_df["MRR_full"],
    "McFadden_R2_full": base_df["McFadden_R2_full"],
    "rank_median_full": base_df["rank_median_full"],
    "Top1_full": base_df["Top1_full"],
    "Top10_full": base_df["Top10_full"],
    "Top50_full": base_df["Top50_full"],
    "Top100_full": base_df["Top100_full"],
    "share_spearman_full": base_df["share_spearman_full"],
    "share_JS_full": base_df["share_JS_full"],
})

weighted_std = weighted_df[[
    "K", "segment", "strategy", "alpha_emu",
    "train_NLL", "train_MRR", "train_McFadden_R2", "train_rank_median",
    "train_Top1", "train_Top10", "train_Top50", "train_Top100",
    "oos_NLL", "oos_MRR", "oos_McFadden_R2", "oos_rank_median",
    "oos_Top1", "oos_Top10", "oos_Top50", "oos_Top100",
    "NLL_full", "MRR_full", "McFadden_R2_full", "rank_median_full",
    "Top1_full", "Top10_full", "Top50_full", "Top100_full",
    "share_spearman_full", "share_JS_full",
]].copy()

# ------------------------------------------------------------
# LONG comparison table
# ------------------------------------------------------------
cmp_long = pd.concat([uniform_std, weighted_std], axis=0, ignore_index=True)
cmp_long = cmp_long.sort_values(["segment", "strategy"]).reset_index(drop=True)

cmp_long_path = os.path.join(OUT_DIR, "sampling_compare_uniform_vs_weighted_long.csv")
cmp_long.to_csv(cmp_long_path, index=False)
print(f"[OK] wrote {cmp_long_path}")

# ------------------------------------------------------------
# WIDE comparison table
# ------------------------------------------------------------
value_cols = [c for c in cmp_long.columns if c not in ("K", "segment", "strategy")]
cmp_wide = cmp_long.pivot(index="segment", columns="strategy", values=value_cols)
cmp_wide.columns = [f"{v}__{s}" for (v, s) in cmp_wide.columns]
cmp_wide = cmp_wide.reset_index()

cmp_wide_path = os.path.join(OUT_DIR, "sampling_compare_uniform_vs_weighted_wide.csv")
cmp_wide.to_csv(cmp_wide_path, index=False)
print(f"[OK] wrote {cmp_wide_path}")

# ------------------------------------------------------------
# DELTAS: weighted - uniform
# ------------------------------------------------------------
delta = pd.DataFrame({"segment": cmp_wide["segment"]})

for col in value_cols:
    c_u = f"{col}__uniform"
    c_w = f"{col}__emu_weighted"
    if c_u in cmp_wide.columns and c_w in cmp_wide.columns:
        delta[f"{col}__delta_weighted_minus_uniform"] = cmp_wide[c_w] - cmp_wide[c_u]

delta["K"] = K
delta["TOP_M"] = TOP_M
delta["TEMP"] = TEMP

# optional nicer column order
front_cols = ["segment", "K", "TOP_M", "TEMP"]
other_cols = [c for c in delta.columns if c not in front_cols]
delta = delta[front_cols + other_cols]

delta_path = os.path.join(OUT_DIR, "sampling_compare_deltas_weighted_minus_uniform.csv")
delta.to_csv(delta_path, index=False)
print(f"[OK] wrote {delta_path}")

print("\n[DONE] Outputs in:", OUT_DIR)
print(" - sampling_results_emu_weighted.csv")
print(" - sampling_compare_uniform_vs_weighted_long.csv")
print(" - sampling_compare_uniform_vs_weighted_wide.csv")
print(" - sampling_compare_deltas_weighted_minus_uniform.csv")

# R4

## Split By Origin

In [ ]:
# ============================================================
# OOS STRATEGY COMPARISON: byPerson vs byOrigin
# FIXED (hygienic + deterministic):
#   - K = 1000
#   - uniform sampling (chosen + K non-chosen)
#   - z-standardize destination variables on the FULL TZ universe (no train-only scaling)
#   - deterministic seeds per segment (no Python hash randomness)
#   - same sampled choice sets across strategies (fair comparison)
#
# For each strategy and segment:
#   Sampled TRAIN metrics: NLL, MRR, McFadden_R2, rank_median, Top1/10/50/100
#   Sampled OOS   metrics: NLL, MRR, McFadden_R2, rank_median, Top1/10/50/100
#   FULL-choice-set metrics on OOS subset (all zones as alternatives):
#       NLL_full, MRR_full, McFadden_R2_full, rank_median_full
#       Top1/10/50/100_full
#       share_spearman_full, share_JS_full
#
# Outputs:
#   ModelRuns/OOSStrategy/oos_strategy_results_long.csv
#   ModelRuns/OOSStrategy/oos_strategy_results_wide.csv
#   ModelRuns/OOSStrategy/oos_strategy_deltas_byPerson_minus_byOrigin.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path

# -----------------------------
# CONFIG (everywhere)
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}   # deterministic, hygienic

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT   = "utility_emu"
MAP_NAME  = "NO"

# Two strategies to compare
STRATEGIES = {
    "byOrigin": "Trips/byOrigin",
    "byPerson": "Trips/byPerson",
}

OUT_DIR = "ModelRuns/OOSStrategy"
os.makedirs(OUT_DIR, exist_ok=True)

# Fixed robustness decision
K = 1000
SEED = 123

# Torch
DEVICE = "cpu"
DTYPE  = torch.float32

# Optimization
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# Full-choice-set evaluation subset sizes (OOS)
# (set to None if you want full on all OOS rows; runtime can explode)
FULL_EVAL_MAX = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256

LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",          # you said you moved to raw earlier; update if needed
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

# -----------------------------
# Small helpers
# -----------------------------
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300); q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

# -----------------------------
# IO helpers
# -----------------------------
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df

def load_emu_matrix_and_mapping(omx_path: Path, emu_mat: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if emu_mat not in f.list_matrices():
        raise ValueError(f"Matrix '{emu_mat}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        no = np.asarray(m[:]).astype(int)
    except Exception:
        no = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(no)}

    M = f[emu_mat]
    print(f"[OMX] Loading {emu_mat} into RAM... shape={M.shape} (float32)")
    emu = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return emu, zone_to_idx, no  # idx_to_zone = no

# -----------------------------
# Preprocess helpers
# -----------------------------
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    """
    Z-standardize on the FULL TZ universe (all alternatives),
    i.e., a fixed transformation of alternative attributes.
    """
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)

    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts

    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)  # chosen always first

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2
    )

# -----------------------------
# Model: train/eval on sampled sets
# -----------------------------
def train_mnl(emu_set, X_set, y, w, epochs=30, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, J = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state  # (alpha_hat, beta_hat)

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(1,10,50,100)):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = int(V.shape[1])
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

        out = {
            "NLL": nll,
            "MRR": mrr,
            "McFadden_R2": float(r2),
            "rank_median": rank_median,
            "J": J,
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))

    return out

# -----------------------------
# FULL-choice-set evaluation on OOS subset (all zones available)
# -----------------------------
def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,   # OMX idx -> zone id
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(1,10,50,100),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)
    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    # Aj(j) aligned to OMX indices
    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    # shares on TZ universe
    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}
    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)

    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    ll_model = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    top_num = {k: 0.0 for k in topKs}

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        # pass 1: max V
        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        # pass 2: sumexp + rank
        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        nll_num += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        # pass 3: shares accumulation
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    out = {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    return out

# ============================================================
# MAIN
# ============================================================

print("[LOAD] TZ features...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
need_cols = ["npvm_id"] + LOG1P_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

tz_feat0 = tz[["npvm_id"] + LOG1P_COLS + ["geometry"]].copy()
tz_feat0["npvm_id"] = pd.to_numeric(tz_feat0["npvm_id"], errors="coerce")
tz_feat0 = tz_feat0.dropna(subset=["npvm_id"]).copy()
tz_feat0["npvm_id"] = tz_feat0["npvm_id"].astype(int)

# Z-standardize on ALL TZ (fixed transformation of alternative attributes)
print("[PREP] Z-standardize destination variables on full TZ universe...")
tz_feat0, z_cols, mu_all, sd_all = standardize_all_zones(tz_feat0, LOG1P_COLS, suffix="_z")

print("[LOAD] EMU matrix + mapping...")
emu_mat, zone_to_idx, idx_to_zone = load_emu_matrix_and_mapping(UTIL_OMX, EMU_MAT, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

results = []
TOPKS = (1, 10, 50, 100)

for strategy_name, trips_dir in STRATEGIES.items():
    print("\n" + "#" * 110)
    print(f"# STRATEGY = {strategy_name} | trips_dir={trips_dir} | K={K}")
    print("#" * 110)

    for seg in SEGMENTS:
        print("\n" + "=" * 95)
        print(f"Segment {seg} | strategy={strategy_name}")
        print("=" * 95)

        train_df = load_segment_csv(trips_dir, seg, "train")
        oos_df   = load_segment_csv(trips_dir, seg, "oos")

        # Use the same standardized TZ table for all segments/strategies
        tz_feat_seg = tz_feat0

        # Hygienic deterministic seeds:
        # - same sampled choice sets across strategies (fair)
        seed_tr = SEED + 1000 * SEG_SEED[seg] + 10
        seed_te = SEED + 1000 * SEG_SEED[seg] + 20

        # Build sampled designs
        emu_tr, X_tr, y_tr, w_tr, train_df2 = build_design(
            train_df, tz_feat_seg, z_cols, emu_mat, zone_to_idx, k=K, seed=seed_tr
        )
        emu_te, X_te, y_te, w_te, oos_df2 = build_design(
            oos_df, tz_feat_seg, z_cols, emu_mat, zone_to_idx, k=K, seed=seed_te
        )

        print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
        print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

        # Train
        alpha_hat, beta_hat = train_mnl(
            emu_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
        )

        # Sampled metrics
        met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS)
        met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS)

        # Full metrics on OOS subset (all destinations) - deterministic subset per segment
        max_n = FULL_EVAL_MAX.get(seg, 8000)
        seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg]  # deterministic
        full_met = full_choice_eval_oos_subset(
            oos_df=oos_df2,
            tz_feat_seg=tz_feat_seg,
            z_cols=z_cols,
            emu_mat=emu_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=max_n,
            seed=seed_full,
            chunk=FULL_EVAL_CHUNK,
            topKs=TOPKS,
        )

        row = {
            "strategy": strategy_name,
            "segment": seg,
            "K": K,
            "alpha_emu": float(alpha_hat),

            "NLL_train": met_tr["NLL"],
            "MRR_train": met_tr["MRR"],
            "McFadden_R2_train": met_tr["McFadden_R2"],
            "rank_median_train": met_tr["rank_median"],
            "Top1_train": met_tr["Top1"],
            "Top10_train": met_tr["Top10"],
            "Top50_train": met_tr["Top50"],
            "Top100_train": met_tr["Top100"],

            "NLL_oos": met_te["NLL"],
            "MRR_oos": met_te["MRR"],
            "McFadden_R2_oos": met_te["McFadden_R2"],
            "rank_median_oos": met_te["rank_median"],
            "Top1_oos": met_te["Top1"],
            "Top10_oos": met_te["Top10"],
            "Top50_oos": met_te["Top50"],
            "Top100_oos": met_te["Top100"],
        }
        row.update(full_met)
        results.append(row)

        print("[SAMPLED TRAIN]",
              f"NLL={row['NLL_train']:.4f} MRR={row['MRR_train']:.4f} R2={row['McFadden_R2_train']:.4f} "
              f"Top1={row['Top1_train']:.3f} Top10={row['Top10_train']:.3f} rMed={row['rank_median_train']:.1f}")
        print("[SAMPLED OOS ]",
              f"NLL={row['NLL_oos']:.4f} MRR={row['MRR_oos']:.4f} R2={row['McFadden_R2_oos']:.4f} "
              f"Top1={row['Top1_oos']:.3f} Top10={row['Top10_oos']:.3f} rMed={row['rank_median_oos']:.1f}")

        if full_met:
            print("[FULL OOS   ]",
                  f"NLL_full={row['NLL_full']:.4f} MRR_full={row['MRR_full']:.4f} R2_full={row['McFadden_R2_full']:.4f} "
                  f"Top1_full={row['Top1_full']:.3f} Top10_full={row['Top10_full']:.3f} rMed_full={row['rank_median_full']:.1f}")
            print("[SHARES FULL ]",
                  f"spearman={row['share_spearman_full']:.4f} JS={row['share_JS_full']:.4f} "
                  f"N_full={row['N_full_eval']} J_full={row['J_full']}")

# -----------------------------
# Save outputs
# -----------------------------
res_df = pd.DataFrame(results).sort_values(["segment", "strategy"]).reset_index(drop=True)

long_path = os.path.join(OUT_DIR, "oos_strategy_results_long.csv")
res_df.to_csv(long_path, index=False)
print(f"\n[OK] wrote {long_path}")

# Wide (pivot) for easy comparison
value_cols = [c for c in res_df.columns if c not in ("strategy", "segment")]
wide = res_df.pivot(index="segment", columns="strategy", values=value_cols)

# flatten MultiIndex columns
wide.columns = [f"{v}__{s}" for (v, s) in wide.columns]
wide = wide.reset_index()

wide_path = os.path.join(OUT_DIR, "oos_strategy_results_wide.csv")
wide.to_csv(wide_path, index=False)
print(f"[OK] wrote {wide_path}")

# Deltas: byPerson - byOrigin
delta = pd.DataFrame({"segment": wide["segment"]})
for col in value_cols:
    c_p = f"{col}__byPerson"
    c_o = f"{col}__byOrigin"
    if c_p in wide.columns and c_o in wide.columns:
        delta[col + "__delta_byPerson_minus_byOrigin"] = wide[c_p] - wide[c_o]

delta_path = os.path.join(OUT_DIR, "oos_strategy_deltas_byPerson_minus_byOrigin.csv")
delta.to_csv(delta_path, index=False)
print(f"[OK] wrote {delta_path}")

print("\n[DONE] Outputs in:", OUT_DIR)


# R5 Survey Year

In [ ]:
# ============================================================
# ROBUSTNESS: survey-wave sensitivity
# Compare:
#   - 2015 only
#   - 2021 only
#   - full pooled sample
#
# COHERENT WITH BASELINE:
#   - with LIE
#   - byPerson split
#   - same 13 destination-side variables
#   - log1p variables already in GPKG
#   - GLOBAL z-standardisation on full TZ universe
#   - K = 1000
#   - 50 epochs
#   - uniform sampling
#   - weighted MNL
#   - full-choice-set OOS evaluation
#
# IMPORTANT:
#   - NO artificial matching of row counts or WP totals across waves
#   - each wave is estimated on its own actual train/oos subset
#   - "full" uses the full pooled train/oos sample
#   - coefficient SEs via Fisher information on sampled train set
#
# COEFFICIENT DIFFERENCE TESTS:
#   1) 2021 - 2015
#   2) full - 2021
#   3) full - 2015
#
#   z = delta / sqrt(se_a^2 + se_b^2)
#
# INPUT:
#   Trips/byPerson/destinations_{SEG}_{train|oos}.csv
#   must contain column: survey
#
# OUTPUTS:
#   ModelRuns/WaveComparison_2015_2021/
#   - wave_model_results.csv
#   - wave_results_wide.csv
#   - wave_results_deltas.csv
#   - wave_coefficients.csv
#   - wave_coef_deltas_2021_minus_2015.csv
#   - wave_coef_deltas_full_minus_2021.csv
#   - wave_coef_deltas_full_minus_2015.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

import torch
import openmatrix as omx
from pathlib import Path
from math import erf, sqrt


# -----------------------------
# CONFIG
# -----------------------------
TRIPS_DIR = "Trips/byPerson"
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL   = "orig_zone"
DEST_COL   = "dest_zone"
WP_COL     = "WP"
SURVEY_COL = "survey"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT   = "utility_emu"
MAP_NAME  = "NO"

OUT_DIR = "ModelRuns/WaveComparison_2015_2021"
os.makedirs(OUT_DIR, exist_ok=True)

K = 1000
BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

FULL_EVAL_MAX = {
    "YS": 20000,
    "OS": 8000,
    "YL": 8000,
    "OL": 8000,
}
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256

TOPKS = (1, 10, 50, 100)

BASE_LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]


# ============================================================
# Helpers
# ============================================================
def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)


# ============================================================
# IO helpers
# ============================================================
def load_segment_csv(seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(TRIPS_DIR, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    need = [ORIG_COL, DEST_COL, WP_COL, SURVEY_COL]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"{path}: missing {miss}. Available: {list(df.columns)[:60]}")

    for c in need:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=need).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df[SURVEY_COL] = df[SURVEY_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


# ============================================================
# Preprocess helpers
# ============================================================
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def subset_wave(df: pd.DataFrame, wave: int) -> pd.DataFrame:
    return df[df[SURVEY_COL] == int(wave)].copy()

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)

    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts

    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (
        (orig_idx >= 0)
        & (dest_idx.min(axis=1) >= 0)
        & np.isfinite(w)
        & (w > 0)
        & np.isfinite(X).all(axis=(1, 2))
    )
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping/features/bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
    )


# ============================================================
# Model
# ============================================================
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048,
              weight_decay=0.0, seed: int = 0):
    torch.manual_seed(int(seed))

    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(1, 10, 50, 100)):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = int(V.shape[1])
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

        out = {
            "NLL": nll,
            "MRR": mrr,
            "McFadden_R2": float(r2),
            "rank_median": rank_median,
            "J": J,
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))
        return out

def compute_se_fisher(emu_set, X_set, w, alpha, beta, batch_size=1024):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        N, J = emu_set.shape
        Pdim = X_set.shape[2]
        D = 1 + Pdim
        info = torch.zeros((D, D), dtype=DTYPE, device=DEVICE)

        for s in range(0, N, batch_size):
            e = min(N, s + batch_size)
            emu_b = emu_set[s:e]
            X_b   = X_set[s:e]
            w_b   = w[s:e]

            V = alpha_t * emu_b + torch.einsum("bjp,p->bj", X_b, beta_t)
            P = torch.softmax(V, dim=1)

            Z = torch.empty((e - s, J, D), dtype=DTYPE, device=DEVICE)
            Z[:, :, 0] = emu_b
            Z[:, :, 1:] = X_b

            term1 = torch.einsum("bj,bjd,bjk->bdk", P, Z, Z)
            m = torch.einsum("bj,bjd->bd", P, Z)
            term2 = torch.einsum("bd,be->bde", m, m)

            info_b = term1 - term2
            info += torch.einsum("b,bde->de", w_b, info_b)

        info_cpu = info.detach().cpu().numpy().astype(np.float64)
        cov = np.linalg.pinv(info_cpu)
        se = np.sqrt(np.clip(np.diag(cov), 0.0, np.inf))
        return se, cov


# ============================================================
# FULL-choice-set evaluation
# ============================================================
def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(1, 10, 50, 100),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}
    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)

    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    ll_model = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    top_num = {k: 0.0 for k in topKs}

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        nll_num += wn * (-logp_chosen)
        ll_model += wn * logp_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    out = {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    return out


# ============================================================
# Main runner for one dataset
# ============================================================
def run_one_spec(seg: str, tag: str, train_df: pd.DataFrame, oos_df: pd.DataFrame,
                 tz_feat0: pd.DataFrame, Z_COLS: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, idx_to_zone: np.ndarray,
                 seed_offset: int):
    if len(train_df) == 0 or len(oos_df) == 0:
        raise ValueError(f"{seg} | {tag}: empty train or oos subset.")

    seed_tr = BASE_SEED + 1000 * SEG_SEED[seg] + seed_offset + 10
    seed_te = BASE_SEED + 1000 * SEG_SEED[seg] + seed_offset + 20
    seed_model = BASE_SEED + 1000 * SEG_SEED[seg] + seed_offset + 30
    seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg] + seed_offset

    emu_tr, X_tr, y_tr, w_tr, tr2 = build_design(
        train_df, tz_feat0, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_tr
    )
    emu_te, X_te, y_te, w_te, te2 = build_design(
        oos_df, tz_feat0, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_te
    )

    print(f"  [{tag}] sampled TRAIN rows={len(tr2):,} WP={float(tr2[WP_COL].sum()):.2f}")
    print(f"  [{tag}] sampled OOS   rows={len(te2):,} WP={float(te2[WP_COL].sum()):.2f}")

    alpha_hat, beta_hat = train_mnl(
        emu_tr, X_tr, y_tr, w_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
        seed=seed_model
    )

    met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS)
    met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS)

    max_n = FULL_EVAL_MAX.get(seg, 8000)
    full_met = full_choice_eval_oos_subset(
        oos_df=te2,
        tz_feat_seg=tz_feat0,
        z_cols=Z_COLS,
        emu_mat=emu_mat,
        zone_to_idx=zone_to_idx,
        idx_to_zone=idx_to_zone,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=max_n,
        seed=seed_full,
        chunk=FULL_EVAL_CHUNK,
        topKs=TOPKS,
    )

    se, cov = compute_se_fisher(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, batch_size=1024)

    param_names = ["alpha_emu"] + Z_COLS
    theta = np.concatenate([[alpha_hat], beta_hat])

    row = {
        "segment": seg,
        "wave": tag,
        "K": K,
        "alpha_emu": float(alpha_hat),
        "train_rows": int(len(tr2)),
        "oos_rows": int(len(te2)),
        "train_WP": float(tr2[WP_COL].sum()),
        "oos_WP": float(te2[WP_COL].sum()),
    }

    for k_, v_ in met_tr.items():
        row[f"train_{k_}"] = v_
    for k_, v_ in met_te.items():
        row[f"oos_{k_}"] = v_
    row.update(full_met)

    return row, theta, se, param_names


# ============================================================
# MAIN
# ============================================================
print("[LOAD] TZ features...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)

need_cols = ["npvm_id"] + BASE_LOG1P_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

tz_feat0 = tz[["npvm_id"] + BASE_LOG1P_COLS + ["geometry"]].copy()
tz_feat0["npvm_id"] = pd.to_numeric(tz_feat0["npvm_id"], errors="coerce")
tz_feat0 = tz_feat0.dropna(subset=["npvm_id"]).copy()
tz_feat0["npvm_id"] = tz_feat0["npvm_id"].astype(int)

print("[PREP] Global z-standardize destination vars on full TZ universe...")
tz_feat0, Z_COLS, MU_ALL, SD_ALL = standardize_all_zones(tz_feat0, BASE_LOG1P_COLS, suffix="_z")

print("[LOAD] EMU matrix + mapping...")
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

results = []
coef_rows = []

delta_21_15_rows = []
delta_full_21_rows = []
delta_full_15_rows = []

for seg in SEGMENTS:
    print("\n" + "=" * 110)
    print(f"SEGMENT {seg} | K={K} | byPerson | withLIE | wave comparison 2015 vs 2021 vs full")
    print("=" * 110)

    train_all = load_segment_csv(seg, "train")
    oos_all   = load_segment_csv(seg, "oos")

    tr15 = subset_wave(train_all, 2015)
    tr21 = subset_wave(train_all, 2021)
    te15 = subset_wave(oos_all, 2015)
    te21 = subset_wave(oos_all, 2021)

    print(f"[DATA] TRAIN full rows={len(train_all):,} WP={float(train_all[WP_COL].sum()):.2f}")
    print(f"[DATA] TRAIN 2015 rows={len(tr15):,} WP={float(tr15[WP_COL].sum()):.2f}")
    print(f"[DATA] TRAIN 2021 rows={len(tr21):,} WP={float(tr21[WP_COL].sum()):.2f}")
    print(f"[DATA] OOS   full rows={len(oos_all):,} WP={float(oos_all[WP_COL].sum()):.2f}")
    print(f"[DATA] OOS   2015 rows={len(te15):,} WP={float(te15[WP_COL].sum()):.2f}")
    print(f"[DATA] OOS   2021 rows={len(te21):,} WP={float(te21[WP_COL].sum()):.2f}")

    row15, theta15, se15, names15 = run_one_spec(
        seg=seg, tag="2015",
        train_df=tr15, oos_df=te15,
        tz_feat0=tz_feat0, Z_COLS=Z_COLS,
        emu_mat=emu_mat, zone_to_idx=zone_to_idx, idx_to_zone=idx_to_zone,
        seed_offset=1500
    )

    row21, theta21, se21, names21 = run_one_spec(
        seg=seg, tag="2021",
        train_df=tr21, oos_df=te21,
        tz_feat0=tz_feat0, Z_COLS=Z_COLS,
        emu_mat=emu_mat, zone_to_idx=zone_to_idx, idx_to_zone=idx_to_zone,
        seed_offset=2100
    )

    rowfull, thetafull, sefull, namesfull = run_one_spec(
        seg=seg, tag="full",
        train_df=train_all, oos_df=oos_all,
        tz_feat0=tz_feat0, Z_COLS=Z_COLS,
        emu_mat=emu_mat, zone_to_idx=zone_to_idx, idx_to_zone=idx_to_zone,
        seed_offset=500
    )

    results.extend([row15, row21, rowfull])

    if not (names15 == names21 == namesfull):
        raise RuntimeError("Parameter mismatch across specifications.")

    for name, b, s in zip(names15, theta15, se15):
        coef_rows.append({
            "segment": seg,
            "wave": "2015",
            "param": name,
            "coef": float(b),
            "se": float(s),
        })

    for name, b, s in zip(names21, theta21, se21):
        coef_rows.append({
            "segment": seg,
            "wave": "2021",
            "param": name,
            "coef": float(b),
            "se": float(s),
        })

    for name, b, s in zip(namesfull, thetafull, sefull):
        coef_rows.append({
            "segment": seg,
            "wave": "full",
            "param": name,
            "coef": float(b),
            "se": float(s),
        })

    # 2021 - 2015
    for name, b15, s15, b21, s21 in zip(names15, theta15, se15, theta21, se21):
        delta = float(b21 - b15)
        se_delta = float(np.sqrt((s15 ** 2) + (s21 ** 2)))
        z = delta / (se_delta + 1e-12)
        p = float(2.0 * (1.0 - norm_cdf(abs(z))))
        delta_21_15_rows.append({
            "segment": seg,
            "param": name,
            "coef_2015": float(b15),
            "se_2015": float(s15),
            "coef_2021": float(b21),
            "se_2021": float(s21),
            "delta_2021_minus_2015": delta,
            "se_delta": se_delta,
            "z": float(z),
            "p": p,
        })

    # full - 2021
    for name, b21, s21, bfull, sfull in zip(names21, theta21, se21, thetafull, sefull):
        delta = float(bfull - b21)
        se_delta = float(np.sqrt((s21 ** 2) + (sfull ** 2)))
        z = delta / (se_delta + 1e-12)
        p = float(2.0 * (1.0 - norm_cdf(abs(z))))
        delta_full_21_rows.append({
            "segment": seg,
            "param": name,
            "coef_2021": float(b21),
            "se_2021": float(s21),
            "coef_full": float(bfull),
            "se_full": float(sfull),
            "delta_full_minus_2021": delta,
            "se_delta": se_delta,
            "z": float(z),
            "p": p,
        })

    # full - 2015
    for name, b15, s15, bfull, sfull in zip(names15, theta15, se15, thetafull, sefull):
        delta = float(bfull - b15)
        se_delta = float(np.sqrt((s15 ** 2) + (sfull ** 2)))
        z = delta / (se_delta + 1e-12)
        p = float(2.0 * (1.0 - norm_cdf(abs(z))))
        delta_full_15_rows.append({
            "segment": seg,
            "param": name,
            "coef_2015": float(b15),
            "se_2015": float(s15),
            "coef_full": float(bfull),
            "se_full": float(sfull),
            "delta_full_minus_2015": delta,
            "se_delta": se_delta,
            "z": float(z),
            "p": p,
        })


# -----------------------------
# Save outputs
# -----------------------------
res_df = pd.DataFrame(results).sort_values(["segment", "wave"]).reset_index(drop=True)
res_path = os.path.join(OUT_DIR, "wave_model_results.csv")
res_df.to_csv(res_path, index=False)
print(f"\n[OK] wrote {res_path}")

coef_df = pd.DataFrame(coef_rows).sort_values(["segment", "wave", "param"]).reset_index(drop=True)
coef_path = os.path.join(OUT_DIR, "wave_coefficients.csv")
coef_df.to_csv(coef_path, index=False)
print(f"[OK] wrote {coef_path}")

delta_21_15_df = pd.DataFrame(delta_21_15_rows).sort_values(["segment", "param"]).reset_index(drop=True)
delta_21_15_path = os.path.join(OUT_DIR, "wave_coef_deltas_2021_minus_2015.csv")
delta_21_15_df.to_csv(delta_21_15_path, index=False)
print(f"[OK] wrote {delta_21_15_path}")

delta_full_21_df = pd.DataFrame(delta_full_21_rows).sort_values(["segment", "param"]).reset_index(drop=True)
delta_full_21_path = os.path.join(OUT_DIR, "wave_coef_deltas_full_minus_2021.csv")
delta_full_21_df.to_csv(delta_full_21_path, index=False)
print(f"[OK] wrote {delta_full_21_path}")

delta_full_15_df = pd.DataFrame(delta_full_15_rows).sort_values(["segment", "param"]).reset_index(drop=True)
delta_full_15_path = os.path.join(OUT_DIR, "wave_coef_deltas_full_minus_2015.csv")
delta_full_15_df.to_csv(delta_full_15_path, index=False)
print(f"[OK] wrote {delta_full_15_path}")

# -----------------------------
# Wide results
# -----------------------------
value_cols = [c for c in res_df.columns if c not in ("segment", "wave")]
wide = res_df.pivot(index="segment", columns="wave", values=value_cols)
wide.columns = [f"{v}__{w}" for (v, w) in wide.columns]
wide = wide.reset_index()

wide_path = os.path.join(OUT_DIR, "wave_results_wide.csv")
wide.to_csv(wide_path, index=False)
print(f"[OK] wrote {wide_path}")

# -----------------------------
# Model-results deltas:
#   1) 2021 - 2015
#   2) full - 2021
#   3) full - 2015
# -----------------------------
delta_model = pd.DataFrame({"segment": wide["segment"]})

for col in value_cols:
    c15 = f"{col}__2015"
    c21 = f"{col}__2021"
    cfull = f"{col}__full"

    if c15 in wide.columns and c21 in wide.columns:
        delta_model[f"{col}__delta_2021_minus_2015"] = wide[c21] - wide[c15]

    if cfull in wide.columns and c21 in wide.columns:
        delta_model[f"{col}__delta_full_minus_2021"] = wide[cfull] - wide[c21]

    if cfull in wide.columns and c15 in wide.columns:
        delta_model[f"{col}__delta_full_minus_2015"] = wide[cfull] - wide[c15]

delta_model_path = os.path.join(OUT_DIR, "wave_results_deltas.csv")
delta_model.to_csv(delta_model_path, index=False)
print(f"[OK] wrote {delta_model_path}")

print("\n[DONE] Outputs in:", OUT_DIR)
print(" - wave_model_results.csv")
print(" - wave_results_wide.csv")
print(" - wave_results_deltas.csv")
print(" - wave_coefficients.csv")
print(" - wave_coef_deltas_2021_minus_2015.csv")
print(" - wave_coef_deltas_full_minus_2021.csv")
print(" - wave_coef_deltas_full_minus_2015.csv")

# R6 Regularization

In [ ]:
# ============================================================
# REGULARIZATION COMPARISON
# EMU | with LIE | byPerson split | K=1000 | uniform sampling
# GLOBAL Z-STANDARDIZATION on full TZ universe (same scaling everywhere)
#
# Variants per segment:
#   (A) BASE     : no regularization
#   (B) L2(beta) : L2 shrinkage on beta only (NOT alpha)
#   (C) L1(beta) : L1 penalty on beta only (NOT alpha)
#
# Metrics:
#   - sampled TRAIN (K+1)
#   - sampled OOS
#   - FULL choice set on OOS subset (all destinations)
#
# Outputs:
#   ModelRuns/RegularizationComparison_emu_globalZ/
#     metrics_all.csv
#     coefficient_table.csv
#     deltas_metrics.csv
#     deltas_coefs.csv
#     plot_<metric>.png
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import torch
import openmatrix as omx
from pathlib import Path


# -----------------------------
# CONFIG
# -----------------------------
TRIPS_DIR = "Trips/byPerson"
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT   = "utility_emu"
MAP_NAME  = "NO"

K = 1000

# --- seeds (deterministic + consistent) ---
SEED_BASE = 123
SEED_DESTSET_TRAIN = 12345
SEED_DESTSET_OOS   = 54321
SEED_TORCH_BASE    = 22222
SEED_FULL_EVAL     = 77777

# Torch
DEVICE = "cpu"          # set "mps" if desired on Mac
DTYPE  = torch.float32

# Optimization
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03

# Regularization strengths
L2_LAMBDA_BETA = 1e-5
L1_LAMBDA_BETA = 1e-4

# Full-eval subset sizes (OOS)
FULL_EVAL_MAX = {"YS": 100000, "OS": 100000, "YL": 100000, "OL": 100000}
FULL_EVAL_CHUNK = 256

OUT_DIR = "ModelRuns/RegularizationComparison_emu_globalZ"
os.makedirs(OUT_DIR, exist_ok=True)

LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]


# ============================================================
# IO helpers
# ============================================================
def load_segment_csv(seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(TRIPS_DIR, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    need = [ORIG_COL, DEST_COL, WP_COL]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"{path}: missing {miss}. Available: {list(df.columns)[:60]}")

    df = df.dropna(subset=need).copy()
    for c in need:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=need).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df


def load_emu_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {omx_path.name}:{mat_name} into RAM... shape={M.shape} float32")
    emu = np.asarray(M[:, :], dtype=np.float32)
    f.close()

    return emu, zone_to_idx, idx_to_zone


# ============================================================
# GLOBAL Z-standardization on full TZ universe
# ============================================================
def standardize_global(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd


# ============================================================
# Sampling helpers (DESTSET fixed across variants)
# ============================================================
def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts
    return dest_set


def make_destset(df: pd.DataFrame, tz_feat: pd.DataFrame, k: int, seed: int):
    rng = np.random.default_rng(seed)
    chosen = df[DEST_COL].to_numpy(dtype=int)
    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    return sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)


def build_design_with_fixed_destset(
    df: pd.DataFrame,
    tz_feat: pd.DataFrame,
    feat_cols_z: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    dest_set: np.ndarray
):
    orig = df[ORIG_COL].to_numpy(dtype=int)
    w    = df[WP_COL].to_numpy(dtype=np.float64)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    k = dest_set.shape[1] - 1
    Pdim = len(feat_cols_z)

    X = np.empty((N, k + 1, Pdim), dtype=np.float32)
    flat = dest_set.ravel()

    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]
    dest_set2 = dest_set[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)  # chosen always first

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
        dest_set2,
    )


# ============================================================
# Metrics (sampled)
# ============================================================
def eval_sampled_metrics(emu_set, X_set, w, alpha, beta):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())
        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        top1   = float((w * (rank <= 1).float()).sum().cpu() / (sum_w + 1e-9))
        top10  = float((w * (rank <= 10).float()).sum().cpu() / (sum_w + 1e-9))
        top50  = float((w * (rank <= 50).float()).sum().cpu() / (sum_w + 1e-9))
        top100 = float((w * (rank <= 100).float()).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = V.shape[1]
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    return {
        "NLL": nll,
        "MRR": mrr,
        "Top1": top1,
        "Top10": top10,
        "Top50": top50,
        "Top100": top100,
        "McFadden_R2": float(r2),
        "rank_median": rank_median,
        "J": int(J),
    }


# ============================================================
# FULL choice-set eval on OOS subset (all destinations)
# ============================================================
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

def full_choice_eval_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}
    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)

    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    ll_model = 0.0

    top1_num = top10_num = top50_num = top100_num = 0.0
    ranks = np.empty(len(df), dtype=np.int64)

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

        nll_num += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)

        rank = better + 1
        ranks[n] = rank

        mrr_num += wn * (1.0 / float(rank))
        top1_num   += wn * (1.0 if rank <= 1 else 0.0)
        top10_num  += wn * (1.0 if rank <= 10 else 0.0)
        top50_num  += wn * (1.0 if rank <= 50 else 0.0)
        top100_num += wn * (1.0 if rank <= 100 else 0.0)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))

    top1_full   = float(top1_num   / (sum_w + 1e-9))
    top10_full  = float(top10_num  / (sum_w + 1e-9))
    top50_full  = float(top50_num  / (sum_w + 1e-9))
    top100_full = float(top100_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    return {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "Top1_full": top1_full,
        "Top10_full": top10_full,
        "Top50_full": top50_full,
        "Top100_full": top100_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }


# ============================================================
# Training (BASE / L2(beta) / L1(beta))
# ============================================================
def train_mnl(
    emu_set, X_set, y, w,
    epochs=30, lr=0.03, batch_size=2048,
    l2_lambda_beta=0.0,
    l1_lambda_beta=0.0,
    torch_seed=0,
):
    """
    L2(beta): Adam weight_decay on beta parameter group only.
    L1(beta): explicit penalty on beta only.
    """
    torch.manual_seed(int(torch_seed))
    N, J = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam(
        [
            {"params": [alpha], "weight_decay": 0.0},
            {"params": [beta],  "weight_decay": float(l2_lambda_beta)},
        ],
        lr=lr,
    )
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            base_loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            loss = base_loss
            if l1_lambda_beta > 0.0:
                loss = loss + float(l1_lambda_beta) * torch.sum(torch.abs(beta))

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(base_loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"      epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state


def beta_sparsity(beta: np.ndarray, thr=1e-6):
    b = np.asarray(beta, dtype=np.float64)
    return {
        "beta_l1": float(np.sum(np.abs(b))),
        "beta_l2": float(np.sqrt(np.sum(b * b))),
        "beta_n_small": int(np.sum(np.abs(b) < thr)),
        "beta_share_small": float(np.mean(np.abs(b) < thr)),
    }


# ============================================================
# Plot helper
# ============================================================
def make_metric_plot(df: pd.DataFrame, metric: str, out_path: str):
    plot_df = df[["segment", "variant", metric]].copy()
    if plot_df[metric].isna().all():
        return

    pivot = plot_df.pivot(index="segment", columns="variant", values=metric)
    pivot = pivot.reindex(index=SEGMENTS)

    ax = pivot.plot(kind="bar", figsize=(8, 4))
    ax.set_title(metric)
    ax.set_xlabel("segment")
    ax.set_ylabel(metric)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


# ============================================================
# MAIN
# ============================================================
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
need_cols = ["npvm_id"] + LOG1P_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

tz_feat0 = tz[["npvm_id"] + LOG1P_COLS + ["geometry"]].copy()
tz_feat0["npvm_id"] = pd.to_numeric(tz_feat0["npvm_id"], errors="coerce")
tz_feat0 = tz_feat0.dropna(subset=["npvm_id"]).copy()
tz_feat0["npvm_id"] = tz_feat0["npvm_id"].astype(int)

print("[PREP] Global z-standardization on full TZ universe...")
tz_feat0, z_cols, mu_g, sd_g = standardize_global(tz_feat0, LOG1P_COLS, suffix="_z")

emu_mat, zone_to_idx, idx_to_zone = load_emu_matrix_and_mapping(UTIL_OMX, EMU_MAT, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

VARIANTS = [
    {"name": "BASE", "l2": 0.0,            "l1": 0.0},
    {"name": "L2",   "l2": L2_LAMBDA_BETA, "l1": 0.0},
    {"name": "L1",   "l2": 0.0,            "l1": L1_LAMBDA_BETA},
]

rows_metrics = []
rows_coefs = []

for seg in SEGMENTS:
    print("\n" + "=" * 110)
    print(f"SEGMENT {seg} | EMU | K={K} | byPerson | uniform | GLOBAL-Z | BASE vs L2(beta) vs L1(beta)")
    print("=" * 110)

    train_df = load_segment_csv(seg, "train")
    oos_df   = load_segment_csv(seg, "oos")

    # fixed destsets for fair comparison across variants
    destset_tr = make_destset(train_df, tz_feat0, k=K, seed=SEED_DESTSET_TRAIN + 1000 * SEG_SEED[seg])
    destset_te = make_destset(oos_df,   tz_feat0, k=K, seed=SEED_DESTSET_OOS   + 1000 * SEG_SEED[seg])

    # sampled designs once
    emu_tr, X_tr, y_tr, w_tr, train_df2, _ = build_design_with_fixed_destset(
        train_df, tz_feat0, z_cols, emu_mat, zone_to_idx, destset_tr
    )
    emu_te, X_te, y_te, w_te, oos_df2, _ = build_design_with_fixed_destset(
        oos_df, tz_feat0, z_cols, emu_mat, zone_to_idx, destset_te
    )

    print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
    print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

    for vi, v in enumerate(VARIANTS):
        vname = v["name"]
        print("\n" + "-" * 90)
        print(f"[{vname}] train | L2(beta)={v['l2']} | L1(beta)={v['l1']}")
        print("-" * 90)

        alpha_hat, beta_hat = train_mnl(
            emu_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE,
            l2_lambda_beta=float(v["l2"]),
            l1_lambda_beta=float(v["l1"]),
            torch_seed=SEED_TORCH_BASE + 1000 * SEG_SEED[seg] + 100 * vi
        )

        met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat)
        met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat)

        full_met = full_choice_eval_oos_subset(
            oos_df=oos_df2,
            tz_feat_seg=tz_feat0,
            z_cols=z_cols,
            emu_mat=emu_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=FULL_EVAL_MAX.get(seg, 8000),
            seed=SEED_FULL_EVAL + 1000 * SEG_SEED[seg] + 100 * vi,
            chunk=FULL_EVAL_CHUNK,
        )

        sp = beta_sparsity(beta_hat, thr=1e-6)

        def pack(prefix, met):
            return {
                f"{prefix}_NLL": met["NLL"],
                f"{prefix}_MRR": met["MRR"],
                f"{prefix}_Top1": met["Top1"],
                f"{prefix}_Top10": met["Top10"],
                f"{prefix}_Top50": met["Top50"],
                f"{prefix}_Top100": met["Top100"],
                f"{prefix}_McFadden_R2": met["McFadden_R2"],
                f"{prefix}_rank_median": met["rank_median"],
            }

        row = {
            "segment": seg,
            "variant": vname,
            "K": K,
            "alpha_emu": float(alpha_hat),
            "l2_lambda_beta": float(v["l2"]),
            "l1_lambda_beta": float(v["l1"]),
            "train_rows": int(len(train_df2)),
            "oos_rows": int(len(oos_df2)),
            "train_WP": float(train_df2[WP_COL].sum()),
            "oos_WP": float(oos_df2[WP_COL].sum()),
            **pack("train", met_tr),
            **pack("oos", met_te),
            **full_met,
            **sp,
        }
        rows_metrics.append(row)

        coef_names = ["alpha_emu"] + z_cols
        coef_vals = np.concatenate([[alpha_hat], beta_hat]).astype(np.float64)
        rows_coefs.append(pd.DataFrame({
            "segment": seg,
            "variant": vname,
            "param": coef_names,
            "coef": coef_vals,
        }))

        print(f"[{vname} SAMPLED train] NLL={row['train_NLL']:.4f} | MRR={row['train_MRR']:.4f} | Top10={row['train_Top10']:.4f} | R2={row['train_McFadden_R2']:.4f}")
        print(f"[{vname} SAMPLED oos  ] NLL={row['oos_NLL']:.4f} | MRR={row['oos_MRR']:.4f} | Top10={row['oos_Top10']:.4f} | R2={row['oos_McFadden_R2']:.4f}")
        if full_met:
            print(f"[{vname} FULL oos    ] NLL_full={row['NLL_full']:.4f} | MRR_full={row['MRR_full']:.4f} | Top10_full={row['Top10_full']:.4f} | R2_full={row['McFadden_R2_full']:.4f}")
            print(f"[{vname} SHARES      ] spear={row['share_spearman_full']:.4f} | JS={row['share_JS_full']:.4f}")
        print(f"[{vname} BETA norms  ] L1={row['beta_l1']:.4e} | L2={row['beta_l2']:.4e} | share(|beta|<1e-6)={row['beta_share_small']:.3f}")

# --- save outputs ---
metrics_df = pd.DataFrame(rows_metrics).sort_values(["segment", "variant"]).reset_index(drop=True)
coef_df_all = pd.concat(rows_coefs, ignore_index=True)

metrics_path = os.path.join(OUT_DIR, "metrics_all.csv")
coefs_path   = os.path.join(OUT_DIR, "coefficient_table.csv")

metrics_df.to_csv(metrics_path, index=False)
coef_df_all.to_csv(coefs_path, index=False)

print("\n[OK] wrote:")
print(" -", metrics_path)
print(" -", coefs_path)

# ============================================================
# DELTAS
# ============================================================
base = metrics_df[metrics_df["variant"] == "BASE"].set_index("segment")
l2df = metrics_df[metrics_df["variant"] == "L2"].set_index("segment")
l1df = metrics_df[metrics_df["variant"] == "L1"].set_index("segment")

NUM_COLS = [
    c for c in metrics_df.columns
    if c not in {"segment", "variant"} and pd.api.types.is_numeric_dtype(metrics_df[c])
]

deltas_rows = []

for seg in SEGMENTS:
    # L2 - BASE
    row = {"segment": seg, "delta_variant": "L2_minus_BASE"}
    for c in NUM_COLS:
        row[f"delta_{c}"] = float(l2df.loc[seg, c] - base.loc[seg, c])
    deltas_rows.append(row)

    # BASE - L2
    row = {"segment": seg, "delta_variant": "BASE_minus_L2"}
    for c in NUM_COLS:
        row[f"delta_{c}"] = float(base.loc[seg, c] - l2df.loc[seg, c])
    deltas_rows.append(row)

    # L1 - BASE
    row = {"segment": seg, "delta_variant": "L1_minus_BASE"}
    for c in NUM_COLS:
        row[f"delta_{c}"] = float(l1df.loc[seg, c] - base.loc[seg, c])
    deltas_rows.append(row)

    # BASE - L1
    row = {"segment": seg, "delta_variant": "BASE_minus_L1"}
    for c in NUM_COLS:
        row[f"delta_{c}"] = float(base.loc[seg, c] - l1df.loc[seg, c])
    deltas_rows.append(row)

deltas_df = pd.DataFrame(deltas_rows).sort_values(["segment", "delta_variant"]).reset_index(drop=True)
deltas_path = os.path.join(OUT_DIR, "deltas_metrics.csv")
deltas_df.to_csv(deltas_path, index=False)
print(" -", deltas_path)

# Coefficient deltas
coef_wide = coef_df_all.pivot_table(
    index=["segment", "param"],
    columns="variant",
    values="coef",
    aggfunc="first"
).reset_index()

if "L2" in coef_wide.columns and "BASE" in coef_wide.columns:
    coef_wide["delta_L2_minus_BASE"] = coef_wide["L2"] - coef_wide["BASE"]
    coef_wide["delta_BASE_minus_L2"] = coef_wide["BASE"] - coef_wide["L2"]

if "L1" in coef_wide.columns and "BASE" in coef_wide.columns:
    coef_wide["delta_L1_minus_BASE"] = coef_wide["L1"] - coef_wide["BASE"]
    coef_wide["delta_BASE_minus_L1"] = coef_wide["BASE"] - coef_wide["L1"]

coef_deltas_path = os.path.join(OUT_DIR, "deltas_coefs.csv")
coef_wide.to_csv(coef_deltas_path, index=False)
print(" -", coef_deltas_path)

# ============================================================
# PLOTS
# ============================================================
metrics_to_plot = [
    "train_NLL",
    "oos_NLL",
    "NLL_full",
    "train_MRR",
    "oos_MRR",
    "MRR_full",
    "train_McFadden_R2",
    "oos_McFadden_R2",
    "McFadden_R2_full",
    "share_spearman_full",
    "share_JS_full",
    "beta_l1",
    "beta_l2",
    "beta_share_small",
]

for metric in metrics_to_plot:
    if metric in metrics_df.columns:
        out_plot = os.path.join(OUT_DIR, f"plot_{metric}.png")
        make_metric_plot(metrics_df, metric, out_plot)

print("\n[DONE] Outputs in:", OUT_DIR)

# R7 Spatial Autocorrelation

In [ ]:
# ============================================================
# R7: Spatial spillovers + residual spatial autocorrelation (Moran's I)
# CORRECTED / CONSISTENT VERSION
#
# Setting: byPerson, uniform sampling, EMU impedance, global Z
#
# Model A (BASE):      V = alpha*EMU + X*beta
# Model B (SPILLOVER): V = alpha*EMU + X*beta + (W X)*theta
#
# W: contiguity neighbors (polygon intersection / touches), row-normalized
#
# Evaluation:
# - sampled metrics (train/oos) with K=1000
# - FULL choice-set metrics on fixed OOS subset (for shares + residuals)
# - Moran's I of residual shares: r_j = S_obs_j - S_pred_j
# - diagnostics of spillover-effect magnitude
#
# Outputs: ModelRuns/SpatialSpillover_R7_globalZ/
#   metrics_base_vs_spillover.csv
#   coef_base_vs_spillover.csv
#   residual_moransI.csv
#   spillover_effect_diagnostics.csv
#   deltas_spillover_minus_base.csv
#   deltas_moransI_spillover_minus_base.csv
#   deltas_coefs_spillover_minus_base.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

import torch
import openmatrix as omx


# -----------------------------
# CONFIG
# -----------------------------
TRIPS_DIR = "Trips/byPerson"
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME = "utility_emu"
MAP_NAME  = "NO"

BASE_FEATURE_REQUEST = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

# Sampling
K = 1000

# Deterministic seeds
SEED_BASE = 123
SEED_TRAIN_DESTSET = 12345
SEED_OOS_DESTSET   = 54321
SEED_TORCH_BASE    = 22222
SEED_FULL_EVAL     = 77777
SEED_MORAN         = 88888

# Torch
DEVICE = "cpu"
DTYPE  = torch.float32

# Optimization
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# Full evaluation subset
FULL_EVAL_MAX = {"YS": 100000, "OS": 100000, "YL": 100000, "OL": 100000}
FULL_EVAL_CHUNK = 256

# Moran permutations (0 to skip permutation p-value)
MORAN_PERM = 199

OUT_DIR = "ModelRuns/SpatialSpillover_R7_globalZ"
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# IO helpers
# ============================================================
def load_segment_csv(seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(TRIPS_DIR, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    need = [ORIG_COL, DEST_COL, WP_COL]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"{path}: missing columns {miss}")

    df = df.dropna(subset=need).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=need).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy().reset_index(drop=True)
    return df


def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")
    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {omx_path.name}:{mat_name} into RAM... shape={M.shape} float32")
    mat = np.asarray(M[:, :], dtype=np.float32)
    f.close()
    return mat, zone_to_idx, idx_to_zone


# ============================================================
# Feature helpers
# ============================================================
def resolve_feature_cols(tz_cols: list[str], requested: list[str]) -> list[str]:
    """
    Allows fallback between base name and _log1p version.
    Useful if one layer stores F3_outdoor_LUmix and another F3_outdoor_LUmix_log1p.
    """
    tz_set = set(tz_cols)
    out = []
    missing = []

    for c in requested:
        if c in tz_set:
            out.append(c)
            continue

        if (c + "_log1p") in tz_set:
            out.append(c + "_log1p")
            continue

        if c.endswith("_log1p"):
            base = c.replace("_log1p", "")
            if base in tz_set:
                out.append(base)
                continue

        missing.append(c)

    if missing:
        raise ValueError(f"Missing TZ columns: {missing}")
    return out


def global_z_standardize(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)
    return tz_feat, z_cols, mu, sd


# ============================================================
# Spatial W (contiguity) + lag features
# ============================================================
def build_contiguity_W(tz_gdf: gpd.GeoDataFrame, id_col="npvm_id"):
    """
    Returns:
      ids: zone ids in the same order
      neighbors: list-of-lists of neighbor positions
    Neighbors are defined by polygon intersection / touching.
    """
    g = tz_gdf[[id_col, "geometry"]].copy()
    g = g[g["geometry"].notnull()].copy().reset_index(drop=True)

    ids = g[id_col].to_numpy(dtype=int)
    geoms = g.geometry.values
    sidx = g.sindex

    neighbors = [[] for _ in range(len(g))]
    for i, geom in enumerate(geoms):
        cand_idx = list(sidx.intersection(geom.bounds))
        cand_idx = [j for j in cand_idx if j != i]
        for j in cand_idx:
            if geoms[j].intersects(geom):
                neighbors[i].append(j)

    for i in range(len(neighbors)):
        for j in neighbors[i]:
            if i not in neighbors[j]:
                neighbors[j].append(i)
        neighbors[i] = sorted(set(neighbors[i]))

    return ids, neighbors


def compute_lag_features(neighbors: list[list[int]], Xz: np.ndarray):
    """
    WX = row-normalized spatial lag = mean of neighbors' feature values.
    """
    n, P = Xz.shape
    WX = np.zeros((n, P), dtype=np.float32)
    for i in range(n):
        nb = neighbors[i]
        WX[i] = Xz[nb].mean(axis=0) if len(nb) else 0.0
    return WX


# ============================================================
# Sampling + design (fixed dest-sets)
# ============================================================
def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray, k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts
    return dest_set


def make_destset(df: pd.DataFrame, all_dests: np.ndarray, k: int, seed: int):
    rng = np.random.default_rng(seed)
    chosen = df[DEST_COL].to_numpy(dtype=int)
    return sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)


def build_design_with_fixed_destset(df, tz_feat, z_cols, wz_cols, emu_mat, zone_to_idx, dest_set):
    orig = df[ORIG_COL].to_numpy(dtype=int)
    w    = df[WP_COL].to_numpy(dtype=np.float64)

    feat_map  = tz_feat.set_index("npvm_id")[z_cols]
    wfeat_map = tz_feat.set_index("npvm_id")[wz_cols]

    N = len(df)
    J = dest_set.shape[1]
    P = len(z_cols)

    X  = np.empty((N, J, P), dtype=np.float32)
    WX = np.empty((N, J, P), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(z_cols):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, J)
    for p, col in enumerate(wz_cols):
        WX[:, :, p] = wfeat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, J)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, J)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy().reset_index(drop=True)
    X2 = X[good]
    WX2 = WX[good]
    w2 = w[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(WX2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2
    )


# ============================================================
# Train models
# ============================================================
def train_base(emu_set, X_set, y, w, epochs=30, lr=0.03, batch_size=2048, weight_decay=0.0, torch_seed=0):
    torch.manual_seed(int(torch_seed))
    N, J = emu_set.shape
    P = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(P, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s+batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        nll = tot_num / (tot_w + 1e-9)
        if ep % 10 == 0 or ep == 1:
            print(f"      epoch {ep:4d} | weighted NLL={nll:.6f}")

        if nll < best:
            best = nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state


def train_spillover(emu_set, X_set, WX_set, y, w, epochs=30, lr=0.03, batch_size=2048, weight_decay=0.0, torch_seed=0):
    torch.manual_seed(int(torch_seed))
    N, J = emu_set.shape
    P = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(P, device=DEVICE, dtype=DTYPE, requires_grad=True)
    theta = torch.zeros(P, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta, theta], lr=lr, weight_decay=weight_decay)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s+batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            WX_b  = WX_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = (
                alpha * emu_b
                + torch.einsum("bjp,p->bj", X_b, beta)
                + torch.einsum("bjp,p->bj", WX_b, theta)
            )
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        nll = tot_num / (tot_w + 1e-9)
        if ep % 10 == 0 or ep == 1:
            print(f"      epoch {ep:4d} | weighted NLL={nll:.6f}")

        if nll < best:
            best = nll
            best_state = (
                float(alpha.detach().cpu()),
                beta.detach().cpu().numpy().copy(),
                theta.detach().cpu().numpy().copy(),
            )

    return best_state


# ============================================================
# Sampled metrics
# ============================================================
def eval_sampled_metrics_fromV(V, w):
    with torch.no_grad():
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        chosen_logp = logP[:, 0]

        sum_w = float(w.sum().cpu())
        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        top1   = float((w * (rank <= 1).float()).sum().cpu() / (sum_w + 1e-9))
        top10  = float((w * (rank <= 10).float()).sum().cpu() / (sum_w + 1e-9))
        top50  = float((w * (rank <= 50).float()).sum().cpu() / (sum_w + 1e-9))
        top100 = float((w * (rank <= 100).float()).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        ll_model = float((w * chosen_logp).sum().cpu())
        J = V.shape[1]
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    return {
        "NLL": nll,
        "MRR": mrr,
        "Top1": top1,
        "Top10": top10,
        "Top50": top50,
        "Top100": top100,
        "McFadden_R2": float(r2),
        "rank_median": rank_median,
        "J": int(J),
    }


def eval_sampled_base(emu_set, X_set, w, alpha, beta):
    beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
    alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)
    V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
    return eval_sampled_metrics_fromV(V, w)


def eval_sampled_spill(emu_set, X_set, WX_set, w, alpha, beta, theta):
    beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
    theta_t = torch.tensor(theta, dtype=DTYPE, device=DEVICE)
    alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)
    V = (
        alpha_t * emu_set
        + torch.einsum("bjp,p->bj", X_set, beta_t)
        + torch.einsum("bjp,p->bj", WX_set, theta_t)
    )
    return eval_sampled_metrics_fromV(V, w)


# ============================================================
# FULL eval (shares) + Moran
# ============================================================
def make_fixed_oos_subset(oos_df: pd.DataFrame, zone_to_idx: dict, max_n: int | None, seed: int):
    df = oos_df.copy().reset_index(drop=True)
    w = df[WP_COL].to_numpy(dtype=np.float64)
    o = df[ORIG_COL].to_numpy(dtype=int)
    d = df[DEST_COL].to_numpy(dtype=int)

    o_ok = np.fromiter((zone_to_idx.get(int(oi), -1) >= 0 for oi in o), dtype=bool, count=len(o))
    d_ok = np.fromiter((zone_to_idx.get(int(di), -1) >= 0 for di in d), dtype=bool, count=len(d))
    ok = o_ok & d_ok & np.isfinite(w) & (w > 0)
    df = df.loc[ok].copy().reset_index(drop=True)

    if (max_n is not None) and (len(df) > max_n):
        df = df.sample(n=max_n, random_state=seed).copy().reset_index(drop=True)
    return df


def full_eval_shares(
    df_oos_fixed,
    tz_feat,
    emu_mat,
    zone_to_idx,
    idx_to_zone,
    alpha,
    Aj_by_zoneid,
    chunk=256
):
    df = df_oos_fixed
    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)
    df = df.loc[good].copy().reset_index(drop=True)
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}, None, None, None

    Msize = emu_mat.shape[0]
    all_zone_ids = tz_feat["npvm_id"].to_numpy(dtype=int)

    Aj_by_idx = np.zeros(Msize, dtype=np.float32)
    for zid in all_zone_ids:
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_idx[jidx] = float(Aj_by_zoneid.get(int(zid), 0.0))

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_pos = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )
    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)

    nll_num = 0.0
    ll_model = 0.0
    mrr_num = 0.0
    top1_num = top10_num = top50_num = top100_num = 0.0
    ranks = np.empty(len(df), dtype=np.int64)

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_idx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_idx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_idx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_idx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_pos.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

        nll_num += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        top1_num   += wn * (1.0 if rank <= 1 else 0.0)
        top10_num  += wn * (1.0 if rank <= 10 else 0.0)
        top50_num  += wn * (1.0 if rank <= 50 else 0.0)
        top100_num += wn * (1.0 if rank <= 100 else 0.0)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    top1_full   = float(top1_num   / (sum_w + 1e-9))
    top10_full  = float(top10_num  / (sum_w + 1e-9))
    top50_full  = float(top50_num  / (sum_w + 1e-9))
    top100_full = float(top100_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)

    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    metrics = {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "Top1_full": top1_full,
        "Top10_full": top10_full,
        "Top50_full": top50_full,
        "Top100_full": top100_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    return metrics, S_obs, S_pred, all_zone_ids


def morans_I(residuals: np.ndarray, neighbors: list[list[int]]):
    x = residuals.astype(np.float64)
    x = x - x.mean()
    n = len(x)
    den = np.sum(x * x) + 1e-300

    num = 0.0
    wsum = 0.0
    for i in range(n):
        nb = neighbors[i]
        if len(nb) == 0:
            continue
        xi = x[i]
        wij = 1.0 / len(nb)
        for j in nb:
            num += xi * x[j] * wij
        wsum += 1.0

    I = (n / (wsum + 1e-300)) * (num / den)
    return float(I)


def morans_pvalue_perm(residuals: np.ndarray, neighbors: list[list[int]], nperm=199, seed=1234):
    rng = np.random.default_rng(seed)
    I_obs = morans_I(residuals, neighbors)
    more_extreme = 0
    for _ in range(nperm):
        perm = rng.permutation(residuals)
        I_p = morans_I(perm, neighbors)
        if abs(I_p) >= abs(I_obs):
            more_extreme += 1
    p = (more_extreme + 1) / (nperm + 1)
    return I_obs, float(p)


# ============================================================
# Spillover-effect diagnostics
# ============================================================
def spillover_effect_diagnostics(tz_feat: pd.DataFrame, z_cols: list[str], wz_cols: list[str],
                                 beta: np.ndarray, theta: np.ndarray):
    Xz_all  = tz_feat[z_cols].to_numpy(dtype=np.float32)
    WXz_all = tz_feat[wz_cols].to_numpy(dtype=np.float32)

    direct = Xz_all @ beta.astype(np.float32)
    spill  = WXz_all @ theta.astype(np.float32)

    direct_sd = float(np.std(direct))
    spill_sd = float(np.std(spill))
    corr = float(pd.Series(direct).corr(pd.Series(spill), method="pearson")) if len(direct) > 2 else np.nan

    return {
        "theta_l1": float(np.sum(np.abs(theta))),
        "theta_l2": float(np.sqrt(np.sum(theta * theta))),
        "spill_mean_abs": float(np.mean(np.abs(spill))),
        "spill_sd": spill_sd,
        "direct_sd": direct_sd,
        "spill_to_direct_sd_ratio": float(spill_sd / (direct_sd + 1e-12)),
        "corr_direct_spill": corr,
    }


# ============================================================
# MAIN
# ============================================================
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
tz["npvm_id"] = pd.to_numeric(tz["npvm_id"], errors="coerce")
tz = tz.dropna(subset=["npvm_id"]).copy()
tz["npvm_id"] = tz["npvm_id"].astype(int)

feat_cols = resolve_feature_cols(list(tz.columns), BASE_FEATURE_REQUEST)
tz_feat = tz[["npvm_id"] + feat_cols + ["geometry"]].copy()

# Global z
tz_feat, z_cols, mu_global, sd_global = global_z_standardize(tz_feat, feat_cols, suffix="_z")

# EMU matrix
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

# W matrix
print("[W] building contiguity neighbors (intersects/touches)...")
tz_ids_W, neighbors_W = build_contiguity_W(tz_feat[["npvm_id", "geometry"]].copy(), id_col="npvm_id")
mean_deg = float(np.mean([len(n) for n in neighbors_W])) if len(neighbors_W) else 0.0
print(f"[W] zones in W: {len(tz_ids_W):,} | mean degree={mean_deg:.2f}")

posW = {int(z): i for i, z in enumerate(tz_ids_W)}

# compute WX on W-order then attach to tz_feat
tz_feat_idx = tz_feat.set_index("npvm_id")
XzW = np.zeros((len(tz_ids_W), len(z_cols)), dtype=np.float32)
for i, zid in enumerate(tz_ids_W):
    if int(zid) in tz_feat_idx.index:
        XzW[i, :] = tz_feat_idx.loc[int(zid), z_cols].to_numpy(dtype=np.float32)
    else:
        XzW[i, :] = 0.0

WXzW = compute_lag_features(neighbors_W, XzW)

wz_cols = []
vals_by_wcol = {}
for p, zc in enumerate(z_cols):
    wcol = "W_" + zc
    wz_cols.append(wcol)
    vals_by_wcol[wcol] = WXzW[:, p].astype(np.float32)

for wcol in wz_cols:
    arr = np.zeros(len(tz_feat), dtype=np.float32)
    for r_i, zid in enumerate(tz_feat["npvm_id"].to_numpy(dtype=int)):
        j = posW.get(int(zid), None)
        arr[r_i] = float(vals_by_wcol[wcol][j]) if j is not None else 0.0
    tz_feat[wcol] = arr

ALL_DESTS = tz_feat["npvm_id"].to_numpy(dtype=int)

rows_metrics = []
rows_coefs = []
rows_moran = []
rows_diag = []

for seg in SEGMENTS:
    print("\n" + "=" * 110)
    print(f"SEGMENT {seg} | K={K} | byPerson | EMU | BASE vs SPILLOVER (WX) + Moran residuals | globalZ")
    print("=" * 110)

    train_df = load_segment_csv(seg, "train")
    oos_df   = load_segment_csv(seg, "oos")

    # fixed dest-sets across BASE/SPILLOVER
    destset_tr = make_destset(train_df, ALL_DESTS, k=K, seed=SEED_TRAIN_DESTSET + 1000 * SEG_SEED[seg])
    destset_te = make_destset(oos_df,   ALL_DESTS, k=K, seed=SEED_OOS_DESTSET   + 1000 * SEG_SEED[seg])

    emu_tr, X_tr, WX_tr, y_tr, w_tr, train_df2 = build_design_with_fixed_destset(
        train_df, tz_feat[["npvm_id"] + z_cols + wz_cols].copy(), z_cols, wz_cols,
        emu_mat, zone_to_idx, destset_tr
    )
    emu_te, X_te, WX_te, y_te, w_te, oos_df2 = build_design_with_fixed_destset(
        oos_df, tz_feat[["npvm_id"] + z_cols + wz_cols].copy(), z_cols, wz_cols,
        emu_mat, zone_to_idx, destset_te
    )

    print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
    print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

    # fixed full-OOS subset across models
    oos_fixed = make_fixed_oos_subset(
        oos_df2,
        zone_to_idx,
        max_n=FULL_EVAL_MAX.get(seg, 20000),
        seed=SEED_FULL_EVAL + 1000 * SEG_SEED[seg]
    )

    # ---------------- BASE ----------------
    print("\n--- [BASE] train ---")
    alpha_b, beta_b = train_base(
        emu_tr, X_tr, y_tr, w_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
        torch_seed=SEED_TORCH_BASE + 1000 * SEG_SEED[seg] + 10
    )
    met_tr_b = eval_sampled_base(emu_tr, X_tr, w_tr, alpha_b, beta_b)
    met_te_b = eval_sampled_base(emu_te, X_te, w_te, alpha_b, beta_b)

    Xz_all = tz_feat[z_cols].to_numpy(dtype=np.float32)
    Aj_base = Xz_all @ beta_b.astype(np.float32)
    Aj_by_zone_base = {int(z): float(a) for z, a in zip(ALL_DESTS, Aj_base)}

    full_b, Sobs_b, Spred_b, ids_full = full_eval_shares(
        oos_fixed,
        tz_feat[["npvm_id"] + z_cols + wz_cols].copy(),
        emu_mat, zone_to_idx, idx_to_zone,
        float(alpha_b), Aj_by_zone_base,
        chunk=FULL_EVAL_CHUNK
    )

    # ---------------- SPILLOVER ----------------
    print("\n--- [SPILLOVER] train ---")
    alpha_s, beta_s, theta_s = train_spillover(
        emu_tr, X_tr, WX_tr, y_tr, w_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
        torch_seed=SEED_TORCH_BASE + 1000 * SEG_SEED[seg] + 20
    )
    met_tr_s = eval_sampled_spill(emu_tr, X_tr, WX_tr, w_tr, alpha_s, beta_s, theta_s)
    met_te_s = eval_sampled_spill(emu_te, X_te, WX_te, w_te, alpha_s, beta_s, theta_s)

    WXz_all = tz_feat[wz_cols].to_numpy(dtype=np.float32)
    Aj_sp = Xz_all @ beta_s.astype(np.float32) + WXz_all @ theta_s.astype(np.float32)
    Aj_by_zone_sp = {int(z): float(a) for z, a in zip(ALL_DESTS, Aj_sp)}

    full_s, Sobs_s, Spred_s, ids_full_s = full_eval_shares(
        oos_fixed,
        tz_feat[["npvm_id"] + z_cols + wz_cols].copy(),
        emu_mat, zone_to_idx, idx_to_zone,
        float(alpha_s), Aj_by_zone_sp,
        chunk=FULL_EVAL_CHUNK
    )

    # ---------------- Moran's I on residual shares ----------------
    def residual_to_Worder(ids_full_local, Sobs, Spred):
        r = (Sobs - Spred).astype(np.float64)
        id_to_r = {int(z): float(rv) for z, rv in zip(ids_full_local, r)}
        rW = np.zeros(len(tz_ids_W), dtype=np.float64)
        for i, zid in enumerate(tz_ids_W):
            rW[i] = id_to_r.get(int(zid), 0.0)
        return rW

    rW_b = residual_to_Worder(ids_full, Sobs_b, Spred_b)
    rW_s = residual_to_Worder(ids_full_s, Sobs_s, Spred_s)

    if MORAN_PERM and MORAN_PERM > 0:
        I_b, p_b = morans_pvalue_perm(rW_b, neighbors_W, nperm=MORAN_PERM, seed=SEED_MORAN + 1000 * SEG_SEED[seg] + 10)
        I_s, p_s = morans_pvalue_perm(rW_s, neighbors_W, nperm=MORAN_PERM, seed=SEED_MORAN + 1000 * SEG_SEED[seg] + 20)
    else:
        I_b = morans_I(rW_b, neighbors_W)
        I_s = morans_I(rW_s, neighbors_W)
        p_b = np.nan
        p_s = np.nan

    rows_moran.append({"segment": seg, "model": "BASE", "Moran_I_resid": I_b, "p_perm": p_b})
    rows_moran.append({"segment": seg, "model": "SPILLOVER", "Moran_I_resid": I_s, "p_perm": p_s})

    # ---------------- Effect diagnostics ----------------
    diag = spillover_effect_diagnostics(tz_feat, z_cols, wz_cols, beta_s, theta_s)
    diag_row = {"segment": seg, **diag}
    rows_diag.append(diag_row)

    # ---------------- Pack metrics rows ----------------
    def pack(prefix, met):
        return {
            f"{prefix}_NLL": met["NLL"],
            f"{prefix}_MRR": met["MRR"],
            f"{prefix}_Top1": met["Top1"],
            f"{prefix}_Top10": met["Top10"],
            f"{prefix}_Top50": met["Top50"],
            f"{prefix}_Top100": met["Top100"],
            f"{prefix}_McFadden_R2": met["McFadden_R2"],
            f"{prefix}_rank_median": met["rank_median"],
            f"{prefix}_J": met["J"],
        }

    row_base = {
        "segment": seg,
        "K": K,
        "model": "BASE",
        "alpha": float(alpha_b),
        **pack("train", met_tr_b),
        **pack("oos", met_te_b),
        **full_b,
    }
    row_sp = {
        "segment": seg,
        "K": K,
        "model": "SPILLOVER",
        "alpha": float(alpha_s),
        **pack("train", met_tr_s),
        **pack("oos", met_te_s),
        **full_s,
        **diag,
    }
    rows_metrics += [row_base, row_sp]

    # ---------------- Coefs table ----------------
    coef_base = pd.DataFrame({
        "segment": seg,
        "model": "BASE",
        "param": ["alpha"] + z_cols,
        "coef": np.concatenate([[alpha_b], beta_b]).astype(np.float64),
    })
    coef_sp = pd.DataFrame({
        "segment": seg,
        "model": "SPILLOVER",
        "param": ["alpha"] + z_cols + ["theta__" + c for c in z_cols],
        "coef": np.concatenate([[alpha_s], beta_s, theta_s]).astype(np.float64),
    })
    rows_coefs.append(coef_base)
    rows_coefs.append(coef_sp)

# -----------------------------
# Save outputs
# -----------------------------
metrics_df = pd.DataFrame(rows_metrics).sort_values(["segment", "model"]).reset_index(drop=True)
coef_df = pd.concat(rows_coefs, ignore_index=True)
moran_df = pd.DataFrame(rows_moran).sort_values(["segment", "model"]).reset_index(drop=True)
diag_df = pd.DataFrame(rows_diag).sort_values(["segment"]).reset_index(drop=True)

metrics_path = os.path.join(OUT_DIR, "metrics_base_vs_spillover.csv")
coef_path    = os.path.join(OUT_DIR, "coef_base_vs_spillover.csv")
moran_path   = os.path.join(OUT_DIR, "residual_moransI.csv")
diag_path    = os.path.join(OUT_DIR, "spillover_effect_diagnostics.csv")

metrics_df.to_csv(metrics_path, index=False)
coef_df.to_csv(coef_path, index=False)
moran_df.to_csv(moran_path, index=False)
diag_df.to_csv(diag_path, index=False)

# -----------------------------
# Deltas: SPILLOVER - BASE
# -----------------------------
base = metrics_df[metrics_df["model"] == "BASE"].copy()
sp   = metrics_df[metrics_df["model"] == "SPILLOVER"].copy()
merged = sp.merge(base, on=["segment", "K"], suffixes=("_sp", "_base"), how="inner")

delta_rows = []
for _, r in merged.iterrows():
    d = {"segment": r["segment"], "K": int(r["K"])}
    for col in metrics_df.columns:
        if col in ["segment", "K", "model"]:
            continue
        csp = col + "_sp"
        cb  = col + "_base"
        if (csp in merged.columns) and (cb in merged.columns) and pd.notna(r[csp]) and pd.notna(r[cb]):
            d["delta_" + col] = float(r[csp] - r[cb])
    delta_rows.append(d)

deltas_df = pd.DataFrame(delta_rows).sort_values(["segment"]).reset_index(drop=True)
deltas_path = os.path.join(OUT_DIR, "deltas_spillover_minus_base.csv")
deltas_df.to_csv(deltas_path, index=False)

# -----------------------------
# Moran deltas
# -----------------------------
moran_base = moran_df[moran_df["model"] == "BASE"].set_index("segment")
moran_sp   = moran_df[moran_df["model"] == "SPILLOVER"].set_index("segment")

moran_delta_rows = []
for seg in SEGMENTS:
    row = {
        "segment": seg,
        "Moran_I_base": float(moran_base.loc[seg, "Moran_I_resid"]),
        "Moran_I_spillover": float(moran_sp.loc[seg, "Moran_I_resid"]),
        "delta_Moran_I_spillover_minus_base": float(moran_sp.loc[seg, "Moran_I_resid"] - moran_base.loc[seg, "Moran_I_resid"]),
        "p_perm_base": float(moran_base.loc[seg, "p_perm"]) if pd.notna(moran_base.loc[seg, "p_perm"]) else np.nan,
        "p_perm_spillover": float(moran_sp.loc[seg, "p_perm"]) if pd.notna(moran_sp.loc[seg, "p_perm"]) else np.nan,
    }
    moran_delta_rows.append(row)

moran_delta_df = pd.DataFrame(moran_delta_rows).sort_values(["segment"]).reset_index(drop=True)
moran_delta_path = os.path.join(OUT_DIR, "deltas_moransI_spillover_minus_base.csv")
moran_delta_df.to_csv(moran_delta_path, index=False)

# -----------------------------
# Coefficient deltas
# -----------------------------
coef_wide = coef_df.pivot_table(
    index=["segment", "param"],
    columns="model",
    values="coef",
    aggfunc="first"
).reset_index()

# common params between BASE and SPILLOVER: alpha + beta params
if "BASE" in coef_wide.columns and "SPILLOVER" in coef_wide.columns:
    coef_wide["delta_spillover_minus_base"] = coef_wide["SPILLOVER"] - coef_wide["BASE"]

coef_delta_path = os.path.join(OUT_DIR, "deltas_coefs_spillover_minus_base.csv")
coef_wide.to_csv(coef_delta_path, index=False)

print("\n[OK] wrote:")
print(" -", metrics_path)
print(" -", coef_path)
print(" -", moran_path)
print(" -", diag_path)
print(" -", deltas_path)
print(" -", moran_delta_path)
print(" -", coef_delta_path)
print("[DONE]")

In [ ]:
# ============================================================
# BASELINE ESTIMATION + FULL REPORT + VCOV + COEFFICIENT TESTS
# (withLIE, byPerson, log1p, global z, EMU)
#
# UNWEIGHTED VERSION
#
# DIFFERENCE FROM WEIGHTED VERSION:
#   - WP is NOT used in estimation
#   - WP is NOT used in evaluation
#   - WP is only kept in some exported trip-level files if present in input,
#     but it does not affect any metric, share, distance summary, Hessian, or test
#
# OUTPUT FOLDER:
#   ModelRuns/NoWeighted
# ============================================================

import os
import math
import random
import itertools
import numpy as np
import pandas as pd
import geopandas as gpd
import tables as tb

import torch
import openmatrix as omx
from pathlib import Path

import matplotlib.pyplot as plt


# -----------------------------
# CONFIG
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"          # kept only if available in raw input / export
DIST_COL  = "dist_km"

TRIPS_DIR = "Trips/byPerson"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

UTIL_OMX      = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME  = "utility_emu"

DIST_OMX      = READY_DIR / "distance_avg_2023_ready.omx"
DIST_MAT_NAME = "avg_distances_ready"

MAP_NAME   = "NO"

OUT_DIR = "ModelRuns/NoWeighted"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DIST_OOS_DIR = os.path.join(OUT_DIR, "distance_oos_triplevel")
os.makedirs(DIST_OOS_DIR, exist_ok=True)

RESULT_DIR = os.path.join(OUT_DIR, "Result")
os.makedirs(RESULT_DIR, exist_ok=True)

VCOV_DIR = os.path.join(OUT_DIR, "vcov")
os.makedirs(VCOV_DIR, exist_ok=True)

TEST_DIR = os.path.join(OUT_DIR, "coef_tests")
os.makedirs(TEST_DIR, exist_ok=True)

ATTR_CSV_OUT  = os.path.join(RESULT_DIR, "Attractivity.csv")
ATTR_GPKG_OUT = os.path.join(RESULT_DIR, "Attractivity.gpkg")
UTIL_SEG_OMX  = os.path.join(RESULT_DIR, "utilities_by_segment.omx")

K = 1000
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

FULL_EVAL_MAX = {
    "YS": 20000,
    "OS": 8000,
    "YL": 8000,
    "OL": 8000,
}
FULL_EVAL_SEED  = 777
FULL_EVAL_CHUNK = 256

SE_MAX_TRAIN = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
SE_SEED = 2024

BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

TOPKS_SAMPLED = (5, 10)
TOPKS_FULL    = (5, 10)

BASE_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

PARAM_NAMES = ["alpha_emu"] + BASE_COLS


# ============================================================
# Global reproducibility helper
# ============================================================
def set_all_seeds(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Small helpers
# ============================================================
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300); q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def p_value_two_sided_z(z):
    return 2.0 * (1.0 - norm_cdf(abs(float(z))))

def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.1:
        return "."
    return ""

def safe_sqrt(x):
    return float(np.sqrt(max(float(x), 0.0)))


# ============================================================
# IO helpers
# ============================================================
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)

    if WP_COL in df.columns:
        df[WP_COL] = pd.to_numeric(df[WP_COL], errors="coerce")

    if DIST_COL in df.columns:
        df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")

    return df

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


# ============================================================
# Preprocess helpers
# ============================================================
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0]  = c
        dest_set[i, 1:] = alts
    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping.")

    df2 = df.loc[good].copy()
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]
    dest_set2 = dest_set[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        df2,
        dest_set2,
        orig_idx2,
        dest_idx2,
    )


# ============================================================
# Model
# ============================================================
def train_mnl(emu_set, X_set, y, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_n = 0.0, 0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)

            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -ll.sum()

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu())
            tot_n += int(len(b))

        epoch_nll = tot_loss_num / max(tot_n, 1)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | avg NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, alpha, beta, topKs=(5,10), want_ff=True):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        P = torch.exp(logP)

        chosen_logp = logP[:, 0]
        chosen_p    = P[:, 0]

        n = int(V.shape[0])
        nll_sum = float((-(chosen_logp)).sum().cpu())
        nll_avg = nll_sum / max(n, 1)

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((1.0 / rank).mean().cpu())

        out = {
            "NLL_sum": nll_sum,
            "NLL_avg": nll_avg,
            "MRR": mrr,
            "N": int(V.shape[0]),
            "J": int(V.shape[1]),
        }

        for k in topKs:
            out[f"Top{k}"] = float((rank <= k).float().mean().cpu())

        if want_ff:
            out["FF"] = float(chosen_p.mean().cpu())

    return out


# ============================================================
# FULL-choice-set evaluation on an OOS subset
# ============================================================
def full_eval_and_shares_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    dist_mat: np.ndarray | None,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(5,10),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)

    has_dist = (DIST_COL in df.columns) and df[DIST_COL].notna().any()
    obs_dist = df[DIST_COL].to_numpy(dtype=np.float64) if has_dist else None

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0)

    if has_dist:
        good = good & np.isfinite(obs_dist) & (obs_dist >= 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    if has_dist:
        obs_dist = obs_dist[good]

    n_obs = int(len(df))
    if n_obs == 0:
        return {}, None, (None, None), None

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass  = (
        df.groupby(DEST_COL).size()
        .reindex(all_zone_ids, fill_value=0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    ll_model = 0.0
    mrr_num = 0.0
    ff_num = 0.0
    top_num = {k: 0.0 for k in topKs}
    ranks   = np.empty(len(df), dtype=np.int64)

    exp_dist = np.zeros(len(df), dtype=np.float64) if (dist_mat is not None and has_dist) else None

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse
        p_chosen = math.exp(logp_chosen)

        nll_num  += (-logp_chosen)
        ll_model += (logp_chosen)
        ff_num   += p_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += 1.0

        if exp_dist is not None:
            expd = 0.0

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += float(pj)

            if exp_dist is not None:
                dist_blk = dist_mat[o, j0:j1].astype(np.float64)
                expd += float(np.sum(Pblk * dist_blk))

        if exp_dist is not None:
            exp_dist[n] = expd

    nll_full = float(nll_num)
    nll_full_avg = float(nll_num / max(n_obs, 1))
    mrr_full = float(mrr_num / max(n_obs, 1))
    ff_full = float(ff_num / max(n_obs, 1))
    rank_median_full = float(np.median(ranks))

    ll0 = n_obs * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs  = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)

    pear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="pearson")) if len(S_obs) > 2 else np.nan
    spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    mae  = float(np.mean(np.abs(S_obs - S_pred)))
    rmse = float(np.sqrt(np.mean((S_obs - S_pred) ** 2)))
    js   = float(js_div(S_obs, S_pred))

    dist_summary = {}
    if exp_dist is not None:
        obs_mean = float(np.mean(obs_dist))
        pred_mean = float(np.mean(exp_dist))
        dist_summary = {
            "dist_obs_mean_km": obs_mean,
            "dist_pred_mean_km": pred_mean,
            "dist_mean_diff_km": float(pred_mean - obs_mean),
            "dist_obs_p50_km": float(np.quantile(obs_dist, 0.50)),
            "dist_pred_p50_km": float(np.quantile(exp_dist, 0.50)),
            "dist_obs_p90_km": float(np.quantile(obs_dist, 0.90)),
            "dist_pred_p90_km": float(np.quantile(exp_dist, 0.90)),
        }

    out = {
        "NLL_full_sum": nll_full,
        "NLL_full_avg": nll_full_avg,
        "MRR_full": mrr_full,
        "FF_full": ff_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Pearson_share": pear,
        "Spearman_share": spear,
        "MAE_share": mae,
        "RMSE_share": rmse,
        "JS_share": js,
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / max(n_obs, 1))
    out.update(dist_summary)

    trip_distance_df = None
    if exp_dist is not None:
        trip_distance_df = pd.DataFrame({
            "orig_zone": df[ORIG_COL].to_numpy(dtype=int),
            "dest_zone_obs": df[DEST_COL].to_numpy(dtype=int),
            "dist_obs_km": obs_dist.astype(np.float64),
            "dist_exp_km": exp_dist.astype(np.float64),
        })
        if WP_COL in df.columns:
            trip_distance_df["WP"] = pd.to_numeric(df[WP_COL], errors="coerce")

    return out, {
        "tz_ids": all_zone_ids,
        "S_obs": S_obs,
        "S_pred": S_pred,
        "Aj": Aj.astype(np.float64),
        "orig_idx": orig_idx,
    }, (obs_dist, exp_dist), trip_distance_df


# ============================================================
# Hessian-based approximate inference
# ============================================================
def approx_hessian_inference(
    emu_set: torch.Tensor,
    X_set: torch.Tensor,
    y: torch.Tensor,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int,
    seed: int,
    ridge: float = 1e-8,
):
    """
    Returns:
      vcov_sum : approximate covariance matrix of theta_hat
      se       : sqrt(diag(vcov_sum))
      H_sum    : Hessian of unweighted SUM objective
      used_n   : number of observations used
    """
    N = emu_set.shape[0]
    rng = np.random.default_rng(seed)

    if N > max_n:
        idx = rng.choice(N, size=max_n, replace=False)
        idx = torch.tensor(idx, dtype=torch.long, device=DEVICE)
        emu = emu_set[idx]
        X   = X_set[idx]
        yy  = y[idx]
    else:
        emu, X, yy = emu_set, X_set, y

    used_n = int(emu.shape[0])
    Pdim = X.shape[2]

    alpha = torch.tensor([alpha_hat], dtype=DTYPE, device=DEVICE, requires_grad=True)
    beta  = torch.tensor(beta_hat, dtype=DTYPE, device=DEVICE, requires_grad=True)

    def loss_avg(a, b):
        V = a * emu + torch.einsum("bjp,p->bj", X, b)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        ll = logP[torch.arange(V.shape[0], device=DEVICE), yy]
        return -ll.mean()

    L = loss_avg(alpha, beta)
    g = torch.autograd.grad(L, [alpha, beta], create_graph=True)
    g_vec = torch.cat([g[0].reshape(1), g[1].reshape(-1)], dim=0)

    H_avg = torch.zeros((1 + Pdim, 1 + Pdim), dtype=torch.float64, device=DEVICE)
    for i in range(1 + Pdim):
        gi = g_vec[i]
        hi = torch.autograd.grad(gi, [alpha, beta], retain_graph=True)
        hi_vec = torch.cat([hi[0].reshape(1), hi[1].reshape(-1)], dim=0)
        H_avg[i, :] = hi_vec.detach().to(torch.float64)

    H_avg = H_avg.cpu().numpy()
    H_sum = used_n * H_avg

    H_sum = 0.5 * (H_sum + H_sum.T)
    H_sum = H_sum + ridge * np.eye(H_sum.shape[0], dtype=np.float64)

    vcov = np.linalg.pinv(H_sum)
    vcov = 0.5 * (vcov + vcov.T)

    se = np.sqrt(np.clip(np.diag(vcov), 0.0, np.inf))
    return vcov, se, H_sum, used_n


# ============================================================
# Coefficient-difference tests
# ============================================================
def build_within_segment_diff_tests(segment, params, coef_vec, vcov):
    rows = []
    for i in range(len(params)):
        for j in range(i + 1, len(params)):
            p1 = params[i]
            p2 = params[j]
            delta = float(coef_vec[i] - coef_vec[j])
            var_delta = float(vcov[i, i] + vcov[j, j] - 2.0 * vcov[i, j])
            se_delta = safe_sqrt(var_delta)
            z_delta = delta / (se_delta + 1e-12)
            p_delta = p_value_two_sided_z(z_delta)
            rows.append({
                "segment": segment,
                "param_1": p1,
                "param_2": p2,
                "coef_1": float(coef_vec[i]),
                "coef_2": float(coef_vec[j]),
                "delta_1_minus_2": delta,
                "se_delta": se_delta,
                "z_delta": float(z_delta),
                "p_delta": float(p_delta),
                "stars_delta": stars(p_delta),
            })
    return rows

def build_across_segment_sameparam_tests(seg_to_coef, seg_to_vcov, params):
    rows = []
    seg_pairs = list(itertools.combinations(sorted(seg_to_coef.keys()), 2))
    for param_idx, param in enumerate(params):
        for s1, s2 in seg_pairs:
            b1 = float(seg_to_coef[s1][param_idx])
            b2 = float(seg_to_coef[s2][param_idx])

            var_delta = float(seg_to_vcov[s1][param_idx, param_idx] + seg_to_vcov[s2][param_idx, param_idx])
            se_delta = safe_sqrt(var_delta)
            delta = b1 - b2
            z_delta = delta / (se_delta + 1e-12)
            p_delta = p_value_two_sided_z(z_delta)

            rows.append({
                "param": param,
                "segment_1": s1,
                "segment_2": s2,
                "coef_1": b1,
                "coef_2": b2,
                "delta_1_minus_2": float(delta),
                "se_delta": se_delta,
                "z_delta": float(z_delta),
                "p_delta": float(p_delta),
                "stars_delta": stars(p_delta),
                "assumption": "independent_segment_estimates",
            })
    return rows


# ============================================================
# Destination-level summaries and plots
# ============================================================
def incoming_accessibility_logsum(
    tz_ids: np.ndarray,
    tz_pos: dict,
    emu_mat: np.ndarray,
    alpha: float,
    orig_idx: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    chunk: int = 256,
):
    Msize = emu_mat.shape[0]

    counts_by_o = np.zeros(Msize, dtype=np.float64)
    for o in orig_idx:
        counts_by_o[int(o)] += 1.0

    origins = np.where(counts_by_o > 0)[0]
    p_o = counts_by_o[origins]
    p_o = p_o / (p_o.sum() + 1e-12)

    L = np.full(len(tz_ids), np.nan, dtype=np.float64)

    tz_omx = np.array([zone_to_idx.get(int(z), -1) for z in tz_ids], dtype=int)
    good = (tz_omx >= 0)
    tz_omx_good = tz_omx[good]

    good_pos = np.where(good)[0]
    for local_idx, j in enumerate(tz_omx_good):
        vals = alpha * emu_mat[origins, j].astype(np.float64)
        m = np.max(vals)
        s = np.sum(p_o * np.exp(vals - m))
        L[good_pos[local_idx]] = float(m + np.log(s + 1e-300))

    return L

def plot_distance_plausibility(obs_dist, exp_dist, seg, out_png):
    obs_dist = np.asarray(obs_dist, dtype=np.float64)
    exp_dist = np.asarray(exp_dist, dtype=np.float64)

    finite = np.isfinite(obs_dist) & np.isfinite(exp_dist)
    obs_dist = obs_dist[finite]
    exp_dist = exp_dist[finite]

    if len(obs_dist) == 0:
        return

    obs_mean = float(np.mean(obs_dist))
    exp_mean = float(np.mean(exp_dist))

    xmax = float(max(np.max(obs_dist), np.max(exp_dist)))
    bins = np.linspace(0.0, xmax, 51)

    plt.figure()
    plt.hist(obs_dist, bins=bins, density=True, alpha=0.6,
             label=f"Observed dist_km (mean={obs_mean:.2f})")
    plt.hist(exp_dist, bins=bins, density=True, alpha=0.6,
             label=f"Predicted E[distance] (mean={exp_mean:.2f})")
    plt.xlabel("Distance (km)")
    plt.ylabel("Density")
    plt.title(f"Distance plausibility | {seg} | obs mean={obs_mean:.2f}, pred mean={exp_mean:.2f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_attr_vs_access(Aj, L_in, S_pred, seg, out_png):
    plt.figure()
    s = 20 + 4000 * (S_pred / (S_pred.max() + 1e-12))
    plt.scatter(L_in, Aj, s=s, alpha=0.5)
    plt.xlabel("Incoming accessibility logsum (from origins, using EMU)")
    plt.ylabel("Attractivity index A_j = X_j beta (z-scale)")
    plt.title(f"Attractivity vs incoming accessibility | {seg}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_share_maps(tz_gdf, col, seg, out_png, title):
    plt.figure()
    ax = tz_gdf.plot(column=col, legend=True)
    ax.set_axis_off()
    plt.title(f"{title} | {seg}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


# ============================================================
# MAIN
# ============================================================
set_all_seeds(BASE_SEED)

print("[LOAD] TZ features...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
if "npvm_id" not in tz.columns:
    raise ValueError("TZ layer missing 'npvm_id'")
tz["npvm_id"] = pd.to_numeric(tz["npvm_id"], errors="coerce")
tz = tz.dropna(subset=["npvm_id"]).copy()
tz["npvm_id"] = tz["npvm_id"].astype(int)

need_cols = ["npvm_id"] + BASE_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

print("[PREP] Global z-standardize destination variables on FULL TZ universe...")
tz_feat = tz.copy()
tz_feat, Z_COLS, MU_ALL, SD_ALL = standardize_all_zones(tz_feat, BASE_COLS, suffix="_z")

print("[LOAD] EMU matrix + mapping...")
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

print("[LOAD] Average distance matrix...")
dist_mat, zone_to_idx_dist, idx_to_zone_dist = load_matrix_and_mapping(DIST_OMX, DIST_MAT_NAME, MAP_NAME)
if len(idx_to_zone_dist) != len(idx_to_zone) or not np.all(idx_to_zone_dist == idx_to_zone):
    print("[WARN] Distance OMX mapping differs from utilities OMX mapping. Distance plausibility may be misaligned.")

rows_sampled = []
rows_full    = []
rows_coef    = []

seg_alpha = {}
seg_beta  = {}
seg_Aj_tz = {}
seg_Aj_by_omx = {}

seg_vcov = {}
seg_se   = {}
seg_coef_vec = {}
seg_hsum = {}
seg_inference_meta = {}

tz_ids_master = np.array(sorted(tz_feat["npvm_id"].astype(int).unique().tolist()), dtype=int)

GPKG_OUT = os.path.join(OUT_DIR, "tz_outputs.gpkg")
if os.path.exists(GPKG_OUT):
    os.remove(GPKG_OUT)

for seg in SEGMENTS:
    print("\n" + "#" * 120)
    print(f"# SEGMENT = {seg} | K={K} | epochs={EPOCHS}")
    print("#" * 120)

    set_all_seeds(BASE_SEED + 1000 * SEG_SEED[seg])

    train_df = load_segment_csv(TRIPS_DIR, seg, "train")
    oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

    seed_tr = BASE_SEED + 1000 * SEG_SEED[seg] + 10
    seed_te = BASE_SEED + 1000 * SEG_SEED[seg] + 20
    seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg]

    emu_tr, X_tr, y_tr, train_df2, destset_tr, origidx_tr, destidx_tr = build_design(
        train_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_tr
    )
    emu_te, X_te, y_te, oos_df2, destset_te, origidx_te, destidx_te = build_design(
        oos_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_te
    )

    print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")

    alpha_hat, beta_hat = train_mnl(
        emu_tr, X_tr, y_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
    )

    seg_alpha[seg] = float(alpha_hat)
    seg_beta[seg]  = beta_hat.copy()

    coef_vec = np.concatenate([[float(alpha_hat)], beta_hat.astype(np.float64)])
    seg_coef_vec[seg] = coef_vec.copy()

    Xz_all_master = tz_feat.set_index("npvm_id").loc[tz_ids_master, Z_COLS].to_numpy(dtype=np.float32)
    Aj_master = Xz_all_master @ beta_hat.astype(np.float32)
    seg_Aj_tz[seg] = Aj_master.astype(np.float64)

    Msize = emu_mat.shape[0]
    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(tz_ids_master, Aj_master):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)
    seg_Aj_by_omx[seg] = Aj_by_omxidx

    met_tr = eval_sampled_metrics(emu_tr, X_tr, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)
    met_te = eval_sampled_metrics(emu_te, X_te, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)

    rows_sampled.append({
        "segment": seg,
        "split": "train",
        "alpha_emu": float(alpha_hat),
        "K": K,
        **met_tr,
    })
    rows_sampled.append({
        "segment": seg,
        "split": "oos",
        "alpha_emu": float(alpha_hat),
        "K": K,
        **met_te,
    })

    print("[SAMPLED OOS ]",
          f"NLL_sum={met_te['NLL_sum']:.2f} NLL_avg={met_te['NLL_avg']:.4f} "
          f"MRR={met_te['MRR']:.4f} Top5={met_te['Top5']:.3f} Top10={met_te['Top10']:.3f} FF={met_te['FF']:.4f}")

    max_n = FULL_EVAL_MAX.get(seg, None)
    full_out, share_pack, dist_pack, trip_distance_df = full_eval_and_shares_oos_subset(
        oos_df=oos_df2,
        tz_feat=tz_feat,
        z_cols=Z_COLS,
        emu_mat=emu_mat,
        dist_mat=dist_mat,
        zone_to_idx=zone_to_idx,
        idx_to_zone=idx_to_zone,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=max_n,
        seed=seed_full,
        chunk=FULL_EVAL_CHUNK,
        topKs=TOPKS_FULL,
    )

    rows_full.append({
        "segment": seg,
        "alpha_emu": float(alpha_hat),
        "K": K,
        **full_out,
    })

    print("[FULL OOS   ]",
          f"NLL_full_avg={full_out['NLL_full_avg']:.4f} MRR_full={full_out['MRR_full']:.4f} "
          f"FF_full={full_out['FF_full']:.6f} "
          f"R2_full={full_out['McFadden_R2_full']:.4f} Top5_full={full_out['Top5_full']:.3f} Top10_full={full_out['Top10_full']:.3f}")
    print("[SHARES FULL]",
          f"Pear={full_out['Pearson_share']:.4f} Spear={full_out['Spearman_share']:.4f} "
          f"MAE={full_out['MAE_share']:.6f} RMSE={full_out['RMSE_share']:.6f} JS={full_out['JS_share']:.6f}")

    obs_dist, exp_dist = dist_pack
    if (obs_dist is not None) and (exp_dist is not None):
        png = os.path.join(PLOT_DIR, f"distance_plausibility_{seg}.png")
        plot_distance_plausibility(obs_dist, exp_dist, seg, png)
        print(f"[PLOT] wrote {png}")

        if trip_distance_df is not None:
            dist_csv = os.path.join(DIST_OOS_DIR, f"distance_triplevel_{seg}.csv")
            trip_distance_df.to_csv(dist_csv, index=False)
            print(f"[CSV ] wrote {dist_csv}")

    vcov_mat, se_vec, H_sum, used_n = approx_hessian_inference(
        emu_set=emu_tr,
        X_set=X_tr,
        y=y_tr,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=SE_MAX_TRAIN.get(seg, 20000),
        seed=SE_SEED + 1000 * SEG_SEED[seg],
        ridge=1e-8,
    )

    seg_vcov[seg] = vcov_mat.copy()
    seg_se[seg]   = se_vec.copy()
    seg_hsum[seg] = H_sum.copy()
    seg_inference_meta[seg] = {
        "segment": seg,
        "used_n_for_hessian": used_n,
        "num_params": len(PARAM_NAMES),
    }

    df_vcov = pd.DataFrame(vcov_mat, index=PARAM_NAMES, columns=PARAM_NAMES)
    df_hsum = pd.DataFrame(H_sum, index=PARAM_NAMES, columns=PARAM_NAMES)
    df_vcov.to_csv(os.path.join(VCOV_DIR, f"vcov_{seg}.csv"))
    df_hsum.to_csv(os.path.join(VCOV_DIR, f"hessian_sum_{seg}.csv"))

    df_diag = pd.DataFrame({
        "segment": seg,
        "param": PARAM_NAMES,
        "coef": coef_vec,
        "se": se_vec,
        "var": np.diag(vcov_mat),
    })
    df_diag.to_csv(os.path.join(VCOV_DIR, f"coef_se_diag_{seg}.csv"), index=False)

    for name, b, s in zip(PARAM_NAMES, coef_vec.tolist(), se_vec.tolist()):
        zval = b / (s + 1e-12)
        pval = p_value_two_sided_z(zval)
        rows_coef.append({
            "segment": seg,
            "param": name,
            "coef": float(b),
            "se": float(s),
            "z": float(zval),
            "p": float(pval),
            "stars": stars(pval),
        })

    if share_pack is not None:
        tz_ids  = share_pack["tz_ids"]
        S_obs   = share_pack["S_obs"]
        S_pred  = share_pack["S_pred"]
        Aj      = share_pack["Aj"]
        orig_i  = share_pack["orig_idx"]

        tz_pos = {int(z): i for i, z in enumerate(tz_ids.tolist())}

        L_in = incoming_accessibility_logsum(
            tz_ids=tz_ids,
            tz_pos=tz_pos,
            emu_mat=emu_mat,
            alpha=float(alpha_hat),
            orig_idx=orig_i,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            chunk=FULL_EVAL_CHUNK,
        )

        png = os.path.join(PLOT_DIR, f"attr_vs_access_{seg}.png")
        plot_attr_vs_access(Aj, L_in, S_pred, seg, png)
        print(f"[PLOT] wrote {png}")

        tz_out = tz_feat[["npvm_id", "geometry"]].copy()
        tz_out = tz_out.set_index("npvm_id").loc[tz_ids].reset_index()

        tz_out[f"A_attr_{seg}"] = Aj
        tz_out[f"S_obs_{seg}"]  = S_obs
        tz_out[f"S_pred_{seg}"] = S_pred
        tz_out[f"L_in_{seg}"]   = L_in

        png1 = os.path.join(PLOT_DIR, f"map_S_pred_{seg}.png")
        png2 = os.path.join(PLOT_DIR, f"map_A_attr_{seg}.png")
        plot_share_maps(tz_out, f"S_pred_{seg}", seg, png1, "Predicted destination share (OOS subset, full denom)")
        plot_share_maps(tz_out, f"A_attr_{seg}", seg, png2, "Attractivity index A_j = X_j beta (z-scale)")
        print(f"[PLOT] wrote {png1}")
        print(f"[PLOT] wrote {png2}")

        layer_name = f"TZ_{seg}"
        tz_out.to_file(GPKG_OUT, layer=layer_name, driver="GPKG")
        print(f"[GPKG] wrote layer {layer_name} -> {GPKG_OUT}")


# ============================================================
# Save summary tables
# ============================================================
df_sampled = pd.DataFrame(rows_sampled)
df_full    = pd.DataFrame(rows_full)
df_coef    = pd.DataFrame(rows_coef)
df_infmeta = pd.DataFrame(list(seg_inference_meta.values()))

p1 = os.path.join(OUT_DIR, "metrics_sampled.csv")
p2 = os.path.join(OUT_DIR, "metrics_full_and_shares.csv")
p3 = os.path.join(OUT_DIR, "coef_table.csv")
p4 = os.path.join(OUT_DIR, "hessian_inference_meta.csv")

df_sampled.to_csv(p1, index=False)
df_full.to_csv(p2, index=False)
df_coef.to_csv(p3, index=False)
df_infmeta.to_csv(p4, index=False)

# ============================================================
# coefficient difference tests
# ============================================================
rows_within = []
for seg in SEGMENTS:
    rows_within.extend(
        build_within_segment_diff_tests(
            segment=seg,
            params=PARAM_NAMES,
            coef_vec=seg_coef_vec[seg],
            vcov=seg_vcov[seg],
        )
    )
df_within = pd.DataFrame(rows_within)
df_within.to_csv(os.path.join(TEST_DIR, "coef_diff_within_segment.csv"), index=False)

rows_across = build_across_segment_sameparam_tests(
    seg_to_coef=seg_coef_vec,
    seg_to_vcov=seg_vcov,
    params=PARAM_NAMES,
)
df_across = pd.DataFrame(rows_across)
df_across.to_csv(os.path.join(TEST_DIR, "coef_diff_across_segments.csv"), index=False)

wide_coef = df_coef.pivot(index="param", columns="segment", values="coef").reset_index()
wide_se   = df_coef.pivot(index="param", columns="segment", values="se").reset_index()
wide_coef.to_csv(os.path.join(TEST_DIR, "coef_wide_by_segment.csv"), index=False)
wide_se.to_csv(os.path.join(TEST_DIR, "se_wide_by_segment.csv"), index=False)

print("\n" + "=" * 110)
print("[OK] wrote", p1)
print("[OK] wrote", p2)
print("[OK] wrote", p3)
print("[OK] wrote", p4)
print("[OK] wrote", GPKG_OUT)
print("[OK] vcov matrices in", VCOV_DIR)
print("[OK] coefficient tests in", TEST_DIR)
print("[OK] plots in", PLOT_DIR)
print("[OK] trip-level distance CSVs in", DIST_OOS_DIR)
print("=" * 110)

print("\nSAMPLED OOS METRICS (unweighted):")
print(df_sampled[df_sampled["split"] == "oos"][["segment","NLL_sum","NLL_avg","MRR","Top5","Top10","FF","N","J"]]
      .sort_values("segment").to_string(index=False))

print("\nFULL OOS METRICS + SHARE DIAGNOSTICS (unweighted):")
keep_cols = [
    "segment","NLL_full_sum","NLL_full_avg","MRR_full","FF_full","McFadden_R2_full","Top5_full","Top10_full",
    "Pearson_share","Spearman_share","MAE_share","RMSE_share","JS_share","N_full_eval","J_full",
    "dist_obs_mean_km","dist_pred_mean_km","dist_mean_diff_km","dist_obs_p50_km","dist_pred_p50_km","dist_obs_p90_km","dist_pred_p90_km",
]
cols_exist = [c for c in keep_cols if c in df_full.columns]
print(df_full[cols_exist].sort_values("segment").to_string(index=False))

print("\nTOP OF coef_diff_across_segments.csv")
print(df_across.head(20).to_string(index=False))

# ============================================================
# EXPORT "Result" PACKAGE FOR SIMBA INTEGRATION
# ============================================================
print("\n" + "=" * 110)
print("[EXPORT] Building Result package (Attractivity + utilities omx) ...")

omx_idx_master = np.array([zone_to_idx.get(int(z), -1) for z in tz_ids_master], dtype=int)

attr_df = pd.DataFrame({
    "zone_id": tz_ids_master.astype(int),
    "omx_idx": omx_idx_master.astype(int),
    "Attr_YS": seg_Aj_tz["YS"].astype(np.float64),
    "Attr_OS": seg_Aj_tz["OS"].astype(np.float64),
    "Attr_YL": seg_Aj_tz["YL"].astype(np.float64),
    "Attr_OL": seg_Aj_tz["OL"].astype(np.float64),
})

attr_df.to_csv(ATTR_CSV_OUT, index=False)
print("[OK] wrote", ATTR_CSV_OUT)

if os.path.exists(ATTR_GPKG_OUT):
    os.remove(ATTR_GPKG_OUT)

geom = tz_feat[["npvm_id", "geometry"]].drop_duplicates().set_index("npvm_id").loc[tz_ids_master].reset_index()
geom = geom.rename(columns={"npvm_id": "zone_id"})
attr_gdf = gpd.GeoDataFrame(
    geom.merge(attr_df, on="zone_id", how="left"),
    geometry="geometry",
    crs=tz_feat.crs,
)
attr_gdf.to_file(ATTR_GPKG_OUT, driver="GPKG")
print("[OK] wrote", ATTR_GPKG_OUT)

if os.path.exists(UTIL_SEG_OMX):
    os.remove(UTIL_SEG_OMX)

Msize = emu_mat.shape[0]
print(f"[OMX-OUT] Creating {UTIL_SEG_OMX} with 4 matrices of shape {emu_mat.shape} (float32) ...")

fout = omx.open_file(UTIL_SEG_OMX, "w")
try:
    fout.create_mapping(MAP_NAME, idx_to_zone.astype(np.int32))
except Exception:
    fout.create_mapping(MAP_NAME, [int(x) for x in idx_to_zone.tolist()])

for seg in SEGMENTS:
    float_atom = tb.Atom.from_dtype(np.dtype("float32"))
    fout.create_matrix(f"utility_{seg}", atom=float_atom, shape=(Msize, Msize))

ROW_BLOCK = 512
for seg in SEGMENTS:
    alpha32 = np.float32(seg_alpha[seg])
    Aj_omx = seg_Aj_by_omx[seg].astype(np.float32)

    print(f"[OMX-OUT] Writing utility_{seg} ... alpha={float(seg_alpha[seg]):.6f}")
    mat = fout[f"utility_{seg}"]

    for i0 in range(0, Msize, ROW_BLOCK):
        i1 = min(Msize, i0 + ROW_BLOCK)
        emu_blk = emu_mat[i0:i1, :]
        util_blk = alpha32 * emu_blk + Aj_omx[None, :]
        mat[i0:i1, :] = util_blk.astype(np.float32)

fout.close()
print("[OK] wrote", UTIL_SEG_OMX)

print("[EXPORT] Done. Result files are in:", RESULT_DIR)
print("=" * 110)